# [1.3.3] SAE를 이용한 Interpretability (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/13_[1.3.3]_Interpretability_with_SAEs)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part33_interp_with_saes/1.3.3_Interpretability_with_SAEs_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part33_interp_with_saes/1.3.3_Interpretability_with_SAEs_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 내용에 관한 질문은 전용 채널에 문의해 주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 이동하는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> 참고 - **이 연습 문제 세트에는 매우 방대한 양의 내용이 포함되어 있으며**, ARENA의 다른 단일 연습 문제 세트보다 쉽게 두 배 이상 많습니다 (일부 연습 문제 세트는 며칠 동안 진행하도록 설계되었습니다). 이 연습 문제들의 목적은 모든 문제를 하나하나 다 푸는 것이 아니라, **본인이 가장 관심 있는 부분들을 선택해서 학습하는 것**입니다.
>
> 또한, 이 자료를 연습 문제로 사용하는 대신, 특정 SAE 기법이나 forward pass / causal intervention 유형을 빠르게 구현하고 싶을 때 유용한 참조 코드 소스로 활용하실 수도 있습니다.
>
> 아래의 인터랙티브 맵을 통해 학습 내용과 각 섹션 간의 의존 관계를 더 잘 파악하실 수 있습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-13-2.png" width="350">
<br>

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src="https://link.excalidraw.com/readonly/lwbmz3hSOL38907OaCBR", width=1000, height=500))

# 소개

이 실습에서는 sparse autoencoder를 통해 수행할 수 있는 interpretability 연구를 깊이 있게 다룹니다. 우리는 두 가지 중요한 도구인 `SAELens` (본질적으로 SAE를 위한 TransformerLens이며, TransformerLens와도 매우 잘 통합됩니다)와 interpretability 연구를 위한 오픈 플랫폼인 **Neuronpedia**를 소개하며, 실제 language model에 SAE를 사용하는 단계로 바로 들어갑니다. 그 후, 여러 카테고리(예: latent 이해 및 분류, SAE 학습 및 평가)로 그룹화된 SAE interpretability의 다른 흥미로운 영역들을 살펴볼 것입니다.

이 실습을 위해서는 어느 정도의 선수 지식이 필요합니다. 특히 다음 내용을 이해하고 계신다면 매우 도움이 될 것입니다:

- **superposition**이란 무엇인가
- **sparse autoencoder** architecture가 무엇이며, 왜 이것이 superposition으로부터 feature를 disentangle하는 데 도움이 되는가

SAE와 superposition의 배후에 있는 이론에 관심이 있다면, 이를 자세히 다루는 **1.5.4 Toy Models of SAEs & Superposition** 실습으로 이동하시기 바랍니다. 특히 해당 실습 세트의 섹션 1과 섹션 5의 전반부를 빠르게 훑어보신다면, superposition과 SAE의 핵심 이론을 이해하는 데 충분할 것입니다.

시작하기 전 한 가지 참고 사항이 있습니다. 우리는 주로 **features**를 베이스 모델이 학습한 기본 데이터 분포의 특성으로, **SAE latents**(또는 단순히 "latents")를 SAE 내의 방향으로 정의하는 용어를 채택할 것입니다. 이는 "feature"라는 용어의 과부하를 피하고, "SAE features"가 데이터의 실제 feature와 일치한다는 암묵적인 가정을 피하기 위함입니다. 다만, SAE latents가 데이터의 특정 interpretable feature와 매우 명확하게 일치하는 경우에는 이러한 용어 구분을 완화하여 사용하겠습니다.

## 읽기 자료

이 내용의 대부분은 선택 사항이며, 여러분의 관심사와 배경 지식 수준에 따라 편하게 읽으시면 됩니다. 단 하나만 추천해야 한다면 "Towards Monosemanticity"를 추천하며, 특히 "Problem Setup"의 전반부와 개별 latent를 깊게 분석한 섹션들을 추천합니다.

- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) 은 superposition의 핵심 아이디어, 즉 그것이 무엇인지, 왜 interpretability에 중요한지, 그리고 우리가 이에 대해 무엇을 할 수 있는지에 대해 설명합니다.
- [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html) 은 SAE를 통해 mechanistic interpretability에서 첫 번째 큰 진전을 이룬 연구로 볼 수 있습니다. 1-layer 모델에서 SAE를 학습시키고, 수많은 interpretable feature를 추출했습니다.
- [Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html) 는 SAE 과학을 더 큰 모델, 특히 (당시) SOTA 모델이었던 Claude 3 Sonnet으로 어떻게 확장할 수 있는지 보여줍니다. 이는 이 분야가 가까운 미래에 어디로 나아갈지에 대한 흥미로운 통찰을 제공합니다.
- [Improving Dictionary Learning with Gated Sparse Autoencoders](https://arxiv.org/pdf/2404.16014) 은 DeepMind의 논문으로, Gated SAE architecture를 소개하며 이것이 표준 architecture보다 어떻게 더 나은 성능을 보이는지 입증하고, 기저의 feature 분포에 대한 추론을 통해 그 사용 근거를 제시합니다.
- [Gemma Scope](https://deepmind.google/discover/blog/gemma-scope-helping-the-safety-community-shed-light-on-the-inner-workings-of-language-models/) 는 DeepMind가 오픈 소스로 공개한 포괄적인 SAE 모델 제품군(JumpReLU architecture로 학습됨)에 대한 발표입니다. 이후 실습에서 Gemma Scope 모델을 많이 사용하게 될 것입니다!
- [LessWrong, SAEs tag](https://www.lesswrong.com/tag/sparse-autoencoders-saes) 는 SAE를 다루는 LessWrong 포스트 모음집이며, 추가적인 독립 연구를 위한 훌륭한 영감의 원천이 될 것입니다!

## 콘텐츠 및 학습 목표

완료된 자료와 현재 개발 중인 자료, 그리고 각 섹션의 끝에서 여러분이 무엇을 만들게 될지를 보여주는 그림들을 확인하시려면 [this interactive map](https://link.excalidraw.com/l/9KwMnW35Xt8/86r7xyurK0g) 을(를) 참조하시기 바랍니다. 여기에는 아래의 거의 모든 정보가 포함되어 있으며, 더 시각화하기 쉬운 형식으로 제공됩니다.

맵을 통해 명확히 알 수 있는 몇 가지 중요한 사항들을 강조해 드립니다:

- 이 자료를 학습하는 데 정해진 순서는 없습니다! 섹션 1️⃣의 처음 몇 부분만 제외하면, 여러분의 목표에 따라 기본적으로 원하는 내용을 선택해서 학습하실 수 있습니다. 맵에는 서로 다른 섹션 간의 의존성이 표시되어 있어, 이를 학습 가이드로 활용하실 수 있습니다.
- 일부 섹션은 아직 개발 중이며, 10월과 11월에 걸쳐 계속 추가될 예정입니다 (다만, 제가 인터뷰 과정을 진행하는 10월 중순에는 개발이 대부분 중단될 것입니다). 이 자료를 더욱 개선하거나 추가할 방법에 대한 제안이나 추천은 언제든 환영합니다!

### 1️⃣ SAE Interpretability 입문

이 섹션의 아이디어는 훈련 및 evals(섹션 3️⃣에서 다시 다룰 예정)를 제외한 모든 기본적인 SAE 주제에 대한 MVP가 되는 것입니다. 초점은 SAE latent를 어떻게 이해하고 해석할 것인가(특히 [SAE dashboard](https://transformer-circuits.pub/2023/monosemantic-features/vis/a1.html) 의 모든 구성 요소)에 맞춰질 것입니다. 또한 latent를 찾는 기술(예: ablation 및 attribution 방법)과 attention SAE 및 그 작동 방식에 대한 심층 분석도 살펴볼 것입니다.

> ##### 학습 목표
>   
> - `SAELens` 라이브러리를 사용하여 SAE를 로드하고 실행하는 방법(연결된 TransformerLens 모델과 함께)을 배웁니다.
> - **Neuronpedia**의 기본 기능과 이를 steering 및 feature 검색 등에 활용하는 방법을 이해합니다.
> - **SAE dashboards**를 이해하고, 각 부분이 특정 latent에 대해 무엇을 알려주는지(그리고 이를 직접 계산하는 방법)를 이해합니다.
> - **direct logit attribution**, **ablation**, **attribution patching**을 포함하여 latent를 찾는 기술을 배웁니다.
> - **attention SAEs**를 사용하고, 이것이 일반적인 SAE와 어떻게 다른지(그리고 **direct latent attribution**과 같이 attention SAE에 특화된 주제들)를 이해합니다.
> - 다양한 SAE architecture 또는 훈련 방법(예: gated, end-to-end, meta-saes, transcoders)에 대해 조금 배웁니다. 이 중 일부는 나중에 더 자세히 다룰 예정입니다.

### 2️⃣ SAE Latents 이해하기: 심층 분석

이 섹션은 기본적으로 여러 가지 서로 다른 주제들의 a-la-carte 모음입니다. 이 주제들은 SAE circuit나 SAE 훈련과 직접적으로 관련되어 있지는 않지만, 입문 섹션에서 다루기에는 너무 지엽적이거나 심층적인 내용들입니다. 이 중 상당 부분은 현재 SAE interpretability의 최전선에서 이루어지고 있는 연구를 나타내며, 여러분만의 연구를 시작하기 위한 흥미로운 출발점이 될 수 있습니다!

> ##### 학습 목표
>
> - feature splitting과 이것이 SAE 훈련에 무엇을 의미하는지 연구합니다.
> - UMAP 및 기타 차원 축소 기술을 사용하여 SAE latent geometry를 더 잘 이해합니다.
> - feature absorption을 이해하고, meta-SAEs가 이 문제를 해결하는 데 어떻게 도움이 될 수 있는지 이해합니다 (아직 구현되지 않음).
> - logit lens 및 token enrichment analysis와 같은 기술을 사용하여 SAE latent를 더 잘 이해하고 특성화합니다 (아직 구현되지 않음).
> - autointerp 기반의 evals 및 patch scoping을 다루는 automated interpretability를 심층적으로 분석합니다 (아직 구현되지 않음).

### 3️⃣ SAE 훈련 및 평가

이 섹션에서는 먼저 SAE 훈련에 관한 기본 자료를 다룹니다. `SAELens` 이(가) SAE 훈련을 어떻게 지원하는지 보여드리고 몇 가지 일반적인 조언을 다룬 후, 몇 가지 훈련 사례 연구를 진행합니다. 이러한 각 훈련 실습은 훈련된 모델을 추가로 조사하기 위한 좋은 출발점이 됩니다 (다만, 이 튜토리얼들은 간결함과 낮은 연산 요구 사항에 최적화되어 있으므로, 2L 모델에서 훈련된 attention SAE와 같은 작은 모델의 경우 베이스 모델의 복잡도 수준에 맞는 feature를 찾았을 가능성이 더 높기 때문에 작은 모델에서 더 유효합니다!).

이 섹션의 후반부에는 evals를 다루는 내용을 추가할 계획이지만, 현재는 아직 개발 중입니다 (단, 이 섹션의 전반부나 섹션 2️⃣의 autointerp 자료 등 다른 부분에서 많은 evals 관련 내용을 다루고 있습니다).

> ##### 학습 목표
>
> - `SAELens` 을(를) 사용하여 SAE를 훈련하는 방법을 배웁니다.
> - 훈련 중 다양한 metric을 해석하는 방법을 이해하고, SAE 훈련이 왜, 그리고 언제 interpretable한 latent를 생성하는 데 실패하는지 이해합니다.
> - TinyStories-1L의 MLP output, Gemma-2-2B의 residual stream, 2L 모델의 attention output 등 다양한 컨텍스트에서 SAE를 훈련하는 실무 경험을 쌓습니다.
> - SAE를 평가하는 방법과 단순한 metric이 왜 기만적일 수 있는지 이해합니다 (아직 구현되지 않음).

## A note on memory usage

In these exercises, we'll be loading some pretty large models into memory (e.g. Gemma 2-2B and its SAEs, as well as a host of other models in later sections of the material). It's useful to have functions which can help profile memory usage for you, so that if you encounter OOM errors you can try and clear out unnecessary models. For example, we've found that with the right memory handling (i.e. deleting models and objects when you're not using them any more) it should be possible to run all the exercises in this material on a Colab Pro notebook, and all the exercises minus the handful involving Gemma on a free Colab notebook.

<details>
<summary>See this dropdown for some functions which you might find helpful, and how to use them.</summary>

First, we can run some code to inspect our current memory usage. Here's me running this code during the exercise set on SAE circuits, after having already loaded in the Gemma models from the previous section. This was on a Colab Pro notebook.

```python
import part33_interp_with_saes.utils as utils

# Profile memory usage, and delete gemma models if we've loaded them in
namespace = globals().copy() | locals()
utils.profile_pytorch_memory(namespace=namespace, filter_device="cuda:0")
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 35.88 GB
Total = 39.56 GB
Free = 3.68 GB
┌──────────────────────┬────────────────────────┬──────────┬─────────────┐
│ Name                 │ Object                 │ Device   │   Size (GB) │
├──────────────────────┼────────────────────────┼──────────┼─────────────┤
│ gemma_2_2b           │ HookedSAETransformer   │ cuda:0   │       11.94 │
│ gpt2                 │ HookedSAETransformer   │ cuda:0   │        0.61 │
│ gemma_2_2b_sae       │ SAE                    │ cuda:0   │        0.28 │
│ sae_resid_dirs       │ Tensor (4, 24576, 768) │ cuda:0   │        0.28 │
│ gpt2_sae             │ SAE                    │ cuda:0   │        0.14 │
│ logits               │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ logits_with_ablation │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ clean_logits         │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ _                    │ Tensor (16, 128, 768)  │ cuda:0   │        0.01 │
│ clean_sae_acts_post  │ Tensor (4, 15, 24576)  │ cuda:0   │        0.01 │
└──────────────────────┴────────────────────────┴──────────┴─────────────┘</pre>

From this, we see that we've allocated a lot of memory for the the Gemma model, so let's delete it. We'll also run some code to move any remaining objects on the GPU which are larger than 100MB to the CPU, and print the memory status again.

```python
del gemma_2_2b
del gemma_2_2b_sae

THRESHOLD = 0.1  # GB
for obj in gc.get_objects():
    try:
        if isinstance(obj, t.nn.Module) and part32_utils.get_tensors_size(obj) / 1024**3 > THRESHOLD:
            if hasattr(obj, "cuda"):
                obj.cpu()
            if hasattr(obj, "reset"):
                obj.reset()
    except:
        pass

# Move our gpt2 model & SAEs back to GPU (we'll need them for the exercises we're about to do)
gpt2.to(device)
gpt2_saes = {layer: sae.to(device) for layer, sae in gpt2_saes.items()}

part32_utils.print_memory_status()
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 14.90 GB
Reserved = 39.56 GB
Free = 24.66</pre>

Mission success! We've managed to free up a lot of memory. Note that the code which moves all objects collected by the garbage collector to the CPU is often necessary to free up the memory. We can't just delete the objects directly because PyTorch can still sometimes keep references to them (i.e. their tensors) in memory. In fact, if you add code to the for loop above to print out `obj.shape` when `obj` is a tensor, you'll see that a lot of those tensors are actually Gemma model weights, even once you've deleted `gemma_2_2b`.

</details>

## 설정 (읽지 말고 실행만 하세요)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias

import circuitsvis as cv
import einops
import pandas as pd
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, display
from jaxtyping import Float, Int
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])


# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part33_interp_with_saes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part33_interp_with_saes.tests as tests
import part33_interp_with_saes.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# For displaying sae-vis inline
if IN_COLAB:
    import http.server
    import socketserver
    import threading

    from google.colab import output

    PORT = 8000

    def display_vis_inline(filename: Path, height: int = 850):
        """
        Displays the HTML files in Colab. Uses global `PORT` variable defined in prev cell, so that each
        vis has a unique port without having to define a port within the function.
        """
        global PORT

        def serve(directory):
            os.chdir(directory)
            handler = http.server.SimpleHTTPRequestHandler
            with socketserver.TCPServer(("", PORT), handler) as httpd:
                print(f"Serving files from {directory} on port {PORT}")
                httpd.serve_forever()

        thread = threading.Thread(target=serve, args=("/content",))
        thread.start()

        filename = str(filename).split("/content")[-1]

        output.serve_kernel_port_as_iframe(PORT, path=filename, height=height, cache_in_notebook=True)

        PORT += 1

# 1️⃣ SAE Interpretability 입문

> ##### 학습 목표
>
> - `SAELens` 라이브러리를 사용하여 SAE를 로드하고 실행하는 방법(및 SAE가 연결된 TransformerLens 모델과 함께 사용하는 방법)을 배웁니다.
> - **Neuronpedia**의 기본 기능과 steering 및 feature 검색 등에 활용하는 방법을 이해합니다.
> - **SAE dashboard**를 이해하고, 각 부분이 특정 latent에 대해 무엇을 알려주는지(및 이를 직접 계산하는 방법)를 이해합니다.
> - **direct logit attribution**, **ablation**, **attribution patching**을 포함하여 latent를 찾는 기법들을 배웁니다.
> - **attention SAE**를 사용하고, 이것이 일반적인 SAE와 어떻게 다른지(및 **direct latent attribution**과 같이 attention SAE에 특화된 주제들)를 이해합니다.
> - 다양한 SAE architecture 또는 학습 방법(예: gated, end-to-end, meta-saes, transcoders)에 대해 간단히 배웁니다. 이 중 일부는 나중에 더 자세히 다룰 예정입니다.

강조하자면, 이 섹션의 목적은 훈련 및 평가(섹션 4에서 다시 다룰 예정)를 제외한 모든 기본적인 SAE 주제를 빠르게 훑어보는 것입니다. 초점은 SAE latent를 어떻게 이해하고 해석할 것인가(특히 [SAE dashboard](https://transformer-circuits.pub/2023/monosemantic-features/vis/a1.html)의 모든 구성 요소)에 맞춰질 것입니다. 또한 latent를 찾는 기술(예: ablation 및 attribution 방법)과 attention SAE 및 그 작동 방식에 대해 더 깊이 있게 살펴볼 것입니다. 이 섹션에서 다뤄야 할 내용이 많기 때문에, 각 주요 헤더 섹션의 상단에 핵심 포인트 요약을 제공합니다. 시작하기 전에 편의를 위해 모든 요약을 아래에 포함했습니다. 이 요약들은 학습 과정에서 방향을 잡는 데 도움을 줄 뿐만 아니라, 일부 섹션만 학습하고 싶을 때 어떤 섹션으로 건너뛸 수 있을지에 대한 아이디어를 제공할 것입니다.

<details>
<summary>SAELens 소개</summary>

이 섹션에서는 `SAELens`가 무엇인지, 그리고 이를 사용하여 지원되는 다양한 SAE의 config를 로드하고 검사하는 방법을 배웁니다. 핵심 포인트는 다음과 같습니다:

- SAELens는 SAE를 훈련하고 분석하기 위한 라이브러리입니다. SAE를 위한 TransformerLens의 대응물이라고 생각할 수 있습니다 (다만, "Running SAEs" 섹션에서 보게 되겠지만 TransformerLens와 밀접하게 통합되어 있습니다).
- SAELens에는 많은 다양한 모델 릴리스가 포함되어 있으며, 각 릴리스는 여러 개의 SAE(예: 서로 다른 모델 layer / hook point에서 훈련되었거나 서로 다른 architecture를 가진 SAE)를 포함합니다.
- `SAE` 인스턴스의 `cfg` 속성에는 이러한 정보와 forward pass를 수행할 때 관련 있는 다른 모든 정보가 포함되어 있습니다.

</details>

<details>
<summary>대시보드를 이용한 SAE 시각화</summary>

이 섹션에서는 특정 SAE latent가 무엇을 나타내는지 빠르게 이해하기 위한 시각적 도구인 SAE 대시보드에 대해 배웁니다. 핵심 포인트는 다음과 같습니다:

- Neuronpedia는 SAE latent를 이해하는 데 도움이 되는 대시보드를 호스팅합니다.
- 대시보드의 5가지 주요 구성 요소는 top logit 테이블, logits 히스토그램, activation 밀도 플롯, top activating 시퀀스, 그리고 autointerp입니다.
- 이 모든 구성 요소는 latent가 무엇을 나타내는지에 대한 전체적인 그림을 얻는 데 중요하지만, 동시에 모두 오해의 소지가 있을 수 있습니다.
- `IFrame`을 사용하여 이러한 대시보드를 인라인으로 표시할 수 있습니다.

</details>

<details>
<summary>SAE 실행하기</summary>

이 섹션에서는 SAE를 사용하여 forward pass를 실행하는 방법을 배웁니다. 이는 TransformerLens 모델의 기존 인프라를 많이 활용하는 매우 간단한 과정입니다. 핵심 포인트는 다음과 같습니다:

- forward pass를 수행할 때 hook 함수를 추가하는 것과 거의 동일한 방식으로 TransformerLens 모델에 SAE를 추가할 수 있습니다 (SAE를 일종의 특수한 hook 함수라고 생각하면 됩니다).
- `sae.error_term=False` (기본값)일 때는 transformer activation을 SAE의 출력으로 대체합니다. True일 때는 대체하지 않습니다 (이는 activation을 캐싱할 때 가끔 필요한 방식입니다).
- `run_with_hooks`과 유사하게 작동하는 대응되는 `run_with_saes`이 있습니다.
- `run_with_cache`과 유사하게 작동하면서 원하는 모든 SAE activation을 캐싱할 수 있게 해주는 `run_with_cache_with_saes`도 있습니다.
- `ActivationStore`을 사용하여 한 번에 대량의 activation 배치를 얻을 수 있습니다.

</details>

<details>
<summary>SAE 대시보드 재현하기</summary>

이 섹션에서는 SAE 대시보드의 5가지 주요 구성 요소인 top logits 테이블, logits 히스토그램, activation 밀도 플롯, top activating 시퀀스, 그리고 autointerp를 직접 재현해 봅니다. 여기에는 새로운 내용이 거의 없으며, 이전 두 섹션인 "대시보드를 이용한 SAE 시각화"와 "SAE 실행하기"에서 배운 내용을 실제로 적용하는 과정입니다.

</details>

<details>
<summary>Attention SAEs</summary>

이 섹션에서는 attention SAE, 그 작동 방식(대부분 표준 SAE와 매우 유사하지만 몇 가지 다른 고려 사항이 있음), 그리고 feature 대시보드를 이해하는 방법을 배웁니다. 핵심 포인트는 다음과 같습니다:

- Attention SAE는 일반 SAE와 동일한 architecture를 가지지만, 모든 attention head의 pre-projection 출력물을 연결(concatenate)한 데이터로 훈련됩니다.
- latent가 destination token에서 활성화되면, **direct latent attribution**을 사용하여 해당 latent가 주로 어떤 source token에서 왔는지 확인할 수 있습니다.
- 일반 SAE와 마찬가지로, 모델의 서로 다른 layer에서 발견되는 latent들은 종종 서로 질적으로 다릅니다.

</details>

<details>
<summary>feature를 위한 latent 찾기</summary>

이 섹션에서는 특정 feature에 해당하는 SAE 내의 latent를 찾기 위한 다양한 방법(인과적 방법과 그렇지 않은 방법 모두)을 탐구합니다. 핵심 포인트는 다음과 같습니다:

- 특정 입력 프롬프트에서 **max activating latents**를 살펴볼 수 있으며, 이는 기본적으로 가장 간단한 방법입니다.
- **Direct logit attribution (DLA)**은 조금 더 정교한 방법으로, 특정 logit에 직접적인 영향을 주는 latent를 찾을 수 있습니다.
- SAE latent의 **ablation**은 직접적이지 않은 방식으로 중요한 latent를 찾는 데 도움이 될 수 있습니다.
- ...하지만 많은 수의 latent에 대해 수행하기에는 비용이 많이 들므로, ablation의 저렴한 선형 근사치인 **attribution patching**을 사용할 수 있습니다.

</details>

<details>
<summary>GemmaScope</summary>

이 짧은 섹션에서는 DeepMind의 GemmaScope 시리즈를 소개합니다. 이는 성능이 매우 뛰어난 SAE 세트로, 여러분의 interpretability 프로젝트에서 훌륭한 연구 자료가 될 수 있습니다!

</details>

<details>
<summary>Feature steering</summary>

이 섹션에서는 흥미로운 모델 출력을 생성하기 위해 latent를 steering하는 방법을 배웁니다. 핵심 포인트는 다음과 같습니다:

- Steering은 forward pass 중에 개입하여 모델의 activation을 특정 latent 방향으로 변경하는 것을 포함합니다.
- steering 동작은 때때로 예측 불가능하며, 항상 "latent가 강하게 활성화되는 것과 동일한 유형의 텍스트를 생성하는 것"과 일치하지는 않습니다.
- Neuronpedia에는 코드 없이도 steering을 할 수 있는 steering 인터페이스가 있습니다.

</details>

<details>
<summary>다른 유형의 SAEs</summary>

이 섹션에서는 몇 가지 다른 SAE architecture를 소개하며, 그 중 일부는 이후 섹션에서 더 자세히 다룰 예정입니다. 여기에는 실습 과제가 없으며 간단한 설명만 제공됩니다. 핵심 포인트는 다음과 같습니다:

- **TopK**, **JumpReLU**, **Gated** 모델과 같은 서로 다른 activation 함수 / encoder architecture는 feature suppression 문제나 표준 모델에서 SAE가 연속적이어야 한다는 압박과 같은 문제들을 해결할 수 있습니다.
- **End-to-end SAEs**는 다른 loss 함수로 훈련되어, 단순히 MSE reconstruction error를 최소화하는 것이 아니라 모델의 출력에 기능적으로 유용한 feature를 학습하도록 유도합니다.
- **Transcoders**는 단순히 activation을 재구성하는 것이 아니라 모델의 계산(예: MLP 입력에서 MLP 출력으로의 sparse mapping)을 재구성하도록 학습하는 SAE의 한 유형입니다. 이는 때때로 더 쉬운 circuit 분석으로 이어질 수 있습니다.

</details>

## SAELens 소개

> 이 섹션에서는 `SAELens`이 무엇인지, 그리고 이를 사용하여 지원되는 다양한 SAE의 config를 로드하고 검사하는 방법을 배웁니다. 핵심 포인트는 다음과 같습니다:
>
> - SAELens는 SAE를 훈련하고 분석하기 위한 라이브러리입니다. 이는 SAE를 위한 TransformerLens의 대응물이라고 생각할 수 있습니다 (비록 "Running SAEs" 섹션에서 보게 되겠지만, TransformerLens와도 밀접하게 통합되어 있습니다).
> - SAELens에는 많은 다양한 모델 릴리스가 포함되어 있으며, 각 릴리스는 여러 개의 SAE를 포함하고 있습니다 (예: 서로 다른 모델 layer / hook point에서 훈련되었거나, 서로 다른 architecture를 가진 경우).
> - `SAE` 인스턴스의 `cfg` 속성에는 이 정보와 forward pass를 수행할 때 관련이 있는 다른 모든 정보가 포함되어 있습니다.

[SAELens](https://github.com/jbloomAus/SAELens)은 연구자들을 돕기 위해 설계된 라이브러리입니다:

- sparse autoencoder 학습,
- sparse autoencoder 분석 / mechanistic interpretability 연구,
- 안전하고 정렬된 AI 시스템을 더 쉽게 만들기 위한 인사이트 생성.

이를 sparse autoencoder를 위한 TransformerLens의 대응물이라고 생각하시면 됩니다 (또한 곧 살펴보겠지만, TransformerLens 모델과도 매우 잘 통합됩니다).

추가적으로, SAELens는 Joseph Bloom과 Johnny Lin의 공동 노력으로 개발된 interpretability 연구를 위한 오픈 플랫폼인 [Neuronpedia](https://neuronpedia.org)과 밀접하게 통합되어 있으며, 우리는 이번 장 전반에 걸쳐 이를 사용할 것입니다. Neuronpedia를 통해 latent를 검색하고, SAE를 실행하며, 심지어 직접 만든 SAE를 업로드할 수도 있습니다!

SAE를 로드하기 전에, 어떤 것들이 사용 가능한지 확인하는 것이 유용할 수 있습니다. 다음 코드 스니펫은 현재 SAELens에서 사용 가능한 SAE 릴리스를 보여주며, SAELens에 더 많은 SAE가 추가됨에 따라 계속 최신 상태로 유지될 것입니다.

In [ ]:
print(get_pretrained_saes_directory())

이 모든 데이터를 더 읽기 쉬운 형식으로, 일부 속성만 선택하여 출력해 보겠습니다. `model` (base model), `release` (SAE release 이름), `repo_id` (SAE가 포함된 HuggingFace repo id), 그리고 각 release에 포함된 SAE의 개수(예를 들어, 하나의 release가 base model의 각 layer에서 학습된 SAE를 포함할 수 있습니다)를 살펴보겠습니다.

In [ ]:
metadata_rows = [
    [data.model, data.release, data.repo_id, len(data.saes_map)] for data in get_pretrained_saes_directory().values()
]

# Print all SAE releases, sorted by base model
print(
    tabulate(
        sorted(metadata_rows, key=lambda x: x[0]),
        headers=["model", "release", "repo_id", "n_saes"],
        tablefmt="simple_outline",
    )
)

제공되는 각 SAE 릴리스에는 여러 개의 서로 다른 모델이 포함되어 있을 수 있습니다. 이 모델들은 모델의 서로 다른 hookpoint나 layer에서 학습되었거나, 서로 다른 hyperparameter 등으로 학습되었을 수 있습니다. 각 릴리스와 관련된 데이터는 다음과 같이 확인할 수 있습니다:

In [ ]:
def format_value(value):
    return "{{{0!r}: {1!r}, ...}}".format(*next(iter(value.items()))) if isinstance(value, dict) else repr(value)


release = get_pretrained_saes_directory()["gpt2-small-res-jb"]

print(
    tabulate(
        [[k, format_value(v)] for k, v in release.__dict__.items()],
        headers=["Field", "Value"],
        tablefmt="simple_outline",
    )
)

각 릴리스와 관련된 SAE들에 대해 더 자세한 정보를 알아보겠습니다. SAE id, 경로(즉, SAE 모델 가중치를 가리키는 HuggingFace repo 내의 경로), 그리고 Neuronpedia ID(feature 대시보드를 확인하는 방법이며, 이에 대해서는 곧 더 자세히 설명하겠습니다)를 출력할 수 있습니다.

In [ ]:
data = [[id, path, release.neuronpedia_id[id]] for id, path in release.saes_map.items()]

print(
    tabulate(
        data,
        headers=["SAE id", "SAE path (HuggingFace)", "Neuronpedia ID"],
        tablefmt="simple_outline",
    )
)

다음으로, 대부분의 실습에서 사용할 SAE를 로드하겠습니다. 바로 **GPT2 Small SAEs**의 **layer 7 resid pre model**입니다 (또한 이를 연결할 GPT2 Small의 복사본도 함께 로드합니다). 이 SAE는 TransformerLens의 `HookedTransformer` 클래스에서 파생된 `HookedSAETransformer` 클래스를 사용합니다.

참고로, `SAE.from_pretrained` 함수는 SAE 객체를 직접 반환합니다. SAE의 config와 metadata는 각각 `sae.cfg`과 `sae.cfg.metadata`를 통해 접근할 수 있습니다.

In [ ]:
t.set_grad_enabled(False)

gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)

`sae` 객체는 `SAE` (Sparse Autoencoder) 클래스의 인스턴스입니다. SAE 아키텍처에는 서로 다른 가중치나 activation 함수를 가질 수 있는 다양한 종류가 있습니다. SAE 작업을 단순화하기 위해, SAELens가 이러한 복잡성의 대부분을 대신 처리합니다. 아래 셀을 실행하여 우리가 사용할 SAE의 각 config 파라미터를 확인할 수 있습니다.

<details>
<summary>각 SAE config 파라미터의 설명을 읽으려면 클릭하십시오.</summary>

1. `architecture`: 사용 중인 SAE 아키텍처의 유형을 지정합니다. 이 경우에는 표준 아키텍처(gated SAE와 달리 hidden activation이 있는 encoder와 decoder)입니다.
2. `d_in`: SAE의 입력 차원을 정의하며, 이 설정에서는 768입니다.
3. `d_sae`: SAE hidden layer의 차원을 설정하며, 여기서는 24576입니다. 이는 가능한 feature activation의 수를 나타냅니다.
4. `activation_fn_str`: SAE에서 사용되는 activation 함수를 지정하며, 이 경우에는 ReLU입니다. TopK는 여기서 다루지 않을 또 다른 옵션입니다.
5. `apply_b_dec_to_input`: 입력에 decoder bias를 적용할지 여부를 결정하며, 여기서는 True로 설정되어 있습니다.
6. `finetuning_scaling_factor`: 가중치 초기화 및 forward pass에 scaling factor를 사용할지 여부를 나타냅니다. 이는 보통 사용되지 않으며 [solution for shrinkage](https://www.lesswrong.com/posts/3JuSjTZyMzaSeTxKk/addressing-feature-suppression-in-saes)을 지원하기 위해 도입되었습니다.
7. `context_size`: context window의 크기를 정의하며, 이 경우에는 128 token입니다. 작은 프롬프트의 작은 activation으로 학습된 SAE가 [often don't perform well on longer prompts](https://www.lesswrong.com/posts/baJyjpktzmcmRfosq/stitching-saes-of-different-sizes) 것으로 나타났습니다.
8. `model_name`: 사용 중인 모델의 이름을 지정하며, 여기서는 'gpt2-small'입니다. [This is a valid model name in TransformerLens](https://transformerlensorg.github.io/TransformerLens/generated/model_properties_table.html).
9. `hook_name`: SAE가 적용되는 모델 내의 특정 hook을 나타냅니다.
10. `hook_layer`: hook이 적용되는 레이어 번호를 지정하며, 이 경우에는 layer 7입니다.
11. `hook_head_index`: 어떤 attention head에 hook을 걸지 정의합니다. 여기서는 residual stream SAE를 살펴보고 있으므로 해당되지 않습니다.
12. `prepend_bos`: beginning-of-sequence token을 앞에 추가할지 여부를 결정하며, True로 설정되어 있습니다.
13. `dataset_path`: 학습 또는 평가에 사용되는 데이터셋의 경로를 지정합니다. (로컬 경로이거나 huggingface 데이터셋일 수 있습니다.)
14. `dataset_trust_remote_code`: 데이터셋을 로드할 때 (HuggingFace의) 원격 코드를 신뢰할지 여부를 나타내며, True로 설정되어 있습니다.
15. `normalize_activations`: activation을 정규화하는 방법을 지정하며, 이 config에서는 'none'으로 설정되어 있습니다.
16. `dtype`: tensor 연산을 위한 데이터 타입을 정의하며, 32-bit floating point로 설정되어 있습니다.
17. `device`: 사용할 계산 장치(device)를 지정합니다.
18. `sae_lens_training_version`: 학습에 사용된 SAE Lens의 버전을 나타내며, 여기서는 None으로 설정되어 있습니다.
19. `activation_fn_kwargs`: activation 함수를 위한 추가 키워드 인자를 허용합니다. 예를 들어 `activation_fn_str`이 `topk`로 설정되어 `k`를 지정해야 하는 경우에 사용됩니다.

</details>

In [ ]:
print(
    tabulate(
        list(gpt2_sae.cfg.__dict__.items()) + list(gpt2_sae.cfg.metadata.items()),
        headers=["name", "value"],
        tablefmt="simple_outline",
    )
)

## 대시보드를 이용한 SAE 시각화

> 이 섹션에서는 특정 SAE latent가 무엇을 나타내는지 빠르게 이해하기 위한 시각적 도구인 SAE 대시보드에 대해 배웁니다. 핵심 포인트는 다음과 같습니다:
>
> - Neuronpedia는 SAE latent를 이해하는 데 도움이 되는 대시보드를 호스팅합니다.
> - 대시보드의 5가지 주요 구성 요소는 top logit 테이블, logits 히스토그램, activation 밀도 플롯, top activating 시퀀스, 그리고 autointerp입니다.
> - 이 모든 구성 요소는 latent가 무엇을 나타내는지 전체적인 그림을 파악하는 데 중요하지만, 동시에 모두 오해의 소지가 있을 수 있습니다.
> - `IFrame`을 사용하여 이러한 대시보드를 인라인으로 표시할 수 있습니다.

이 섹션에서는 SAE를 살펴보고, 그것들이 실제로 우리에게 무엇을 알려주는지 확인해 보겠습니다.

하지만 너무 깊이 들어가기 전에, 한 가지를 다시 짚어보겠습니다. SAE latent란 실제로 무엇일까요?

SAE latent는 SAE가 학습한 베이스 모델의 activation 공간 내의 **특정 방향**입니다. 종종 이것들은 데이터의 **feature**에 대응합니다. 다시 말해, 베이스 모델이 학습한 데이터 분포에 존재하며 베이스 모델에 의해 학습된, 의미 있는 세만틱, 구문론적 또는 기타 해석 가능한 패턴이나 개념을 의미합니다. 이러한 feature들은 보통 매우 sparse합니다. 즉, 어떤 주어진 feature에 대해 전체 데이터 분포 중 아주 적은 일부만이 해당 feature를 activate합니다. 일반적으로 더 sparse한 feature일수록 더 해석 가능성이 높은 경향이 있습니다.

**참고 - 엄밀히 말하면 여기서 "방향"이라고 말하는 것은 지나친 단순화입니다. 왜냐하면 주어진 latent는 activation 공간에서 여러 개의 방향과 연관될 수 있기 때문입니다. 예를 들어, 표준 untied SAE의 경우 별도의 encoder 및 decoder 방향이 존재합니다. 우리가 latent 방향 또는 feature 방향이라고 언급할 때는 보통(항상 그런 것은 아니지만) decoder weight를 의미합니다.

아래에 표시된 대시보드는 단일 SAE latent에 대한 상세한 뷰를 제공합니다.

In [ ]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=800,
    height=600,
):
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))


latent_idx = random.randint(0, gpt2_sae.cfg.d_sae)
display_dashboard(latent_idx=latent_idx)

시각화의 개별 구성 요소들을 자세히 살펴보겠습니다:

1. **Latent Activation Distribution**. 이는 latent가 활성화되는 token의 비율을 보여주며, 보통 0.01%에서 1% 사이입니다. 또한 양수 activation의 분포도 함께 보여줍니다.
2. **Logits Distribution**. 이는 decoder weight를 unembed에 투영한 것이며, 대략적으로 latent에 의해 촉진되는 token들에 대한 감을 제공합니다. 큰 모델이나 중간 layer에서는 유용성이 떨어집니다.
3. **Top / Bottom Logits**. 이는 logit weight 분포에서 가장 양수 값인 10개와 가장 음수 값인 10개의 logit입니다.
4. **Max Activating Examples**. 이는 latent가 활성화되는 텍스트 예시들이며, 보통 latent가 무엇을 의미하는지 파악하는 데 가장 많은 정보를 제공합니다.
5. **Autointerp**. 이는 LLM이 생성한 latent 설명으로, 대시보드의 나머지 데이터(특히 max activating examples)를 사용합니다.

더 자세한 정보는 [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features#setup-interface)의 이 섹션을 참조하시기 바랍니다.

*Neuronpedia*는 SAE 대시보드를 호스팅하고, 모델을 실행하여 latent activation을 확인할 수 있는 서버를 운영하는 웹사이트입니다. 이를 통해 latent가 실제로 활성화되어야 한다고 생각하는 텍스트 분포에서 활성화되는지 매우 편리하게 확인할 수 있습니다. 위 대시보드들을 위해 Neuronpedia에서 데이터를 다운로드하여 사용해 왔습니다.

### 연습 문제 - 흥미로운 latent 찾기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-15 minutes on this exercise.
> ```

SAE 대시보드를 살펴보며 시간을 보내십시오 (즉, 서로 다른 무작위 인덱스로 코드를 실행해 보십시오). 어떤 흥미로운 latent를 찾을 수 있습니까? 다음과 같은 유형의 latent를 찾아보십시오:

- **token-level feature**를 위한 latent: 특정 token에서만 활성화되고 다른 token에서는 거의 활성화되지 않는 것처럼 보이는 것들입니다. top logit들을 bigram 빈도로 보았을 때 타당해 보입니까?
- **concept-level feature**를 위한 latent: 단일 token이 아니라, 텍스트에 특정 개념이 존재할 때 여러 token에 걸쳐 활성화되는 것들입니다 (예를 들어 `latent_idx=4527` 이 이에 해당합니다). 이 latent가 나타내는 개념은 무엇입니까? 이 token에 대한 positive logit들이 어떻게 타당한지 알 수 있습니까?
- **매우 희소한(Highly sparse) latent**: activation density가 0.05% 미만인 것들입니다. 평균적인 latent보다 더 해석 가능해 보입니까?

<details>
<summary>각 유형의 예시를 확인하려면 이 드롭다운을 클릭하십시오. 여러분이 찾은 것들과 비교해 볼 수 있습니다.</summary>

Latent 9은 새로운 것들의 복수형을 설명하는 문맥(종종 정책, 비즈니스 추정치, 채용 등과 관련됨)에서 "new"라는 단어에만 활성화되는 것으로 보입니다. "new arrivals"나 "new developments"와 같은 bigram들이 boost 되어 positive logit들이 이를 뒷받침합니다. 흥미롭게도 "newbie(s)"라는 bigram 또한 boost 되었습니다.

Latent 67은 더 concept-level인 것으로 보이며, 특정 국가가 특정 정책이나 결정을 시행하기로 한 결정에 대해 이야기하는 구절(특히 해당 국가가 그렇게 하는 유일한 국가로 묘사될 때)에서 활성화됩니다. positive logit들 또한 국가나 정부 정책과 관련이 있지만, bigram으로서 직접적으로 타당하지는 않습니다. 이는 해당 latent가 문장에 질문 대상이 되는 개념이 포함되어 있을 때 문장 내의 여러 서로 다른 token에서 활성화되기 때문에 예상할 수 있는 결과입니다.

Latent 13은 0.049%의 빈도로 활성화됩니다. 이는 "win"이라는 token, 특히 사람들의 마음을 얻는 문맥(예: 대통령 당선, 마음과 정신을 얻음, 또는 논쟁에서 이김)이나 경주에서 이기는 문맥에서 활성화되는 것으로 보입니다. 이는 상당히 구체적이고 해석 가능해 보이지만, latent를 해석할 때는 **interpretability illusion**을 기억하는 것이 중요합니다. 즉, top activating pattern을 보는 것만으로 특정 해석에 대해 잘못된 확신을 가질 수 있습니다. 이후 섹션에서는 latent에 대한 이해를 정교화하기 위해 더 신중한 가설 검정을 수행할 것입니다.

</details>

## SAE 실행하기

> 이 섹션에서는 SAE를 사용하여 forward pass를 실행하는 방법을 배웁니다. 이는 TransformerLens 모델의 기존 인프라를 많이 활용하므로 매우 간단한 과정입니다. 핵심 사항은 다음과 같습니다:
>
> - TransformerLens 모델에서 forward pass를 수행할 때, hook 함수를 추가하는 것과 거의 동일한 방식으로 SAE를 추가할 수 있습니다 (SAE를 일종의 특수한 hook 함수라고 생각하시면 됩니다).
> - `sae.error_term=False` (기본값)일 때는 transformer activation을 SAE의 출력값으로 대체합니다. True일 때는 대체하지 않습니다 (이는 activation을 캐싱할 때 가끔 필요한 설정입니다).
> - `run_with_hooks`와 유사하게 작동하는 `run_with_saes`이 있습니다.
> - `run_with_cache`와 유사하게 작동하지만, 원하는 모든 SAE activation을 캐싱할 수 있게 해주는 `run_with_cache_with_saes`도 있습니다.
> - `ActivationStore`를 사용하여 대량의 activation 배치를 한 번에 가져올 수 있습니다.

Neuronpedia를 통해 몇 가지 SAE를 살펴보았으니, 이제 직접 SAE를 로드하고 실행해 볼 시간입니다!

`HookedSAETransformer`의 핵심 기능 중 하나는 SAE를 "splice in" 하여 model activation을 SAE reconstruction으로 대체할 수 있다는 점입니다. SAE가 연결된 상태로 forward pass를 실행하려면 `model.run_with_saes(tokens, saes=[list_of_saes])`을 사용할 수 있습니다. 이 함수는 표준 forward pass(또는 `model.run_with_hooks`)와 유사한 구문을 가지며, 예를 들어 return type을 loss로 할지 logit으로 할지 지정하기 위해 `return_type`과 같은 인자를 받을 수 있습니다. 연결된 SAE는 forward pass 직후에 즉시 리셋되어 모델을 원래 상태로 되돌립니다. 내부적으로는 TransformerLens에서 hook을 추가하는 것과 동일하게 작동하며, 다만 이 경우에는 hook이 "이 activation들을 SAE reconstruction으로 대체하라"는 역할을 수행합니다.

일반적인 hooked forward pass를 수행하는 여러 방법과 마찬가지로, SAE-hooked forward pass를 수행하는 다양한 방법이 있습니다. 예를 들어, `with model.hooks(fwd_hooks=...)`를 context manager로 사용하여 일시적으로 hook을 추가할 수 있는 것처럼, `with model.saes(saes=...)`를 사용하여 SAE가 연결된 상태로 forward pass를 실행할 수 있습니다. 또한 `model.add_hook`과 `model.reset_hooks`을 사용할 수 있는 것처럼, `model.add_sae`과 `model.reset_saes`도 사용할 수 있습니다.

In [ ]:
prompt = "Mitigating the risk of extinction from AI should be a global"
answer = " priority"

# First see how the model does without SAEs
test_prompt(prompt, answer, gpt2)

# Test our prompt, to see what the model says
with gpt2.saes(saes=[gpt2_sae]):
    test_prompt(prompt, answer, gpt2)

# Same thing, done in a different way
gpt2.add_sae(gpt2_sae)
test_prompt(prompt, answer, gpt2)
gpt2.reset_saes()  # Remember to always do this!

# Using `run_with_saes` method in place of standard forward pass
logits = gpt2(prompt, return_type="logits")
logits_sae = gpt2.run_with_saes(prompt, saes=[gpt2_sae], return_type="logits")
answer_token_id = gpt2.to_single_token(answer)

# Getting model's prediction
top_prob, token_id_prediction = logits[0, -1].softmax(-1).max(-1)
top_prob_sae, token_id_prediction_sae = logits_sae[0, -1].softmax(-1).max(-1)

print(f"""Standard model:
    top prediction = {gpt2.to_string(token_id_prediction)!r}
    prob = {top_prob.item():.2%}
SAE reconstruction:
    top prediction = {gpt2.to_string(token_id_prediction_sae)!r}
    prob = {top_prob_sae.item():.2%}
""")

모델의 출력을 SAE 출력으로 대체하여 forward pass를 수행하려는 경우에는 이 방법이 적절하지만, 단순히 SAE activation만 얻고 싶다면 어떻게 해야 할까요? 바로 이때 cache와 함께 실행하는 방법이 필요합니다! `HookedSAETransformer`을 사용하면 `logits, cache = model.run_with_cache_with_saes(tokens, saes=saes)`을 통해 SAE activation(및 다른 모든 표준 activation)을 cache에 저장할 수 있습니다. `run_with_saes`가 표준 forward pass의 wrapper인 것처럼, `run_with_cache_with_saes`은 `run_with_cache`의 wrapper이며, 모델을 원래 상태로 되돌리기 전 단 한 번의 forward pass 동안에만 이러한 SAE들을 추가합니다.

cache에서 SAE activation에 접근하려면, 해당 hook 이름은 일반적으로 HookedTransformer `hook_name` (예: `"blocks.5.attn.hook_z"`)와 SAE hook 이름 (예: `"hook_sae_acts_post"`)을 마침표로 연결한 형태가 됩니다. 아래에서 모든 이름을 출력해 보겠습니다:

In [ ]:
_, cache = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

for name, param in cache.items():
    if "hook_sae" in name:
        print(f"{name:<43}: {tuple(param.shape)}")

`run_with_cache_with_saes`을 사용하면 어떤 input에서 어떤 SAE latent가 활성화되는지 쉽게 탐색할 수 있습니다. 또한, SAE layer 이후의 activation은 계산할 필요가 없으므로, forward pass에서 `stop_at_layer` 인자와 함께 이를 사용할 수 있습니다.

프롬프트의 마지막 token에서 활성화된 latent를 탐색해 보겠습니다. 첫 번째 latent는 "global"이라는 단어, 특히 "global warming", "global poverty", "global war"와 같은 재난의 문맥에서 활성화되는 것을 확인할 수 있습니다.

In [ ]:
# Get top activations on final token
_, cache = gpt2.run_with_cache_with_saes(
    prompt,
    saes=[gpt2_sae],
    stop_at_layer=_get_hook_layer(gpt2_sae) + 1,
)
sae_acts_post = cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, :]

# Plot line chart of latent activations
px.line(
    sae_acts_post.cpu().numpy(),
    title=f"Latent activations at the final token position ({sae_acts_post.nonzero().numel()} alive)",
    labels={"index": "Latent", "value": "Activation"},
    width=1000,
).update_layout(showlegend=False).show()

# Print the top 5 latents, and inspect their dashboards
for act, ind in zip(*sae_acts_post.topk(3)):
    print(f"Latent {ind} had activation {act:.2f}")
    display_dashboard(latent_idx=ind)

### Error term

중요한 참고 사항 - 파라미터 `sae.use_error_term`은 SAE forward pass 동안 실제로 activation을 SAE reconstruction으로 대체할지 여부를 결정합니다. 만약 `False` (기본값)이라면 activation을 SAE reconstruction으로 대체하지만, `True`라면 transformer activation을 출력값으로 대체하지 않고 SAE의 hidden activation만 계산합니다.

`use_error_term` 파라미터는 forward pass, hooked forward pass, cache를 사용하는 forward pass, 또는 SAE가 부착된 상태로 모델을 실행하는 모든 동작의 거동을 제어합니다 (하지만 당연히 이 파라미터는 값을 caching할 때만 중요합니다. `sae.use_error_term=True` 상태로 forward pass를 수행하면서 값을 caching하지 않는 것은 SAE 없이 기본 모델만 실행하는 것과 동일하기 때문입니다!).

<details>
<summary>왜 <code>use_error_term</code>라고 부르나요?</summary>

True로 설정했을 때 forward pass의 최종 출력이 `sae_out` 대신 `sae_out + sae_error`이 되기 때문에 이렇게 부릅니다. 이 `sae_error` 항은 말 그대로 `sae_in - sae_out`, 즉 원래의 입력과 SAE reconstruction 사이의 차이로 정의됩니다. 따라서 이는 SAE가 단순히 identity function인 것과 동일합니다. 하지만 우리는 transformer의 activation을 SAE reconstruction으로 실제로 대체할 때와 정확히 동일한 방식으로 SAE의 모든 내부 상태를 계산해야 하므로, 이러한 방식을 사용해야 합니다.

</details>

In [ ]:
logits_no_saes, cache_no_saes = gpt2.run_with_cache(prompt)

gpt2_sae.use_error_term = False
logits_with_sae_recon, cache_with_sae_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

gpt2_sae.use_error_term = True
logits_without_sae_recon, cache_without_sae_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

# Both SAE caches contain the hook values
assert f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post" in cache_with_sae_recon
assert f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post" in cache_without_sae_recon

# But final output will be different, because we don't use SAE reconstructions when use_error_term
t.testing.assert_close(logits_no_saes, logits_without_sae_recon)
logit_diff_from_sae = (logits_no_saes - logits_with_sae_recon).abs().mean()
print(f"Average logit diff from using SAE reconstruction: {logit_diff_from_sae:.4f}")

### `ActivationStore` 사용하기

`ActivationsStore` 클래스는 많은 양의 데이터를 직접 로드하는 대신 사용할 수 있는 편리한 대안입니다. 이 클래스는 주어진 데이터셋에서 데이터를 스트리밍합니다. `from_sae` 클래스의 경우, 해당 데이터셋은 SAE의 config에 의해 제공됩니다 (이는 SAE의 원래 학습 데이터셋과 동일합니다).

In [ ]:
print(gpt2_sae.cfg.dataset_path)

이제 하나를 로드해 보겠습니다. 메모리 부족 현상 없이 사용할 수 있도록 여기서는 상당히 보수적인 파라미터를 사용하지만, 가능하다면 이 파라미터들을 늘리셔도 좋습니다 (또는 여전히 메모리가 부족하다면 더 줄이셔도 됩니다).

In [ ]:
gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_sae,
    dataset="NeelNanda/pile-10k",
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=str(device),
)

# Example of how you can use this:
tokens = gpt2_act_store.get_batch_tokens()
assert tokens.shape == (gpt2_act_store.store_batch_size_prompts, gpt2_act_store.context_size)

## SAE 대시보드 재현하기

> 이 섹션에서는 SAE 대시보드의 5가지 주요 구성 요소인 top logits 테이블, logits 히스토그램, activation 밀도 플롯, top activating 시퀀스, 그리고 autointerp를 재현합니다. 여기에는 새로운 내용이 거의 없으며, 이전 두 섹션인 "Visualizing SAEs with dashboards"와 "Running SAEs"에서 배운 내용을 실제로 적용해 보는 과정입니다.

이제 SAE를 로드하고 실행하는 방법을 알았으므로, SAE 대시보드의 구성 요소들을 차례대로 구현해 보겠습니다. 이 실습들은 SAE를 실행하고 activation을 다루는 경험을 쌓는 데 도움이 될 것이며, 대시보드의 각 구성 요소가 갖는 의미와 중요성을 더 깊이 이해하는 데 도움을 줄 것입니다.

복습하자면, 기본적인 SAE 대시보드는 5가지 주요 구성 요소로 이루어져 있습니다:

1. **Activation Distribution** - latent activation의 분포
2. **Logits Distribution** - decoder weight를 모델의 unembedding에 투영한 결과
3. **Top / Bottom Logits** - logit weight 분포에서 가장 양수 값과 가장 음수 값을 갖는 logit들
4. **Max Activating Examples** - 해당 latent가 가장 강하게 활성화되는 시퀀스(및 특정 token)
5. **Autointerp** - LLM이 생성한 latent 설명

이제 이 요소들을 하나씩 살펴보겠습니다. 이번 실습에서는 latent `9`를 사용합니다. 여러분의 결과를 예상 대시보드와 비교해 보시기 바랍니다:

In [ ]:
display_dashboard(latent_idx=9)

### 연습 문제 - activation 분포 구하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

아래 함수는 일정 수의 batch를 반복하며(코드가 너무 오래 걸린다고 판단되면 기본 숫자를 줄이셔도 됩니다), 주어진 latent에 대한 activation의 histogram을 생성해야 합니다. histogram의 제목에 **activation density**도 함께 반환하도록 시도해 보십시오.

참고 - `model.run_with_cache_with_saes`을 사용할 때, `stop_at_layer=_get_hook_layer(sae)+1`와 `names_filter=hook_name` 인자를 사용할 수 있습니다. 이를 통해 불필요한 계산과 메모리 사용을 피할 수 있습니다.

또한, Colab에서 Plotly를 사용하는 경우, figure를 별도의 코드 셀에서 계산하고 렌더링하도록 코드를 수정해야 할 수도 있습니다. 이는 잘 알려져 있지만 아직 수정되지 않은 Colab의 버그입니다.

In [ ]:
def show_activation_histogram(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 200,
):
    """
    Displays the activation histogram for a particular latent, computed across `total_batches`
    batches from `act_store`.
    """
    raise NotImplementedError()


show_activation_histogram(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9)

<details>
<summary>히스토그램을 그리기 위한 코드는 여기를 클릭하십시오 (이 부분이 연습 문제의 일부가 되는 것에 크게 신경 쓰지 않으신다면)</summary>

이 코드는 `all_positive_acts`이 모든 batch에 걸친 0이 아닌 모든 activation 값들의 리스트라고 가정할 때 작동합니다:

```python
frac_active = len(all_positive_acts) / (
    total_batches * act_store.store_batch_size_prompts * act_store.context_size
)

px.histogram(
    all_positive_acts,
    nbins=50,
    title=f"ACTIVATIONS DENSITY {frac_active:.3%}",
    labels={"value": "Activation"},
    width=800,
    template="ggplot2",
    color_discrete_sequence=["darkorange"],
).update_layout(bargap=0.02, showlegend=False).show()
```

Colab을 사용 중이라면, 이 figure를 반환하고 별도의 셀에서 plot해야 할 수도 있습니다 (Colab은 계산을 수행하는 동일한 셀에서 plot하는 방식이 다소 특이하기 때문입니다).

</details>


<details><summary>정답</summary>

```python
def show_activation_histogram(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 200,
):
    """
    Displays the activation histogram for a particular latent, computed across `total_batches`
    batches from `act_store`.
    """
    sae_acts_post_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"
    all_positive_acts = []

    for i in tqdm(range(total_batches), desc="Computing activations for histogram"):
        tokens = act_store.get_batch_tokens()
        _, cache = model.run_with_cache_with_saes(
            tokens,
            saes=[sae],
            stop_at_layer=_get_hook_layer(sae) + 1,
            names_filter=[sae_acts_post_hook_name],
        )
        acts = cache[sae_acts_post_hook_name][..., latent_idx]
        all_positive_acts.extend(acts[acts > 0].cpu().tolist())

    frac_active = len(all_positive_acts) / (total_batches * act_store.store_batch_size_prompts * act_store.context_size)

    px.histogram(
        all_positive_acts,
        nbins=50,
        title=f"ACTIVATIONS DENSITY {frac_active:.3%}",
        labels={"value": "Activation"},
        width=800,
        template="ggplot2",
        color_discrete_sequence=["darkorange"],
    ).update_layout(bargap=0.02, showlegend=False).show()
```
</details>

### 연습 문제 - 최대 활성화 예시 찾기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

먼저 max-activating examples, 즉 latent에서 가장 높은 수준의 activation을 보이는 prompt들을 찾는 것부터 시작하겠습니다. 완성해야 할 docstring이 포함된 함수를 제공해 드렸으며, 데이터를 어떻게 제시할지는 전적으로 여러분의 선택에 달려 있습니다.

다음과 같은 helper 함수들과 이를 사용하는 예시를 제공해 드렸습니다:

- `get_k_largest_indices`: `(batch, seq)` 크기의 tensor에서 가장 큰 k개 요소의 batch 및 seqpos 인덱스를 반환합니다.
- `index_with_buffer`: `get_k_largest_indices`의 결과물을 사용하여 `(batch, seq)` 크기의 tensor를 인덱싱하며, 선택된 인덱스와 동일한 sequence 내에서 `buffer` 범위의 token들을 포함합니다 (이는 선택된 token 주변의 context를 파악하는 데 도움이 됩니다).
- `display_top_seqs`: sequence(관련 token이 강조됨)를 읽기 쉬운 방식으로 표시합니다.

sequence를 디코딩할 때는 `model.to_str_tokens`을 사용하여 token ID의 1D tensor를 문자열 token 리스트로 매핑할 수 있습니다. 출력 결과에 일부 알 수 없는 token `"�"`이 포함될 수 있다는 점에 유의하십시오. 이는 우리가 사용하는 tokenization 방식의 불가피한 부산물이며, 크게 걱정하지 않으셔도 됩니다.

In [ ]:
def get_k_largest_indices(x: Float[Tensor, "batch seq"], k: int, buffer: int = 0) -> Int[Tensor, "k 2"]:
    """
    The indices of the top k elements in the input tensor, i.e. output[i, :] is the (batch, seqpos)
    value of the i-th largest element in x.

    Won't choose any elements within `buffer` from the start or end of their sequence.
    """
    if buffer > 0:
        x = x[:, buffer:-buffer]
    indices = x.flatten().topk(k=k).indices
    rows = indices // x.size(1)
    cols = indices % x.size(1) + buffer
    return t.stack((rows, cols), dim=1)


x = t.arange(40, device=device).reshape((2, 20))
x[0, 10] += 50  # 2nd highest value
x[0, 11] += 100  # highest value
x[1, 1] += 150  # not inside buffer (it's less than 3 from the start of the sequence)
top_indices = get_k_largest_indices(x, k=2, buffer=3)
assert top_indices.tolist() == [[0, 11], [0, 10]]


def index_with_buffer(
    x: Float[Tensor, "batch seq"], indices: Int[Tensor, "k 2"], buffer: int | None = None
) -> Float[Tensor, " k *buffer_x2_plus1"]:
    """
    Indexes into `x` with `indices` (which should have come from the `get_k_largest_indices`
    function), and takes a +-buffer range around each indexed element. If `indices` are less than
    `buffer` away from the start of a sequence then we just take the first `2*buffer+1` elems (same
    for at the end of a sequence).

    If `buffer` is None, then we don't add any buffer and just return the elements at the given indices.
    """
    rows, cols = indices.unbind(dim=-1)
    if buffer is not None:
        rows = einops.repeat(rows, "k -> k buffer", buffer=buffer * 2 + 1)
        cols[cols < buffer] = buffer
        cols[cols > x.size(1) - buffer - 1] = x.size(1) - buffer - 1
        cols = einops.repeat(cols, "k -> k buffer", buffer=buffer * 2 + 1) + t.arange(
            -buffer, buffer + 1, device=cols.device
        )
    return x[rows, cols]


x_top_values_with_context = index_with_buffer(x, top_indices, buffer=3)
assert x_top_values_with_context[0].tolist() == [
    8,
    9,
    10 + 50,
    11 + 100,
    12,
    13,
    14,
]  # highest value in the middle
assert x_top_values_with_context[1].tolist() == [
    7,
    8,
    9,
    10 + 50,
    11 + 100,
    12,
    13,
]  # 2nd highest value in the middle


def display_top_seqs(data: list[tuple[float, list[str], int]]):
    """
    Given a list of (activation: float, str_toks: list[str], seq_pos: int), displays a table of
    these sequences, with the relevant token highlighted.

    We also turn newlines into "\\n", and remove unknown tokens � (usually weird quotation marks)
    for readability.
    """
    table = Table("Act", "Sequence", title="Max Activating Examples", show_lines=True)
    for act, str_toks, seq_pos in data:
        formatted_seq = (
            "".join([f"[b u green]{str_tok}[/]" if i == seq_pos else str_tok for i, str_tok in enumerate(str_toks)])
            .replace("�", "")
            .replace("\n", "↵")
        )
        table.add_row(f"{act:.3f}", repr(formatted_seq))
    rprint(table)


example_data = [
    (0.5, [" one", " two", " three"], 0),
    (1.5, [" one", " two", " three"], 1),
    (2.5, [" one", " two", " three"], 2),
]
display_top_seqs(example_data)

다음 함수를 완성해야 합니다. 이 함수는 `data`를 `(max activation, list of string tokens, sequence position)` 형태의 튜플 리스트로 반환해야 하며, 만약 `display`가 True라면 이 데이터에 대해 `display_top_seqs` 또한 호출해야 합니다 (나중에 autointerp를 구현할 때 이 함수가 유용할 것입니다!).

In [ ]:
def fetch_max_activating_examples(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 10,
    buffer: int = 10,
) -> list[tuple[float, list[str], int]]:
    """
    Returns the max activating examples across a number of batches from the activations store.
    """
    raise NotImplementedError()


# Fetch & display the results
buffer = 10
data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9, buffer=buffer, k=5)
display_top_seqs(data)

# Test one of the results, to see if it matches the expected output
first_seq_str_tokens = data[0][1]
assert first_seq_str_tokens[buffer] == " new"

<details><summary>솔루션</summary>

```python
def fetch_max_activating_examples(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 10,
    buffer: int = 10,
) -> list[tuple[float, list[str], int]]:
    """
    Returns the max activating examples across a number of batches from the activations store.
    """
    sae_acts_post_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"

    # Create list to store the top k activations for each batch. Once we're done,
    # we'll filter this to only contain the top k over all batches
    data = []

    for _ in tqdm(range(total_batches), desc="Computing activations for max activating examples"):
        tokens = act_store.get_batch_tokens()
        _, cache = model.run_with_cache_with_saes(
            tokens,
            saes=[sae],
            stop_at_layer=_get_hook_layer(sae) + 1,
            names_filter=[sae_acts_post_hook_name],
        )
        acts = cache[sae_acts_post_hook_name][..., latent_idx]

        # Get largest indices, get the corresponding max acts, and get the surrounding indices
        k_largest_indices = get_k_largest_indices(acts, k=k, buffer=buffer)
        tokens_with_buffer = index_with_buffer(tokens, k_largest_indices, buffer=buffer)
        str_toks = [model.to_str_tokens(toks) for toks in tokens_with_buffer]
        top_acts = index_with_buffer(acts, k_largest_indices).tolist()
        data.extend(list(zip(top_acts, str_toks, [buffer] * len(str_toks))))

    return sorted(data, key=lambda x: x[0], reverse=True)[:k]
```
</details>

#### 겹치지 않는 시퀀스

위의 latent의 경우, 앞서 수행한 방식대로 시퀀스를 반환하는 것이 아마 꽤 잘 작동했을 것입니다. 하지만 더 개념적인 수준의 latent(문장 내 여러 token이 강하게 활성화되는 경우)는 조금 더 까다롭습니다. `16873`(특정 성경 구절에서 활성화됨)와 같은 latent에 이 함수를 시도해 보십시오. 반환된 시퀀스들은 대부분 동일하며, 단지 서로 다른 양만큼 밀려나 있을 것입니다.

In [ ]:
data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=16873, total_batches=200)
display_top_seqs(data)

이를 해결하는 한 가지 방법은 특정 top-activating token이 단 하나의 sequence에만 포함될 수 있도록 제한을 두는 것입니다. 즉, 해당 token을 선택하면 그 주변 범위 `[-buffer, buffer]` 내의 다른 token은 선택할 수 없게 하는 것입니다. 아래에 새로운 함수 `get_k_largest_indices` 를 제공해 드렸습니다. `no_overlap=True` 로 시도해 보십시오. 결과가 훨씬 더 좋아졌습니까?

In [ ]:
def get_k_largest_indices(
    x: Float[Tensor, "batch seq"],
    k: int,
    buffer: int = 0,
    no_overlap: bool = True,
) -> Int[Tensor, "k 2"]:
    """
    Returns the tensor of (batch, seqpos) indices for each of the top k elements in the tensor x.

    Args:
        buffer:     We won't choose any elements within `buffer` from the start or end of their seq
                    (this helps if we want more context around the chosen tokens).
        no_overlap: If True, this ensures that no 2 top-activating tokens are in the same seq and
                    within `buffer` of each other.
    """
    assert buffer * 2 < x.size(1), "Buffer is too large for the sequence length"
    assert not no_overlap or k <= x.size(0), "Not enough sequences to have a different token in each sequence"

    if buffer > 0:
        x = x[:, buffer:-buffer]

    indices = x.flatten().argsort(-1, descending=True)
    rows = indices // x.size(1)
    cols = indices % x.size(1) + buffer

    if no_overlap:
        unique_indices = t.empty((0, 2), device=x.device).long()
        while len(unique_indices) < k:
            unique_indices = t.cat((unique_indices, t.tensor([[rows[0], cols[0]]], device=x.device)))
            is_overlapping_mask = (rows == rows[0]) & ((cols - cols[0]).abs() <= buffer)
            rows = rows[~is_overlapping_mask]
            cols = cols[~is_overlapping_mask]
        return unique_indices

    return t.stack((rows, cols), dim=1)[:k]


x = t.arange(40, device=device).reshape((2, 20))
x[0, 10] += 150  # highest value
x[0, 11] += 100  # 2nd highest value, but won't be chosen because of overlap
x[1, 10] += 50  # 3rd highest, will be chosen
top_indices = get_k_largest_indices(x, k=2, buffer=3)
assert top_indices.tolist() == [[0, 10], [1, 10]]


data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=16873, total_batches=200)
display_top_seqs(data)

### 연습 문제 - 상위 / 하위 logit 가져오기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

마지막으로 상위 및 하위 logit 테이블을 만들어 보겠습니다. 이 테이블들은 SAE와 모델의 weight에 대한 함수일 뿐이므로 별도의 데이터가 필요하지 않습니다. 기본 모델의 unembedding은 `model.W_U`을 통해 접근할 수 있으며, SAE의 decoder weight는 `sae.W_dec`를 통해 접근할 수 있다는 점을 기억하시기 바랍니다.

In [ ]:
def show_top_logits(
    model: HookedSAETransformer,
    sae: SAE,
    latent_idx: int,
    k: int = 10,
) -> None:
    """
    Displays the top & bottom logits for a particular latent.
    """
    raise NotImplementedError()


show_top_logits(gpt2, gpt2_sae, latent_idx=9)
tests.test_show_top_logits(show_top_logits, gpt2, gpt2_sae)

<details><summary>솔루션</summary>

```python
def show_top_logits(
    model: HookedSAETransformer,
    sae: SAE,
    latent_idx: int,
    k: int = 10,
) -> None:
    """
    Displays the top & bottom logits for a particular latent.
    """
    logits = sae.W_dec[latent_idx] @ model.W_U

    pos_logits, pos_token_ids = logits.topk(k)
    pos_tokens = model.to_str_tokens(pos_token_ids)
    neg_logits, neg_token_ids = logits.topk(k, largest=False)
    neg_tokens = model.to_str_tokens(neg_token_ids)

    print(
        tabulate(
            zip(map(repr, neg_tokens), neg_logits, map(repr, pos_tokens), pos_logits),
            headers=["Bottom tokens", "Value", "Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )
```
</details>

### Exercise - autointerp

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should consider skipping this exercise / reading the solution unless you're really interested in autointerp.
> ```

자동화된 interpretability(automated interpretability)는 현재 특히 흥미로운 연구 분야 중 하나입니다. 이는 OpenAI의 논문 [Language models can explain neurons in language models](https://openai.com/index/language-models-can-explain-neurons-in-language-models/) 에서 시작되었으며, 해당 논문은 GPT-2의 neuron 하나를 가져와 관련 텍스트 시퀀스와 activation을 GPT-4에 보여줌으로써 그 동작에 대한 설명을 생성할 수 있음을 보여주었습니다. 이는 인간이 수동으로 검토하고 레이블을 지정할 필요 없이(이는 규모를 확장하기에 분명히 완전히 비실용적입니다), 대규모로 latent를 평가하고 분류할 수 있는 한 가지 가능한 방법을 제시합니다. [here](https://blog.eleuther.ai/autointerp/) 및 [here](https://www.lesswrong.com/posts/8ev6coxChSWcxCDy8/self-explaining-sae-features) 에서 이루어진 최근의 추가적인 발전 사항들에 대해서도 읽어보실 수 있습니다.

이러한 확장 가능하고 효율적인 분류는 하나의 유스케이스이며, 두 번째 유스케이스인 **SAE evaluations**가 있습니다. 이는 이후 섹션에서 더 자세히 다룰 주제이지만, 지금 요약하자면 다음과 같습니다. SAE evaluations는 "우리의 SAE가 얼마나 좋은지"를 다양한 방식으로 측정하는 방법입니다. 하지만 우리가 선택하는 어떤 metric이든 **Goodhearting**에 취약하며, 우리가 SAE에서 원하는 바를 반드시 대표하지는 않기 때문에 이는 매우 어려운 작업임이 밝혀졌습니다. 예를 들어, (reconstruction loss가 일정하다면) 더 sparse한 latent가 종종 더 interpretable한 것처럼 보이며, 이것이 SAE를 평가하는 일반적인 방법이 **sparsity와 reconstruction loss의 Pareto frontier**를 따르는 이유입니다. 하지만 만약 sparsity가 더 interpretable한 latent로 이어지지 않는다면 어떻게 될까요 (예: [feature absorption](https://www.lesswrong.com/posts/3zBsxeZzd3cvuueMJ/paper-a-is-for-absorption-studying-feature-splitting-and) 때문)? **Autointerp는 latent 설명이 얼마나 좋은지를 직접적으로 정량화할 수 있기 때문에, SAE interpretability를 평가하는 대안적인 방법을 제공합니다!** 그 아이디어는 latent 설명을 일부 프롬프트 테스트 세트에 대한 예측 세트로 변환한 다음, 해당 예측의 정확도를 점수화하는 것입니다. 더 interpretable한 latent는 주어진 설명으로부터 예측 가능한 monosemantic하고 인간이 해석 가능한 패턴을 갖는 경향이 있으므로, 더 나은 예측으로 이어져야 합니다.

하지만 지금은 autointerp의 전반부, 즉 설명 생성에만 집중하겠습니다. 다음 코드를 사용하여 Neuronpedia에서 호스팅하는 latent 설명들을 다운로드하고 읽어보실 수 있습니다.

In [ ]:
def get_autointerp_df(sae_release="gpt2-small-res-jb", sae_id="blocks.7.hook_resid_pre") -> pd.DataFrame:
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = "https://www.neuronpedia.org/api/explanation/export?modelId={}&saeId={}".format(*neuronpedia_id.split("/"))
    headers = {"Content-Type": "application/json"}
    response = requests.get(url, headers=headers)

    data = response.json()
    return pd.DataFrame(data)


explanations_df = get_autointerp_df()
explanations_df.head()

이제 직접 autointerp를 시도해 보겠습니다! 여기에는 다음 3가지 단계가 포함됩니다:

1. `fetch_max_activating_examples`을 호출하여 특정 latent에 대해 가장 높게 activation되는 예시들을 가져옵니다.
2. `create_prompt`을 호출하여 이 데이터를 포함하는 OpenAI API용 system, user 및 assistant prompt를 생성합니다.
3. `get_autointerp_explanation`를 호출하여 이 prompt들을 OpenAI API에 전달하고 응답을 받습니다.

이미 `fetch_max_activating_examples`을 구현하셨고, `get_autointerp_explanation`는 저희가 제공해 드렸습니다. 이제 `create_prompt`만 구현하시면 됩니다.

<details>
<summary>추천하는 autointerp prompt 구조를 보려면 클릭하세요</summary>

Anthropic의 과거 공개 자료에 기반한 한 가지 가능한 방법은, 상위 sequence 리스트와 모든 개별 token에 대한 activation을 보여주고 설명을 요청하는 것입니다 (그 후 scoring 단계에서 모델에게 activation 값을 예측하도록 요청합니다). 하지만 여기서는 조금 더 간단하게, 수치적인 activation 값은 제공하지 않고 주어진 sequence에서 가장 높게 activation되는 token만 강조 표시하겠습니다.

```python
{
    "system": "We're studying neurons in a neural network. Each neuron activates on some particular word or concept in a short document. The activating words in each document are indicated with << ... >>. Look at the parts of the document the neuron activates for and summarize in a single sentence what the neuron is activating on. Try to be specific in your explanations, although don't be so specific that you exclude some of the examples from matching your explanation. Pay attention to things like the capitalization and punctuation of the activating words or concepts, if that seems relevant. Keep the explanation as short and simple as possible, limited to 20 words or less. Omit punctuation and formatting. You should avoid giving long lists of words.",

    "user": """The activating documents are given below:

1. and he was <<over the moon>> to find
2. we'll be laughing <<till the cows come home>>! Pro
3. thought Scotland was boring, but really there's more <<than meets the eye>>! I'd""",

    "assistant": "this neuron fires on",
}
```

우리는 system, user, 그리고 assistant prompt 순으로 모델에 입력합니다. 그 의도는 다음과 같습니다:

- system prompt는 수행할 작업이 무엇인지 설명합니다.
- user prompt는 실제 작업 데이터를 포함합니다.
- assistant prompt는 모델이 응답할 가능성이 높은 형식을 조건화하는 데 도움을 줍니다.

참고 - 이 모든 과정은 매우 기초적인 수준이며, 이후 섹션에서 autointerp를 더 깊게 다룰 때 이러한 방법들을 크게 확장할 예정입니다.

</details>

위에서 제공된 `get_k_largest_indices`의 확장 버전을 사용하시는 것을 권장합니다. 이 버전은 sequence 중복을 허용하지 않습니다. prompt에 중복된 정보를 보내는 것은 원하지 않기 때문입니다!

In [ ]:
def create_prompt(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 15,
    buffer: int = 10,
) -> dict[Literal["system", "user", "assistant"], str]:
    """
    Returns the system, user & assistant prompts for autointerp.
    """
    raise NotImplementedError()


# Test your function
prompts = create_prompt(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9, total_batches=100, k=15, buffer=8)
assert prompts["system"].startswith("We're studying neurons in a neural network.")
assert "<< new>>" in prompts["user"]
assert prompts["assistant"] == "this neuron fires on"

<details><summary>솔루션</summary>

```python
def create_prompt(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 15,
    buffer: int = 10,
) -> dict[Literal["system", "user", "assistant"], str]:
    """
    Returns the system, user & assistant prompts for autointerp.
    """
    data = fetch_max_activating_examples(model, sae, act_store, latent_idx, total_batches, k, buffer)
    str_formatted_examples = "\n".join(
        f"{i + 1}. {''.join(f'<<{tok}>>' if j == buffer else tok for j, tok in enumerate(seq[1]))}"
        for i, seq in enumerate(data)
    )

    return {
        "system": "We're studying neurons in a neural network. Each neuron activates on some particular word or concept in a short document. The activating words in each document are indicated with << ... >>. Look at the parts of the document the neuron activates for and summarize in a single sentence what the neuron is activating on. Try to be specific in your explanations, although don't be so specific that you exclude some of the examples from matching your explanation. Pay attention to things like the capitalization and punctuation of the activating words or concepts, if that seems relevant. Keep the explanation as short and simple as possible, limited to 20 words or less. Omit punctuation and formatting. You should avoid giving long lists of words.",
        "user": f"""The activating documents are given below:\n\n{str_formatted_examples}""",
        "assistant": "this neuron fires on",
    }
```
</details>

`create_prompt`에 대한 테스트를 통과했다면, 이제 전체 `get_autointerp_explanation` 함수를 구현할 수 있습니다:

In [ ]:
def get_autointerp_explanation(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 15,
    buffer: int = 10,
    n_completions: int = 1,
) -> list[str]:
    """
    Queries OpenAI's API using prompts returned from `create_prompt`, and returns a list of the
    completions.
    """
    raise NotImplementedError()


API_KEY = os.environ.get("OPENAI_API_KEY", None)

if API_KEY is not None:
    completions = get_autointerp_explanation(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9, n_completions=5)
    for i, completion in enumerate(completions):
        print(f"Completion {i + 1}: {completion!r}")
else:
    print("No API key found, not running the autointerp code.")

<details><summary>솔루션</summary>

```python
def get_autointerp_explanation(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 15,
    buffer: int = 10,
    n_completions: int = 1,
) -> list[str]:
    """
    Queries OpenAI's API using prompts returned from `create_prompt`, and returns a list of the
    completions.
    """
    client = OpenAI(api_key=API_KEY)

    prompts = create_prompt(model, sae, act_store, latent_idx, total_batches, k, buffer)

    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompts["system"]},
            {"role": "user", "content": prompts["user"]},
            {"role": "assistant", "content": prompts["assistant"]},
        ],
        n=n_completions,
        max_tokens=50,
        stream=False,
    )
    return [choice.message.content for choice in result.choices]
```
</details>

## Attention SAEs

> 이 섹션에서는 attention SAE가 무엇인지, 어떻게 작동하는지(대부분 표준 SAE와 매우 유사하지만 몇 가지 다른 고려 사항이 있습니다), 그리고 feature 대시보드를 어떻게 이해하는지 배웁니다. 핵심 포인트는 다음과 같습니다:
>
> - Attention SAE는 일반적인 SAE와 동일한 아키텍처를 가지지만, 모든 attention head의 pre-projection 출력을 연결(concatenate)한 데이터로 학습된다는 점이 다릅니다.
> - destination token에서 latent가 활성화되면, **direct latent attribution**을 사용하여 해당 latent가 주로 어떤 source token에서 왔는지 확인할 수 있습니다.
> - 일반적인 SAE와 마찬가지로, 모델의 서로 다른 layer에서 발견되는 latent들은 종종 질적으로 서로 다릅니다.

이 섹션에서는 주어진 개념에 대한 latent를 찾는 다양한 방법들을 탐구해 보겠습니다. 하지만 그 전에, 새로운 개념인 **attention SAEs**를 먼저 소개해야 합니다.

MATS 프로그램의 일환으로 [Kissane el al](https://www.lesswrong.com/posts/DtdzGwFh9dCfsekZZ/sparse-autoencoders-work-on-attention-layer-outputs)이 수행한 연구에 따르면, attention layer의 출력에 SAEs를 사용할 수 있으며 이는 매우 효과적으로 작동한다는 것이 밝혀졌습니다. SAEs는 희소하고 해석 가능한 latent를 학습하며, 이를 통해 attention layer가 무엇을 학습하는지에 대한 통찰을 얻을 수 있습니다. 후속 연구에서는 GPT2Small의 모든 layer의 attention 출력에 대해 SAEs를 학습시켰으며, 이것이 오늘 실습에서 사용할 SAEs입니다.

기능적으로 이 SAEs는 일반적인 SAEs와 동일하게 작동하지만, residual stream이나 post-ReLU MLP activation 대신 attention layer의 `z` 출력을 입력으로 받는다는 점이 다릅니다(즉, value 벡터들의 선형 결합을 취한 후, output matrix를 통해 projection 하기 전의 단계입니다). 이러한 `z` 벡터들은 보통 단일 layer 내의 attention head들에 걸쳐 함께 concatenate 됩니다.

<details>
<summary>왜 output matrix를 통한 projection 이후가 아니라 이전의 attention 출력을 사용하는지 이해하시겠습니까?</summary>

그것은 파라미터 낭비가 되기 때문입니다. encoder는 activation space에서 latent space로의 선형 맵이며, attention head의 출력 `z @ W_O`은 `z`보다 더 큰 rank를 가질 수 없습니다(더 작은 rank를 가질 수는 있습니다). 하지만 projection 이후에는 크기가 더 커지게 되어 결과적으로 학습 효율성이 떨어지게 됩니다.

</details>

<details>
<summary>왜 attention head들에 걸쳐 concatenate 하는지 추측할 수 있습니까?</summary>

이는 MLP layer의 neuron들과 마찬가지로 head들도 superposition 상태일 수 있기 때문입니다. 단일 attention head가 많은 latent를 포함하고 있을 수도 있지만, 하나의 latent가 여러 attention head에 걸쳐 나뉘어 있을 수도 있습니다. 일반적인 모델에서 attention head의 기능이 공유된다는 증거는 매우 많습니다. 예를 들어, mech interp ARENA 실습 도입부에서 우리는 두 번째 layer의 head 2개가 합쳐져 copying head를 형성하는 2-layer 모델을 살펴보았습니다. 이 경우, 두 head에 걸쳐 나뉘어 있는 latent를 발견할 수 있을 것으로 기대할 수 있습니다.

</details>

하지만 attention SAEs의 한 가지 흥미로운 점은 destination token뿐만 아니라 source token에 대해서도 생각해야 한다는 것입니다. 다시 말해, 특정 destination token에서 존재하는 어떤 attention latent를 식별했다면, 그것이 어떤 source token으로부터 왔는지에 대한 질문을 여전히 던져야 합니다.

이는 **direct latent attribution** (direct logit attribution과 혼동되지 않도록 "DFA" 또는 "direct feature attribution"으로 약칭하겠습니다)이라는 도구로 이어집니다. direct logit attribution (DLA)에서 어떤 컴포넌트가 특정 logit에 직접적인 영향을 주는 방식으로 residual stream에 썼는지 묻는 것과 마찬가지로, DFA를 통해 해당 latent를 활성화시킨 destination token으로의 입력을 분해할 수 있습니다. 이를 통해 어떤 head가 해당 latent에 가장 많이 기여했는지, 또는 어떤 source token이 기여했는지(혹은 둘 다)를 알 수 있습니다.

### 연습 문제 - attention SAE 대시보드 탐색하기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

아래 코드를 실행하여 GPT2-Small의 layer-9 attention head에 대한 latent 대시보드 예시를 확인해 보십시오. 오른쪽의 초록색 텍스트는 이 latent가 언제 activation 되었는지를 보여주며, 주황색 하이라이트는 이를 활성화시킨 주요 token의 DFA를 보여줍니다.

특히 서로 다른 layer에서 학습된 SAE의 latent들 사이에 어떤 질적인 차이가 있는지 생각하며 살펴보십시오 (아래의 `layer` 변수를 변경하여 이를 조사할 수 있습니다). 무엇이 느껴지십니까? 앞쪽 layer에는 존재하지만 뒤쪽 layer에는 없는 latent의 종류는 무엇입니까? 어떤 layer가 더 해석 가능한 output logit을 가지고 있습니까?

<details>
<summary>확인해야 할 사항들</summary>

일반적으로 다음과 같은 테마들을 발견하실 수 있습니다:

- 초기 layer head latent들은 종종 낮은 수준의 문법적 구문(예: 단일 token 또는 bigram에서 firing)을 나타냅니다.
- 중간 layer head latent들은 더 높은 수준의 semantic 정보에 반응하기 때문에 해석하기 가장 어려운 경우가 많습니다. 또한, 이들은 다른 head나 MLP layer에서 사용되는 중간 표현(intermediate representations)에 기록하기 때문에 output logit 관점에서는 항상 해석 가능하지 않을 가능성이 큽니다.
- 후기 layer head latent들은 종종 output logit 관점에서 이해됩니다 (즉, residual stream에 직접 예측값을 기록합니다). 가장 마지막 layer는 주로 문법적 수정 및 조정을 다루는 것으로 보이기 때문에 이 경우의 예외라고 할 수 있습니다.

이에 대한 더 자세한 내용은 우리가 여기서 다루고 있는 것과 동일한 SAE를 분석한 LessWrong 포스트 [We Inspected Every Head In GPT-2 Small using SAEs So You Don’t Have To](https://www.lesswrong.com/posts/xmegeW5mqiBsvoaim/we-inspected-every-head-in-gpt-2-small-using-saes-so-you-don#Overview_of_Attention_Heads_Across_Layers) 의 표를 통해 확인하실 수 있습니다.

</details>

In [ ]:
attn_saes = {
    layer: SAE.from_pretrained(
        "gpt2-small-hook-z-kk",
        f"blocks.{layer}.hook_z",
        device=str(device),
    )
    for layer in range(gpt2.cfg.n_layers)
}

layer = 9

display_dashboard(
    sae_release="gpt2-small-hook-z-kk",
    sae_id=f"blocks.{layer}.hook_z",
    latent_idx=2,  # or you can try `random.randint(0, attn_saes[layer].cfg.d_sae)`
)

### 연습 문제 - attention DFA 유도하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 25-45 minutes on this exercise.
> ```

latent 대시보드에 새로운 컴포넌트를 추가했으므로, 다시 한번 유도를 진행해 보겠습니다! 주어진 destination token firing에 대해, 특정 source token의 **direct latent attribution (DFA)**은 **해당 source token에서 추출된 value 벡터(attention probability로 스케일링됨)와 이 latent에 대한 SAE encoder direction의 내적**으로 정의할 수 있습니다. 다시 말해, latent의 pre-ReLU activation은 모든 source token에 대한 DFA의 합과 같습니다.

이 문제를 완료하면 attention 대시보드의 마지막 부분을 완성할 수 있습니다. residual stream SAE에서 사용했던 것과 같이, 이를 시각화하기 위한 함수를 제공해 드립니다:

In [ ]:
@dataclass
class AttnSeqDFA:
    act: float
    str_toks_dest: list[str]
    str_toks_src: list[str]
    dest_pos: int
    src_pos: int


def display_top_seqs_attn(data: list[AttnSeqDFA]):
    """
    Same as previous function, but we now have 2 str_tok lists and 2 sequence positions to
    highlight, the first being for top activations (destination token) and the second for top DFA
    (src token). We've given you a dataclass to help keep track of this.
    """
    table = Table(
        "Top Act",
        "Src token DFA (for top dest token)",
        "Dest token",
        title="Max Activating Examples",
        show_lines=True,
    )
    for seq in data:
        formatted_seqs = [
            repr(
                "".join(
                    [f"[b u {color}]{str_tok}[/]" if i == seq_pos else str_tok for i, str_tok in enumerate(str_toks)]
                )
                .replace("�", "")
                .replace("\n", "↵")
            )
            for str_toks, seq_pos, color in [
                (seq.str_toks_src, seq.src_pos, "dark_orange"),
                (seq.str_toks_dest, seq.dest_pos, "green"),
            ]
        ]
        table.add_row(f"{seq.act:.3f}", *formatted_seqs)
    rprint(table)


str_toks = [" one", " two", " three", " four"]
example_data = [
    AttnSeqDFA(act=0.5, str_toks_dest=str_toks[1:], str_toks_src=str_toks[:-1], dest_pos=0, src_pos=0),
    AttnSeqDFA(act=1.5, str_toks_dest=str_toks[1:], str_toks_src=str_toks[:-1], dest_pos=1, src_pos=1),
    AttnSeqDFA(act=2.5, str_toks_dest=str_toks[1:], str_toks_src=str_toks[:-1], dest_pos=2, src_pos=0),
]
display_top_seqs_attn(example_data)

이제 아래 함수를 채워 넣으십시오. `latent_idx=2` (위에서 보여준)에서 생성된 대시보드와 출력을 비교하여 함수를 테스트하게 됩니다. `" weapons"`, `" firearms"`, `" missiles"` 등과 같은 단어 목록을 연결하는 `" and"`, `" or"`와 같은 접속사들이 상위 activation token이라는 점을 관찰하셨을 것입니다. 또한 max DFA token은 보통 컨텍스트 앞부분에 나오는 유사한 단어이며, 때로는 상위 activating token 바로 앞의 단어인 것을 볼 수 있습니다 (positive logit이 이러한 단어들을 강화한다는 점을 고려하면 타당합니다). 다시 말해, 이 latent가 수행하는 역할의 일부는 `weapons and tactics` 또는 `guns ... not missiles`와 같은 접속사 구문의 중간에 있다는 것을 감지하고, 이 구문이 어떻게 끝날지에 대한 논리적인 완성을 예측하는 것입니다.

참고 - 이 문제는 꽤 어렵습니다 (주로 많은 rearrange 작업과 솔루션의 여러 단계 때문입니다). 막히는 부분이 있다면 아래의 힌트를 사용하는 것을 강력히 권장합니다.

<details>
<summary>도움말 - 이 계산 방식이 여전히 헷갈립니다.</summary>

우리는 각 위치에서 `(batch, seq, n_heads, d_head)` 모양의 value 벡터 `v`를 가지고 있습니다. broadcasting 및 attention 확률과의 곱셈을 통해 `(batch, seq_dest, seq_src, n_heads, d_head)` 모양의 `v_weighted`를 얻을 수 있으며, 이는 각 source position에서 가져와 destination position에 더해질 벡터를 나타내고, 이들의 합을 통해 `z` (residual stream에 다시 더하기 위해 output matrix `W_O`로 projection 하기 전의 값들)가 생성됩니다.

SAE가 학습되는 대상은 (attention head들에 대해 flattening 한 후의) 바로 이 `z`이며, 즉 `z @ sae.W_enc`가 SAE의 pre-ReLU activation입니다. 따라서 `z`를 source position들에 대한 `v_weighted`의 합으로 표현함으로써, latent `latent_idx`에 대한 pre-ReLU activation을 모든 `src_pos` 값들에 대한 `v_weighted[:, :, src_pos, :, :] @ sae.W_enc[:, latent_idx]`의 합으로 쓸 수 있습니다. 따라서 배치의 임의의 시퀀스 `b`와 destination position `dest_pos`에 대해, 각 `src_pos`에 대한 스칼라 `v_weighted[b, dest_pos, src_pos, :, :] @ sae.W_enc[:, latent_idx]`를 계산하고 그중 가장 큰 값을 찾을 수 있습니다.

주의 - `W_enc`는 실제로는 `n_heads * d_head` 차원에서 `d_sae` 차원으로의 linear map이므로, 이 계산을 수행하려면 먼저 head들에 대해 value `v_weighted`를 flatten 해야 합니다.

</details>

참고로, 일부 src token 인덱싱이 다소 까다로울 수 있습니다. 특히, 가장 많이 기여하는 source token의 인덱스 위치를 구할 때, 일부는 시퀀스 시작 지점의 `buffer` 이내에 있을 수 있습니다. `index_with_buffer`가 이 경우를 처리해 주는데, 인덱싱 값이 시퀀스의 시작 또는 끝의 `buffer` 이내일 때마다 각각 처음 또는 마지막 `buffer` token을 가져오기 때문입니다.

In [ ]:
def fetch_max_activating_examples_attn(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 250,
    k: int = 10,
    buffer: int = 10,
) -> list[AttnSeqDFA]:
    """
    Returns the max activating examples across a number of batches from the activations store.
    """
    raise NotImplementedError()


# Test your function: compare it to dashboard above
# (max DFA should come from sourcs tokens like " guns", " firearms")
layer = 9
data = fetch_max_activating_examples_attn(gpt2, attn_saes[layer], gpt2_act_store, latent_idx=2)
display_top_seqs_attn(data)

<details><summary>솔루션</summary>

```python
def fetch_max_activating_examples_attn(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 250,
    k: int = 10,
    buffer: int = 10,
) -> list[AttnSeqDFA]:
    """
    Returns the max activating examples across a number of batches from the activations store.
    """
    sae_acts_pre_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_pre"
    v_hook_name = get_act_name("v", _get_hook_layer(sae))
    pattern_hook_name = get_act_name("pattern", _get_hook_layer(sae))
    data = []

    for _ in tqdm(range(total_batches), desc="Computing activations for max activating examples (attn)"):
        tokens = act_store.get_batch_tokens()
        _, cache = model.run_with_cache_with_saes(
            tokens,
            saes=[sae],
            stop_at_layer=_get_hook_layer(sae) + 1,
            names_filter=[sae_acts_pre_hook_name, v_hook_name, pattern_hook_name],
        )
        acts = cache[sae_acts_pre_hook_name][..., latent_idx]  # [batch seq]

        # Get largest indices (i.e. dest tokens), and the tokens at those positions (plus buffer)
        k_largest_indices = get_k_largest_indices(acts, k=k, buffer=buffer)
        top_acts = index_with_buffer(acts, k_largest_indices).tolist()
        dest_toks_with_buffer = index_with_buffer(tokens, k_largest_indices, buffer=buffer)
        str_toks_dest_list = [model.to_str_tokens(toks) for toks in dest_toks_with_buffer]

        # Get source token value vectors & dest-to-src attention patterns, for each of our chosen
        # destination tokens
        batch_indices, dest_pos_indices = k_largest_indices.unbind(-1)
        v = cache[v_hook_name][batch_indices]  # shape [k src n_heads d_head]
        pattern = cache[pattern_hook_name][batch_indices, :, dest_pos_indices]  # [k n_heads src]

        # Multiply them together to get weighted value vectors, and reshape them to d_in = n_heads * d_head
        v_weighted = v * einops.rearrange(pattern, "k n src -> k src n 1")
        v_weighted = v_weighted.flatten(-2, -1)  # [k src d_in]

        # Map through our SAE encoder to get direct feature attribution for each src token, and argmax over src tokens
        dfa = v_weighted @ sae.W_enc[:, latent_idx]  # shape [k src]
        src_pos_indices = dfa.argmax(dim=-1)
        src_toks_with_buffer = index_with_buffer(tokens, t.stack([batch_indices, src_pos_indices], -1), buffer=buffer)
        str_toks_src_list = [model.to_str_tokens(toks) for toks in src_toks_with_buffer]

        # Add all this data to our list
        for act, str_toks_dest, str_toks_src, src_pos in zip(
            top_acts, str_toks_dest_list, str_toks_src_list, src_pos_indices
        ):
            data.append(
                AttnSeqDFA(
                    act=act,
                    str_toks_dest=str_toks_dest,  # top activating dest tokens, with buffer
                    str_toks_src=str_toks_src,  # top DFA src tokens for the dest token, with buffer
                    dest_pos=buffer,  # dest token is always in the middle of its buffer
                    src_pos=min(src_pos, buffer),  # deal with case where src token is near start
                )
            )

    return sorted(data, key=lambda x: x.act, reverse=True)[:k]
```
</details>

## feature를 위한 latent 찾기

> 이 섹션에서는 특정 feature에 해당하는 SAE의 latent를 찾기 위한 다양한 방법(인과적인 방법과 그렇지 않은 방법 모두)을 탐구합니다. 핵심 포인트는 다음과 같습니다:
>
> - 특정 입력 prompt에서 **최대 activation을 보이는 latent**를 살펴볼 수 있으며, 이는 기본적으로 가장 간단한 방법입니다.
> - **Direct logit attribution (DLA)**은 조금 더 정교한 방법입니다. 이를 통해 특정 logit에 직접적인 영향을 주는 latent를 찾을 수 있습니다.
> - SAE latent의 **Ablation**을 통해 간접적인 방식으로 중요한 latent를 찾는 데 도움을 받을 수 있습니다.
> - ...하지만 많은 수의 latent에 대해 수행하기에는 비용이 매우 많이 들므로, ablation의 더 저렴한 선형 근사치인 **attribution patching**을 사용할 수 있습니다.

이제 특정 개념이나 언어 구조에서 활성화되는 feature를 찾기 위해 사용할 수 있는 3가지 서로 다른 방법들을 살펴보겠습니다. 우리는 **IOI feature**, 즉 indirect object identification 패턴에서 활성화되는 것으로 보이는 feature를 찾는 데 집중할 것입니다. ARENA의 IOI 실습을 통해 이미 익숙하실 수도 있습니다. 그렇지 않다면, 이는 기본적으로 `"When John and Mary went to the shops, John gave the bag to"` -> `" Mary"` 형태의 문장들입니다. GPT2-Small과 같은 모델들은 다음과 같은 알고리즘을 통해 이 패턴을 학습할 수 있습니다:

- 0-3 레이어의 **Duplicate token heads**는 두 번째 `" John"` token(우리는 이를 `S2`, 즉 두 번째 subject라고 부릅니다)에서 첫 번째 `" John"` token(`S1`)으로 attention을 보내며, 이것이 중복되었다는 사실을 저장합니다.
- 7-8 레이어의 **S-inhibition heads**는 `" to"`에서 `" John"` token으로 다시 attention을 보내며, 이 token이 중복되었다는 정보를 저장합니다.
- 9-10 레이어의 **Name-mover heads**는 `" to"`에서 중복되지 않은 모든 이름으로 attention을 보냅니다 (중복된 `" John"` token에 attention을 보내는 것을 피하기 위해 S-Inhibition heads의 출력과 Q-composition을 사용합니다). 따라서 이들은 `" Mary"`에 attention을 보내고, 이 정보를 unembedding 공간으로 이동시켜 모델의 예측값으로 사용합니다.

아쉽게도, 우리의 SAE는 아직 **S-inhibition feature**를 포착할 만큼 발전하지 않았습니다. 위에서 논의했듯이, 중간 표현을 포함하는 subspace에서 읽고 쓰는 이러한 중간 레이어 head들은 해석하기가 매우 어렵습니다. 실제로 GPT2-Small attention SAE를 분석한 LessWrong 포스트의 [this section](https://www.lesswrong.com/posts/FSTRedtjuHa4Gfdbr/attention-saes-scale-to-gpt-2-small#Introduction) 부분을 읽어보면, "해석 가능한 % alive features"와 "loss recovered" 수치가 가장 낮은 레이어가 7과 8 주변이며, 이는 정확히 S-Inhibition heads가 위치한 곳입니다!

하지만 해당 포스트의 저자들은 **duplicate token feature**와 **name-mover feature**를 찾아낼 수 있었으며, 이어지는 실습에서 우리는 그들의 작업을 재현해 보겠습니다!

시작하기 전에, 먼저 모델이 실제로 이 문장을 해결할 수 있는지 확인해 보겠습니다. 결과의 강건함을 높이기 위해 (예를 들어, 단순히 "성별 feature" 같은 것만 분리해내는 것이 아니도록), 4가지 서로 다른 prompt를 사용하여 제어하겠습니다: "John"과 "Mary"가 정답으로 서로 바뀌는 경우, 그리고 문장 구조가 바뀌는 경우(ABBA vs ABAB)를 포함합니다. 아래 코드는 나중 실습에서 유용하게 사용할 수 있는 `logits_to_ave_logit_diff` 함수도 제공합니다.

In [ ]:
names = [" John", " Mary"]
name_tokens = [gpt2.to_single_token(name) for name in names]

prompt_template = "When{A} and{B} went to the shops,{S} gave the bag to"
prompts = [
    prompt_template.format(A=names[i], B=names[1 - i], S=names[j]) for i, j in itertools.product(range(2), range(2))
]
correct_answers = names[::-1] * 2
incorrect_answers = names * 2
correct_toks = gpt2.to_tokens(correct_answers, prepend_bos=False)[:, 0].tolist()
incorrect_toks = gpt2.to_tokens(incorrect_answers, prepend_bos=False)[:, 0].tolist()


def logits_to_ave_logit_diff(
    logits: Float[Tensor, "batch seq d_vocab"],
    correct_toks: list[int] = correct_toks,
    incorrect_toks: list[int] = incorrect_toks,
    reduction: Literal["mean", "sum"] | None = "mean",
    keep_as_tensor: bool = False,
) -> list[float] | float:
    """
    Returns the avg logit diff on a set of prompts, with fixed s2 pos and stuff.
    """
    correct_logits = logits[range(len(logits)), -1, correct_toks]
    incorrect_logits = logits[range(len(logits)), -1, incorrect_toks]
    logit_diff = correct_logits - incorrect_logits
    if reduction is not None:
        logit_diff = logit_diff.mean() if reduction == "mean" else logit_diff.sum()
    return logit_diff if keep_as_tensor else logit_diff.tolist()


# Testing a single prompt (where correct answer is John), verifying model gets it right
test_prompt(prompts[1], names, gpt2)

# Testing logits over all 4 prompts, verifying the model always has a high logit diff
logits = gpt2(prompts, return_type="logits")
logit_diffs = logits_to_ave_logit_diff(logits, reduction=None)
print(
    tabulate(
        zip(prompts, correct_answers, logit_diffs),
        headers=["Prompt", "Answer", "Logit Diff"],
        tablefmt="simple_outline",
        numalign="left",
        floatfmt="+.3f",
    )
)

### 연습 문제 - 모델 + SAE가 여전히 이 문제를 해결할 수 있는지 확인하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

특정 layer를 SAE로 대체했을 때도 모델의 logit diff가 여전히 높은지 확인하고자 합니다. 만약 그렇지 않다면, 이는 우리의 SAE가 IOI circuit을 구현할 수 없음을 의미하며, 흥미로운 feature를 찾을 수 있을 것이라고 기대할 이유가 없습니다!

`with model.saes` context manager(또는 SAE를 실행하는 선호하는 방식)를 사용하여, 각 layer의 attention SAE에 대해 4개 prompt의 평균 logit diff를 구하십시오. 이 차이가 여전히 높은지, 즉 SAE가 성능을 저하시키지 않는지 확인하십시오.

In [ ]:
# YOUR CODE HERE - verify model + SAEs can still solve this

<details><summary>솔루션</summary>

```python
logits = gpt2(prompts, return_type="logits")
clean_logit_diff = logits_to_ave_logit_diff(logits)

table = Table("Ablation", "Logit diff", "% of clean")

table.add_row("Clean", f"{clean_logit_diff:+.3f}", "100.0%")

for layer in range(gpt2.cfg.n_layers):
    with gpt2.saes(saes=[attn_saes[layer]]):
        logits = gpt2(prompts, return_type="logits")
        logit_diff = logits_to_ave_logit_diff(logits)
        table.add_row(
            f"SAE in L{layer:02}",
            f"{logit_diff:+.3f}",
            f"{logit_diff / clean_logit_diff:.1%}",
        )

rprint(table)
```
</details>

<details>
<summary>결과 논의</summary>

대부분의 layer가 100%에 가까운 복구율을 보이며, 심지어 10번과 11번 같은 일부 layer는 대체했을 때 logit diff가 오히려 *증가*하는 것을 확인할 수 있습니다. 이번 경우에는 작은 데이터셋으로 작업하고 있기 때문에 어느 정도의 노이즈는 예상되는 일이므로, 이 결과에 너무 큰 의미를 부여하지 않으셔도 됩니다.

logit diff 복구 측면에서 가장 성능이 낮은 layer는 7번과 8번이며, 이는 S-Inhibition head가 포함된 layer라는 점을 고려하면 타당한 결과입니다. 두 layer 모두 logit diff가 여전히 양수이고, 7번과 8번 SAE를 동시에 대체했을 때도 (clean logit diff의 58%로 떨어지기는 하지만) 여전히 양수를 유지한다는 점은, 이 SAE들이 S-Inhibition head의 동작을 어느 정도 캡처하고 있음을 시사합니다. 하지만 이것이 monosemantic한 S-Inhibition feature를 가졌음을 보장하는 것은 아니며, 실제로 그렇지 않습니다.

</details>

이제 이 과정을 마쳤으니, 연습 문제를 풀 시간입니다!

### 연습 문제 - 최대 activation을 가진 name mover feature 찾기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

이제 SAE로 대체한 상태에서도 모델이 여전히 태스크를 해결할 수 있다는 점을 확인했으므로, feature를 찾을 준비가 되었습니다!

첫 번째 연습 문제에서는 **name mover feature**를 찾아보겠습니다. 이 feature들은 프롬프트의 마지막 token에서 activate되며, 다음에 IO token이 올 것임을 예측하는 것처럼 보입니다. layer 9의 attention head들이 name mover 역할을 하므로(주로 9.6과 9.9), 우리는 layer 9 SAE, 즉 `attn_saes[9]`를 살펴볼 것입니다. 아래 셀을 채워 4개의 모든 프롬프트에 대해 평균을 낸 마지막 token에서의 latent activation 그래프를 생성하고, 상위 3개 latent의 대시보드를 표시하십시오. (도움이 필요하다면 이전 섹션, 특히 "Running SAEs" 섹션의 세 번째 코드 셀에서 코드 일부를 가져와 사용할 수 있습니다.) name mover feature에 해당할 것으로 보이는 latent를 찾으셨나요?

In [ ]:
# YOUR CODE HERE - find name mover latents by using max activations

<details>
<summary>스포일러 - 찾아야 할 내용</summary>

feature 11368, 18767 그리고 3101을 반환해야 합니다.

이들은 모두 name mover로 보입니다:

- 11368은 "John"을 위한 name mover입니다 (즉, "John"이 나타날 수 있는 토큰 이전에서, 이전의 "John" 인스턴스로 항상 attend합니다)
- 18767은 "Mary"를 위한 name mover입니다
- 3101은 "Jesus"를 위한 name mover입니다

11368과 18767의 activation은 다른 어떤 feature의 activation보다 훨씬 커야 한다는 점에 유의하십시오.

</details>


<details><summary>솔루션</summary>

```python
layer = 9

# Compute mean post-ReLU SAE activations at last token posn
_, cache = gpt2.run_with_cache_with_saes(prompts, saes=[attn_saes[layer]])
sae_acts_post = cache[f"{attn_saes[layer].cfg.metadata.hook_name}.hook_sae_acts_post"][:, -1].mean(0)

# Plot the activations
px.line(
    sae_acts_post.cpu().numpy(),
    title=f"Activations at the final token position ({sae_acts_post.nonzero().numel()} alive)",
    labels={"index": "Latent", "value": "Activation"},
    template="ggplot2",
    width=1000,
).update_layout(showlegend=False).show()

# Print the top 3 latents, and inspect their dashboards
for act, ind in zip(*sae_acts_post.topk(3)):
    print(f"Latent {ind} had activation {act:.2f}")
    display_dashboard(
        sae_release="gpt2-small-hook-z-kk",
        sae_id=f"blocks.{layer}.hook_z",
        latent_idx=int(ind),
    )
```
</details>

### 연습 문제 - name mover head 식별하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

IOI 논문에서는 head `9.6`과 `9.9`가 주요한 name mover라는 것을 발견했습니다. feature들의 head별 weight 분포를 살펴봄으로써, 즉 이 특정 feature들이 어떤 head에 가장 많이 노출되어 있는지 확인하여 이를 검증할 수 있습니까?

In [ ]:
# YOUR CODE HERE - verify model + SAEs can still solve this

<details>
<summary>힌트 - 수행해야 할 계산</summary>

각 feature는 `(d_in=n_heads*d_head,)` 형상의 관련 decoder weight `sae.W_dec[:, feature_idx]`를 가집니다. 우리는 각 `(d_head,)` 길이 벡터(모델 내 각 attention head에 대한 노출도를 나타냄)의 norm을 측정하여, 어떤 head가 전체 norm에서 가장 큰 비중을 차지하는지 확인할 수 있습니다.

</details>

<details>
<summary>정답</summary>

```python
features = [18767, 10651]
decoder_weights = einops.rearrange(
    attn_saes[layer].W_dec[features],
    "feats (n_heads d_head) -> feats n_heads d_head",
    n_heads=model.cfg.n_heads,
)
norm_per_head = decoder_weights.pow(2).sum(-1).sqrt()
norm_frac_per_head = norm_per_head / norm_per_head.sum(-1, keepdim=True)

table = Table("Head", *[f"Feature {i}" for i in features])
for i in range(model.cfg.n_heads):
    table.add_row(
        f"9.{i}", *[f"{frac:.2%}" for frac in norm_frac_per_head[:, i].tolist()]
    )

rprint(table)
```

이 두 feature 모두 `9.9`에 가장 많이 노출되어 있으며, `9.6`에 두 번째로 많이 노출되어 있음을 알 수 있습니다.

</details>

### 직접적인 logit attribution

최대 activation을 갖는 latent를 찾는 것보다 약간 더 정교한 기법은, 특정 token logit 또는 logit difference에 특정한 영향을 주는 latent를 찾는 것입니다. IOI circuit의 맥락에서, 우리는 indirect object와 subject token 사이의 logit difference를 살펴봅니다 (이를 `IO - S`로 줄여서 표현하겠습니다). 예를 들어, 모델의 unembedding matrix `W_U`를 적용했을 때 latent가 residual stream으로 내보내는 출력값이 `IO - S`의 매우 큰 값을 가진다면, 이는 "When John and Mary went to the shops, John gave the bag to ???"와 같은 문장에서 정답이 "John"이 아니라 "Mary"라는 것을 식별하는 데 해당 latent가 인과적으로 중요했을 수 있음을 시사합니다. 이는 이전 맥락이나 ARENA 실습(예: IOI 실습에서 다루었던 attention head에 대한 DLA)에서 이미 접하셨을 DLA와 본질적으로 동일합니다.

### 연습 문제 - DLA를 사용하여 name mover feature 찾기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise.
> ```

이 계산을 수행하고 결과(선 그래프 및 상위 점수 latent에 대한 대시보드)를 시각화하는 코드를 아래에 작성하십시오. 작성할 코드는 max activation을 통해 name mover를 찾는 코드와 매우 유사해야 합니다. 다만, 평균 latent activation에 대해 argmax를 취하는 대신, `IO - S` 방향으로의 latent 평균 DLA에 대해 argmax를 취해야 합니다.

<details>
<summary>도움말 - 이 계산이 어떤 식이어야 하는지 잘 모르겠습니다.</summary>

먼저 길이가 `d_sae = n_heads * d_head`인 decoder의 latent direction에서 시작합니다. 우리는 이를 모델의 concatenated `z` 벡터(즉, projection matrix를 적용하기 전에 얻는 value 벡터들의 선형 결합) 공간의 벡터로 해석합니다. 따라서 shape이 `(n_heads, d_head, d_model)`인 projection matrix `W_O`를 flatten하고 이를 decoder 벡터와 곱하면(`d_sae` 차원에 대해 reduce 하지 않음), shape이 `(d_sae, d_model)`인 행렬을 얻게 됩니다. 여기서 각 행은 해당 latent가 활성화될 때 residual stream에 더해지는 벡터입니다. 이제 이 행렬을 unembedding `W_U`에 통과시켜 shape이 `(d_sae, d_vocab)`인 행렬을 얻고, 이를 인덱싱하여 각 latent에 대한 logit difference를 구할 수 있습니다 (또는 더 효율적으로 처리하려면, 행렬과 벡터 `W_U[:, IO] - W_U[:, S]`를 dot product 하여 `d_sae`개의 logit difference 벡터를 직접 얻을 수 있습니다).

이것이 단일 prompt에 대해 latent의 DLA를 구하는 방법입니다. 이를 병렬화하여 4개의 prompt를 한 번에 처리하고(즉, shape이 `(4, d_sae)`인 출력), batch 차원에 대해 평균을 내어 단일 DLA 벡터를 얻을 수 있습니다.

</details>

In [ ]:
# YOUR CODE HERE - verify model + SAEs can still solve this

<details>
<summary>솔루션 및 토론</summary>

솔루션 코드:

```python
# Get logits in the "IO - S" direction, of shape (4, d_model)
logit_direction = gpt2.W_U.T[correct_toks] - gpt2.W_U.T[incorrect_toks]

# Get latent activations, of shape (4, d_sae)
sae_acts_post_hook_name = f"{attn_saes[layer].cfg.metadata.hook_name}.hook_sae_acts_post"
_, cache = gpt2.run_with_cache_with_saes(prompts, saes=[attn_saes[layer]], names_filter=[sae_acts_post_hook_name])
sae_acts_post = cache[sae_acts_post_hook_name][:, -1]

# Get values written to the residual stream by each latent
sae_resid_dirs = einops.einsum(
    sae_acts_post,
    attn_saes[layer].W_dec,
    gpt2.W_O[layer].flatten(0, 1),
    "batch d_sae, d_sae nheads_x_dhead, nheads_x_dhead d_model -> batch d_sae d_model",
)

# Get DLA by computing average dot product of each latent's residual dir onto the logit dir
dla = (sae_resid_dirs * logit_direction[:, None, :]).sum(-1).mean(0)

# Display the results
px.line(
    dla.cpu().numpy(),
    title="Latent DLA (in IO - S direction) at the final token position",
    labels={"index": "Latent", "value": "DLA"},
    template="ggplot2",
    width=1000,
).update_layout(showlegend=False).show()

# Print the top 3 features, and inspect their dashboards
for value, ind in zip(*dla.topk(3)):
    print(f"Latent {ind} had max act {sae_acts_post[:, ind].max():.2f}, mean DLA {value:.2f}")
    display_dashboard(
        sae_release="gpt2-small-hook-z-kk",
        sae_id=f"blocks.{layer}.hook_z",
        latent_idx=int(ind),
    )
```

이전과 동일한 상위 2개 feature를 얻게 되지만, 차이점은 이제 다른 feature들이 훨씬 약해졌다는 것입니다. 어떤 feature도 상위 2개 feature의 10%보다 큰 DLA를 가지지 않습니다. 새롭게 발견한 세 번째 feature는 `" Joan"`를 위한 name mover인 것으로 보이며, `" John"`와 `" Joan"`라는 이름이 비슷하기 때문에 (매우 약하게) activation되고 있습니다.

이는 우리가 feature에서 기대할 수 있는 중요한 속성인 **functional form의 sparsity**를 강조합니다. SAE는 특정 종류의 sparsity(즉, feature를 전체 token 중 아주 적은 일부에만 존재하는 데이터셋의 속성으로 보는 것)라는 가정하에 구축되지만, 진정한 feature는 functional sparsity를 가진 functional form으로도 간주되어야 합니다. 다시 말해, 모든 feature는 모델 내에서 매우 좁은 범위의 일들만 수행하는 것으로 설명될 수 있습니다. 여기에서 우리는 단순히 특정 token에서 active한 feature뿐만 아니라, active하면서 *`IO - S` 방향에 기여하는* feature들을 필터링함으로써 name mover들을 더 효과적으로 격리할 수 있음을 확인했습니다.

이러한 **contrastive pairs**라는 개념은 앞으로 계속해서 반복해서 등장하게 됩니다.

</details>

### 연습 문제 (선택 사항) - 감성 분석을 통해 결과 재현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> ```

Anthropic의 [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/#computational-sad) 논문에 나오는 감성 기반 프롬프트를 사용하여 이 결과들을 재현할 수 있습니다:

In [ ]:
prompt = 'John says, "I want to be alone right now." John feels very'
correct_completion = " sad"
incorrect_completion = " happy"

test_prompt(prompt, correct_completion, gpt2)
test_prompt(prompt, incorrect_completion, gpt2)

모델은 "lonely", "alone", "uncomfortable"과 같은 단어들이 상위 예측값으로 나타나고 ("sad"와 "happy" 사이에 양수의 logit diff가 존재하므로), 이 문장의 sentiment가 부정적이라는 것을 이해하고 있는 것으로 보입니다. 이러한 sentiment를 나타내는 feature들을 찾을 수 있을까요? attention SAEs를 사용하는 대신 원래의 layer-7 `sae` 로 돌아가 보는 것이 좋을 것입니다.

DLA가 단순히 feature activation에 대해 argmax를 취하는 것보다 얼마나 더 나은 성능을 보이나요? 이 두 가지 기법 중 어느 것이, 혹은 둘 다 부정적인 sentiment feature를 찾아내나요?

In [ ]:
# YOUR CODE HERE - replicate these results with sentiment prompts

<details><summary>솔루션</summary>

```python
logit_dir = (
    gpt2.W_U[:, gpt2.to_single_token(correct_completion)] - gpt2.W_U[:, gpt2.to_single_token(incorrect_completion)]
)

_, cache = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])
sae_acts_post = cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, :]

sae_attribution = sae_acts_post * (gpt2_sae.W_dec @ logit_dir)

px.line(
    sae_attribution.cpu().numpy(),
    title=f"Attributions for (sad - happy) at the final token position ({sae_attribution.nonzero().numel()} non-zero attribution)",
    labels={"index": "Latent", "value": "Attribution"},
    template="ggplot2",
    width=1000,
).update_layout(showlegend=False).show()

for attr, ind in zip(*sae_attribution.topk(3)):
    print(f"#{ind} had attribution {attr:.2f}, activation {sae_acts_post[ind]:.2f}")
    display_dashboard(latent_idx=int(ind))
```
</details>

### Ablation

DLA와 같은 기법들은 feature가 모델의 출력에 상당한 직접적인 영향을 미칠 것으로 예상될 때 잘 작동합니다. 하지만 feature가 인과적으로 중요하지만, 직접적인 방식이 아닐 때는 어떻게 해야 할까요? 바로 이때 **ablation** 또는 **activation patching** / **path patching**과 같은 기법들이 사용됩니다. forward pass 동안 모델에 인과적으로 개입하여 activation을 0으로 설정하거나(또는 다른 분포에서의 값으로 설정), 그에 따라 하위 메트릭(예: loss 또는 조사 중인 태스크에 특화된 지표)이 어떻게 변하는지 확인할 수 있습니다.

ablation과 activation / path patching에 대해 더 자세히 알고 싶다면 [ARENA IOI exercises](https://arena-chapter1-transformer-interp.streamlit.app/[1.4.1]_Indirect_Object_Identification) 또는 이 장의 후반부에 있는 sparse feature circuit 관련 연습 문제를 참고하시기 바랍니다. 하지만 이번 연습 문제에서는 내용을 단순하게 유지하여, 단일 feature ablation에만 집중하겠습니다.

### 연습 문제 - ablation을 통한 중복 token feature 찾기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

이 연습 문제에서는 ablation을 사용하여 **중복 token feature**를 찾아보겠습니다. 이 feature들은 모델의 초기 레이어(구체적으로 레이어 0 및 3)에서 활성화되며, 특정 token에서 해당 token의 이전 등장 위치로 attention을 보내는 것으로 보입니다. 이들은 logit 출력에 직접적인 영향을 주지는 않지만, 여전히 IOI circuit의 중요한 부분입니다. 따라서 이들을 ablation했을 때 모델의 logit diff에 큰 영향이 있음을 확인할 수 있어야 합니다.

이 ablation을 수행하기 위해 아래 코드를 완성해야 합니다. 다음 작업을 수행하십시오:

- 주어진 위치와 feature의 SAE activation을 0으로 설정(하고 수정된 activation을 반환)하는 `ablate_sae_feature` hook 함수를 완성합니다.
- `(d_sae,)` 모양의 텐서 `ablation_effects`를 생성합니다. 여기서 `i`번째 요소는 `s2_pos` 위치에서 `i`번째 feature를 ablation했을 때의 logit diff 변화량이며, 4개의 prompt에 대해 평균을 낸 값입니다.

이 과정에서 `model.run_with_hooks_with_saes` 함수가 유용하게 사용될 것입니다. 이 함수는 이전 연습 문제에서 했던 것과 같은 `saes` 리스트와, TransformerLens를 사용하며 접했을 법한 `(hook_name, hook_fn)` 튜플들의 리스트인 `fwd_hooks` 리스트를 인자로 받습니다.

In [ ]:
layer = 3
s2_pos = 10
assert gpt2.to_str_tokens(prompts[0])[s2_pos] == " John"


def ablate_sae_latent(
    sae_acts: Tensor,
    hook: HookPoint,
    latent_idx: int | None = None,
    seq_pos: int | None = None,
) -> Tensor:
    """
    Ablate a particular latent at a particular sequence position. If either argument is None, we
    ablate at all latents / sequence positions respectively.
    """
    raise NotImplementedError()


# YOUR CODE HERE - replicate these results with sentiment prompts

<details>
<summary>도움말 - <code>ablation_effects</code>를 어떻게 계산해야 할지 잘 모르겠습니다.</summary>

먼저, 4개의 prompt 전체에서 S2 위치 중 적어도 하나에서 0이 아닌 값을 가지는 모든 feature의 리스트를 얻을 수 있습니다. 이렇게 하면 모든 feature를 일일이 실행할 필요가 없습니다 (당연히 feature가 활성화되지 않았다면 이를 ablate 해도 아무런 효과가 없기 때문입니다!).

다음으로, 해당 feature들을 하나씩 순회하며 ablate 하고, `model.run_with_hooks_with_saes`를 사용하여 logit을 얻은 다음, `logits_to_ave_logit_diff` 함수를 사용하여 logit diff를 구합니다.

마지막으로, 이 값과 SAE를 사용하되 ablation은 수행하지 않았을 때 얻은 logit diff 사이의 차이를 저장하면 됩니다.

</details>

<details>
<summary>솔루션 (및 설명)</summary>

솔루션 코드:

```python
layer = 3
s2_pos = 10
assert gpt2.to_str_tokens(prompts[0])[s2_pos] == " John"


def ablate_sae_latent(
    sae_acts: Tensor,
    hook: HookPoint,
    latent_idx: int | None = None,
    seq_pos: int | None = None,
) -> Tensor:
    """
    Ablate a particular latent at a particular sequence position. If either argument is None, we
    ablate at all latents / sequence positions respectively.
    """
    sae_acts[:, seq_pos, latent_idx] = 0.0
    return sae_acts


_, cache = gpt2.run_with_cache_with_saes(prompts, saes=[attn_saes[layer]])
acts = cache[hook_sae_acts_post := f"{attn_saes[layer].cfg.metadata.hook_name}.hook_sae_acts_post"]

alive_latents = (acts[:, s2_pos] > 0.0).any(dim=0).nonzero().squeeze().tolist()
ablation_effects = t.zeros(attn_saes[layer].cfg.d_sae)

logits = gpt2.run_with_saes(prompts, saes=[attn_saes[layer]])
logit_diff = logits_to_ave_logit_diff(logits)

for i in tqdm(alive_latents, desc="Computing causal effects for ablating each latent"):
    logits_with_ablation = gpt2.run_with_hooks_with_saes(
        prompts,
        saes=[attn_saes[layer]],
        fwd_hooks=[(hook_sae_acts_post, partial(ablate_sae_latent, latent_idx=i, seq_pos=s2_pos))],
    )

    logit_diff_with_ablation = logits_to_ave_logit_diff(logits_with_ablation)
    ablation_effects[i] = logit_diff - logit_diff_with_ablation

px.line(
    ablation_effects.cpu().numpy(),
    title=f"Causal effects of latent ablation on logit diff ({len(alive_latents)} alive)",
    labels={"index": "Latent", "value": "Causal effect on logit diff"},
    template="ggplot2",
    width=1000,
).update_layout(showlegend=False).show()

# Print the top 5 latents, and inspect their dashboards
for value, ind in zip(*ablation_effects.topk(3)):
    print(f"#{ind} had mean act {acts[:, s2_pos, ind].mean():.2f}, causal effect {value:.2f}")
    display_dashboard(
        sae_release="gpt2-small-hook-z-kk",
        sae_id=f"blocks.{layer}.hook_z",
        latent_idx=int(ind),
    )
```

layer 3에서 duplicate token feature로 보이는 몇 가지 feature를 찾을 수 있으며, 이 중 일부(예: 상위 두 개인 7803과 10137)는 이름이나 다른 대문자로 시작하는 단어에서 특히 강한 activation을 보입니다. name mover feature에 사용했던 것과 동일한 방법을 사용하면, 이 feature들이 head `3.0`에 가장 많이 노출되어 있음을 알 수 있습니다 (이는 IOI 논문에서 기대하는 결과입니다).

이상하게도 layer 0 역시 동일한 방법으로 찾을 수 있는 duplicate token feature를 가지고 있지만, 이를 ablate 하면 logit diff가 감소하는 대신 오히려 증가하는 것으로 보입니다. 정확한 이유는 모르겠으나, 동일한 2개의 이름을 가진 4개의 prompt보다 더 큰 데이터셋을 사용한다면 다른 결과가 나올 수도 있습니다. 이 부분은 독자 여러분의 과제로 남겨두겠습니다!

</details>

### Attribution patching

Ablation은 특정 feature의 다운스트림 효과를 측정하는 한 가지 방법입니다. 또 다른 방법은 **attribution patching**으로, 이는 gradient 기반의 attribution 기법이며 ablation이나 다른 종류의 activation patching에 대한 좋은 근사치로 활용될 수 있습니다. 이를 통해 SAE를 이용한 더 효율적이고 확장 가능한 circuit 분석이 가능해집니다. 이는 SAE feature의 수가 매우 많다는 점(기본 모델의 neuron 수나 residual stream 방향 수보다 훨씬 많음)을 고려할 때 특히 가치가 있습니다.

공식은 다음과 같이 작동합니다: 임의의 입력 $x_{\text{clean}}$ 및 $x_{\text{corrupted}}$, 그리고 스칼라 metric 함수 $M = M(x)$(우리의 경우 $M$은 logit 차이입니다)가 있을 때, 1차 근사(first-order approximation)를 사용하여 clean -> corrupt activation으로 patching했을 때의 효과를 다음과 같이 추정할 수 있습니다:

$$
M_{\text{clean}} - M_{\text{corrupted}} \approx (\hat{x}_{\text{clean}} - \hat{x}_{\text{corrupted}}) \cdot \nabla_x M_{\text{corrupted}}
$$

나아가, 여러 개의 독립적인 컴포넌트(한 컴포넌트가 다른 컴포넌트의 함수가 아니라는 의미에서 독립적이며, 예를 들어 모델의 동일한 layer에 있는 feature activation들일 수 있습니다)에 대한 ablation 효과를 한 번에 추정하고 싶다면, 다음과 같이 elementwise multiplication을 수행할 수 있습니다:

$$
\hat{M}_{\text{clean}} - M_\text{corrupted} \approx (\hat{x}_{\text{clean}} - \hat{x}_{\text{corrupted}}) \times \nabla_x M_{\text{corrupted}}
$$

이번 사례에서는 logit 차이 $M_{\text{corrupted}} - M_{\text{clean}}$에 대한 ablation(즉, $\hat{x}_{\text{corrupted}}$을 0으로 설정)의 효과를 근사할 것입니다. ablation은 (약간 수정된 분포로부터의 activation patching 등에 비해) 매우 OOD인 연산이므로, corrupted gradient 대신 clean gradient를 사용하는 더 간단한 버전의 공식을 사용하겠습니다:

$$
\hat{M}_{\text{clean}} - M_\text{ablated} \approx  \hat{x}_{\text{clean}} \times \nabla_x M_{\text{clean}}
$$

다행히도, `TransformerLens`은 `model.add_hook` 메서드를 통해 gradient 계산을 쉽게 만들어 줍니다. 이 메서드는 3가지 중요한 인자를 받습니다:

- 문자열이거나 문자열을 boolean으로 매핑하는 필터 함수일 수 있는 `name`
- 우리의 hook 함수인 `hook`
- forward hook 또는 backward hook 중 무엇을 원하는지 나타내는 문자열 `"fwd"` 또는 `"bwd"`인 `dir`

Backward hook은 계산된 tensor에 대해 `.backward()`을 호출할 때 실행된다는 점을 제외하면 forward hook과 정확히 동일하게 작동합니다. `hook` 함수는 여전히 tensor와 hook 자체라는 2개의 인자를 받으며, 유일한 차이점은 이제 이 tensor가 activation 자체가 아니라 activation에 대한 output의 gradient가 된다는 점입니다. fwd와 bwd 함수에 대한 hook point 이름은 동일합니다.

SAE를 함께 사용할 때도 이 과정은 쉽습니다! `model.add_sae`를 호출하거나(또는 context manager에서 사용하거나) 하면 SAE가 computational graph에 자동으로 삽입되어, attribution patching을 쉽게 구현할 수 있습니다.

### 연습 문제 - ablation과 attribution patching 비교하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> Understanding what's happening is the important thing here; the code itself is pretty short.
> ```

여러분은 아래 코드(`# YOUR CODE HERE`라고 표시된 부분)를 작성하여, feature들에 대한 ablation과 attribution patching의 효과를 비교해야 합니다. 최종 출력물은 각 live feature에 대한 ablation 효과와 attribution patching 값의 관계를 나타내는 선 그래프(line plot)가 될 것이며, 두 값 사이에 거의 정확한 관계가 나타나기를 기대합니다 (결과가 어떻게 나와야 하는지는 솔루션 Colab에서 확인하실 수 있습니다).

여러분을 돕기 위해 `get_cache_fwd_and_bwd` 함수를 제공했습니다. 이 함수는 특정 metric(그리고 추가할 특정 SAE 또는 SAE 리스트)이 주어졌을 때 모델의 forward 및 backward cache를 반환합니다.

아래 코드에 대한 몇 가지 참고 사항입니다:

- 이전 연습 문제에서 `ablation_effects` 텐서를 미리 계산해 두어야 합니다. 그렇지 않으면 코드가 실행되지 않습니다.
- `get_cache_fwd_and_bwd`에 전달되는 metric 함수로 `"mean"` 대신 `reduction="sum"`를 사용했습니다. 그 이유를 알 수 있을까요?

<details>
<summary>정답: 왜 mean이 아닌 sum을 사용하여 reduce하는가</summary>

위의 공식으로 돌아가면, 우리는 $M$이 logit diff metric인 값 $\hat{x}_{\text{clean}} \times \nabla_x M_{\text{clean}}$을 계산하고 있습니다. 우리는 단 한 번의 forward 및 backward pass에서 (각 prompt에 대해 하나씩) 이러한 값 4개의 벡터를 계산한 다음, 해당 벡터에 대해 평균을 내는 방식으로 이를 수행합니다. 하지만 만약 metric에 prompt들에 대한 mean을 사용했다면, 각 sequence는 단독으로 실행했을 때보다 1/4의 gradient만 갖게 됩니다. 이는 결과적으로 $\hat{x}_{\text{clean}} \times \nabla_x M_{\text{clean}} / 4$ 형태의 4개 항에 대해 평균을 내는 것이 되며, 우리가 최종적으로 gradient를 추정하게 될 metric은 전체 logit diff가 아니라 logit diff의 1/4이 됩니다.

이는 이전 섹션에서 toy model의 여러 인스턴스를 동시에 학습시켰을 때 보았던 내용과 관련이 있습니다. 즉, 해당 loss 함수에 대해 한 번 backpropagating 하는 것이 각 인스턴스의 loss 함수에 대해 개별적으로 backpropagating 하는 것과 동일해지려면, loss 함수가 인스턴스들에 대한 loss의 sum이어야 합니다.

</details>

In [ ]:
def get_cache_fwd_and_bwd(
    model: HookedSAETransformer, saes: list[SAE], input, metric
) -> tuple[ActivationCache, ActivationCache]:
    """
    Get forward and backward caches for a model, given a metric.
    """
    filter_sae_acts = lambda name: "hook_sae_acts_post" in name

    # This hook function will store activations in the appropriate cache
    cache_dict = {"fwd": {}, "bwd": {}}

    def cache_hook(act, hook, dir: Literal["fwd", "bwd"]):
        cache_dict[dir][hook.name] = act.detach()

    with model.saes(saes=saes):
        # We add hooks to cache values from the forward and backward pass respectively
        with model.hooks(
            fwd_hooks=[(filter_sae_acts, partial(cache_hook, dir="fwd"))],
            bwd_hooks=[(filter_sae_acts, partial(cache_hook, dir="bwd"))],
        ):
            # Forward pass fills the fwd cache, then backward pass fills the bwd cache (we don't
            # care about metric value)
            metric(model(input)).backward()

    return (
        ActivationCache(cache_dict["fwd"], model),
        ActivationCache(cache_dict["bwd"], model),
    )


clean_logits = gpt2.run_with_saes(prompts, saes=[attn_saes[layer]])
clean_logit_diff = logits_to_ave_logit_diff(clean_logits)

t.set_grad_enabled(True)
clean_cache, clean_grad_cache = get_cache_fwd_and_bwd(
    gpt2,
    [attn_saes[layer]],
    prompts,
    lambda logits: logits_to_ave_logit_diff(logits, keep_as_tensor=True, reduction="sum"),
)
t.set_grad_enabled(False)

# YOUR CODE HERE - compute `attribution_values` from the clean activations & clean grad cache
attribution_values = ...

# Visualize results
px.scatter(
    pd.DataFrame(
        {
            "Ablation": ablation_effects[alive_latents].cpu().numpy(),
            "Attribution Patching": attribution_values.cpu().numpy(),
            "Latent": alive_latents,
        }
    ),
    x="Ablation",
    y="Attribution Patching",
    hover_data=["Latent"],
    title="Attribution Patching vs Ablation",
    template="ggplot2",
    width=800,
    height=600,
).add_shape(
    type="line",
    x0=attribution_values.min(),
    x1=attribution_values.max(),
    y0=attribution_values.min(),
    y1=attribution_values.max(),
    line=dict(color="red", width=2, dash="dash"),
).show()

<details>
<summary>솔루션</summary>

```python
# Extract activations and gradients
hook_sae_acts_post = f"{attn_saes[layer].cfg.metadata.hook_name}.hook_sae_acts_post"
clean_sae_acts_post = clean_cache[hook_sae_acts_post]
clean_grad_sae_acts_post = clean_grad_cache[hook_sae_acts_post]

# Compute attribution values for all features, then index to get live ones
attribution_values = (clean_grad_sae_acts_post * clean_sae_acts_post)[:, s2_pos, alive_features].mean(0)
```

</details>

## GemmaScope

> 참고 - 이 섹션은 일반 Colab에서는 작동하지 않을 수 있으며, Colab Pro 사용을 권장합니다. 여기서 half precision을 사용하는 것도 도움이 될 수 있습니다.

이 섹션의 마지막 연습 문제들을 소개하기 전에, SAE 연구를 희망하는 분들이라면 반드시 알아야 할 Google DeepMind의 최신 sparse autoencoder 릴리스에 대해 잠시 이야기하겠습니다. 2024년 7월 31일에 게시된 관련 [blog post](https://deepmind.google/discover/blog/gemma-scope-helping-the-safety-community-shed-light-on-the-inner-workings-of-language-models/) 에 따르면 다음과 같습니다:

> 오늘 저희는 가벼운 오픈 모델 제품군인 Gemma 2의 내부 작동 원리를 연구자들이 이해하는 것을 돕기 위한 새로운 도구 세트인 Gemma Scope를 발표합니다. Gemma Scope는 Gemma 2 9B 및 Gemma 2 2B를 위해 무료로 제공되는 수백 개의 오픈 sparse autoencoders (SAEs) 모음입니다.

크고 잘 훈련된 sparse autoencoder를 분석하는 데 관심이 있다면, GemmaScope가 현재 사용할 수 있는 가장 좋은 릴리스일 가능성이 높습니다.

먼저 SAE를 로드해 보겠습니다. 여기서는 L0 값(이러한 지표를 어떻게 생각해야 하는지에 대해서는 SAE 훈련 연습 문제를 참조하십시오!)을 기준으로 선택된 GemmaScope SAE들을 다루기 위해 [canonical recommendations](https://opensourcemechanistic.slack.com/archives/C04T79RAW8Z/p1726074445654069) 을 사용합니다. 이 특정 SAE는 Gemma-2-2B 모델의 20번째 layer의 residual stream에서 훈련되었으며, width는 16k이고, **JumpReLU activation function**을 사용합니다. 이 activation function에 대한 자세한 내용은 마지막의 짧은 섹션을 참조하시기 바랍니다. 다만, 지금 당장 세부 사항까지 걱정하실 필요는 없습니다.

이 SAE 모델들에 접근하기 위해서는 몇 가지 단계를 거쳐야 할 것입니다. 다음 과정을 수행하십시오:

1. [gemma-2b HuggingFace repo](https://huggingface.co/google/gemma-2b) 에 방문하여 "Agree and access repository"를 클릭합니다.
2. 접근 권한을 부여받으면, 사용자 설정에서 read token을 생성하여 복사한 다음, `chapter1_transformer_interp/exercises` 디렉토리의 `.env` 파일에 `HF_TOKEN=...` 로 붙여넣습니다. 그 후 아래 셀을 실행하여 로드할 수 있습니다:

In [ ]:
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please set HF_TOKEN in your .env file"

gemma_2_2b = HookedSAETransformer.from_pretrained("gemma-2-2b", device=device)

gemmascope_sae_release = "gemma-scope-2b-pt-res-canonical"
gemmascope_sae_id = "layer_20/width_16k/canonical"
gemma_2_2b_sae = SAE.from_pretrained(gemmascope_sae_release, gemmascope_sae_id, device=str(device))

이 객체들의 config를 살펴보고, 그 구조를 대략적으로 이해했는지 확인하시기 바랍니다. 또한 몇 개의 latent dashboard를 표시하여 latent가 어떻게 생겼는지 감을 잡으실 수도 있습니다.

<details>
<summary>도움말 - "파일을 다운로드하기 위한 여유 디스크 공간이 부족합니다(Not enough free disk space to download the file)"라는 에러가 발생합니다.</summary>

이 경우, 터미널에서 `huggingface-cli delete-cache`을 실행하여(먼저 `pip install huggingface_hub[cli]`를 해야 할 수도 있습니다) huggingface 모델 캐시를 삭제해 공간을 확보해 보시기 바랍니다. 위/아래 화살표 키로 탐색할 수 있는 인터페이스가 나타나며, 스페이스바를 눌러 삭제할 모델을 선택한 후 엔터를 눌러 삭제를 확정할 수 있습니다.

</details>

더 이상 사용하지 않는 모든 모델의 캐시를 삭제한 후에도 위의 에러 메시지가 계속 나타나거나, 모델을 실행할 때 다른 에러(예: OOM)가 발생하는 경우 다음 옵션 중 하나를 권장합니다:

- 지금까지 작업해 온 GPT2-Small 모델의 latent를 선택하여 대신 실습을 진행합니다 (작성 시점에는 GPT2-Medium, Large, 또는 XL 모델에 대해 훈련된 고성능 SAE가 없으나, 이 글을 읽으실 때는 상황이 다를 수 있으며 그럴 경우 해당 모델들을 시도해 보실 수 있습니다!).
- 모델에 32비트 대신 float16 정밀도를 사용합니다 (`from_pretrained` 메서드에 `dtype="float16"`을 전달하면 됩니다).
- 더 강력한 머신을 사용합니다. 예를 들어 vast.ai에서 A100을 대여하거나 Google Colab Pro (또는 Pro+)를 사용하는 방법이 있습니다.

## Feature Steering

> 이 섹션에서는 흥미로운 모델 출력을 생성하기 위해 latent를 steering하는 방법을 배웁니다. 핵심 포인트는 다음과 같습니다:
>
> - Steering은 forward pass 중에 개입하여 모델의 activation을 특정 latent의 방향으로 변경하는 것을 포함합니다.
> - Steering 동작은 때때로 예측 불가능하며, 항상 "latent가 강하게 activation되는 것과 동일한 유형의 텍스트를 생성하는 것"과 동일하지는 않습니다.
> - Neuronpedia에는 코드 없이도 steering을 할 수 있는 steering 인터페이스가 있습니다.

이 연습 문제 세트를 마무리하기 전에, 재미있는 것을 해봅시다!

특정 feature에 해당하는 latent를 찾았다면, 이를 사용하여 **모델을 steering**함으로써 그에 따른 행동 변화를 유도할 수 있습니다. Anthropic의 바이럴 [Golden Gate Claude](https://www.anthropic.com/news/golden-gate-claude) 모델을 통해 이미 접해보셨을 수도 있습니다. Steering은 단순히 forward pass 동안 모델의 activation에 개입하여, feature의 decoder weight의 배수를 residual stream에 더하는 것입니다 (또는 residual stream에 이미 존재하는 성분을 스케일링하거나, 이 성분을 특정 고정 값으로 클램핑하는 방식일 수도 있습니다). 값을 선택할 때는 보통 특정 텍스트 분포에서 해당 feature의 최대 activation 값을 기준으로 삼습니다 (너무 OOD가 되지 않도록 하기 위함입니다).

아쉽게도 GemmaScope SAE로는 Golden Gate Claude를 완전히 재현할 수는 없습니다. "Golden Gate Bridge"와 같은 제목의 문맥에서 특히 "Golden"이라는 단어에 반응하는 것으로 보이는 일부 feature들이 있지만 (예: layer 18 canonical 16k-width residual stream GemmaScope SAE의 [feature 14667](https://www.neuronpedia.org/gemma-2-2b/18-gemmascope-res-16k/14667) 또는 layer 20 SAE의 [feature 1566](https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k/1566)), 이들은 대부분 single-token feature입니다 (즉, Golden Gate Bridge를 논하는 문맥보다는 단순히 "Golden"이라는 단어에만 반응합니다). 따라서 이러한 종류의 행동 변화를 일으키는 효율성은 제한적입니다. 예를 들어, "Golden" 다음에 모델이 "Gate"를 출력하게 만드는 bigram feature를 실제로 찾았다고 가정해 봅시다. 이를 steering하면 결국 모델이 "Gate" token을 끝없이 출력하게 될 것입니다 (실제로 앞서 언급한 두 feature에서 이와 유사한 일이 발생하며, 원하신다면 직접 시도해 보실 수 있습니다). 대신, 우리는 더 나은 **consistent activation heuristic value**를 가진 feature를 찾고자 합니다. 대략적으로 말하자면, 이는 인접한 token들 사이의 feature activation 간의 상관관계이며, 이 값이 높으면 token 수준이 아닌 concept 수준의 feature일 가능성이 큽니다. 구체적으로, 우리는 개에 대한 논의에서 활성화되는 것으로 보이는 "dog feature"를 사용할 것입니다:

In [ ]:
latent_idx = 12082

display_dashboard(sae_release=gemmascope_sae_release, sae_id=gemmascope_sae_id, latent_idx=latent_idx)

### 연습 문제 - `generate_with_steering` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-30 minutes on completing the set of functions below.
> ```

먼저, 아래의 기본 함수 `steering_hook`를 구현해야 합니다. 이 함수는 모델의 forward pass 동안 hook 함수로 추가되며, steering vector(즉, 이 feature에 대한 decoder weight)의 배수 `steering_coefficient`를 activation tensor에 더해야 합니다.

In [ ]:
def steering_hook(
    activations: Float[Tensor, "batch pos d_in"],
    hook: HookPoint,
    sae: SAE,
    latent_idx: int,
    steering_coefficient: float,
) -> Tensor:
    """
    Steers the model by returning a modified activations tensor, with some multiple of the steering
    vector added to all sequence positions.
    """
    return activations + steering_coefficient * sae.W_dec[latent_idx]


tests.test_steering_hook(steering_hook, gemma_2_2b_sae)

이제 `generate_with_steering`을 구현하여 이 실습을 마무리해야 합니다. 이 함수를 실행하여 직접 steering된 출력 텍스트를 생성할 수 있습니다!

<details>
<summary>도움말 - steering을 사용하여 텍스트를 생성하는 모델 구문이 확실하지 않습니다.</summary>

컨텍스트 매니저 내에 hook을 추가한 다음, 다음과 같이 steer할 수 있습니다:

```python
with model.hooks(fwd_hooks=[(hook_name, steering_hook)]):
    output = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        prepend_bos=sae.cfg.prepend_bos,
        **GENERATE_KWARGS
    )
```

`prepend_bos` 인자를 사용하는 것을 잊지 마십시오. 올바른 동작을 얻기 위해 매우 중요한 경우가 많습니다!

`GENERATE_KWARGS` dict에 권장 sampling 파라미터를 제공해 드렸습니다.

출력값은 기본적으로 문자열로 제공됩니다.

</details>

<details>
<summary>도움말 - steering hook을 어디에 추가해야 할지 모르겠습니다.</summary>

SAE에 의해 재구성되는 activation이 `sae.cfg.metadata.hook_name`이므로, 여기에 추가해야 합니다.

</details>

`steering_coefficient`의 값은 우리가 steering하려는 latent의 최대 activation을 기반으로 선택할 수 있다는 점에 유의하십시오 (보통 최대 activation에 가깝게 선택하는 것이 현명하지만, 모델을 distribution 밖으로 너무 멀리 steer할 정도로 높게 설정해서는 안 됩니다. 하지만 이는 latent마다 다르며, 예를 들어 이 특정 latent의 경우 최대 activation 값보다 훨씬 높게 설정해도 여전히 일관된 출력을 생성하는 것을 확인할 수 있습니다). 만약 neuronpedia가 없었다면 이렇게 할 수 없었을 것이며, steering 계수로 어떤 값을 선택할지 안내받기 위해 적절히 큰 데이터셋에 대해 최대 activation을 측정하는 것이 더 나았을 것입니다.

In [ ]:
GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=2.0, verbose=False)


def generate_with_steering(
    model: HookedSAETransformer,
    sae: SAE,
    prompt: str,
    latent_idx: int,
    steering_coefficient: float = 1.0,
    max_new_tokens: int = 50,
):
    """
    Generates text with steering. A multiple of the steering vector (the decoder weight for this
    latent) is added to the last sequence position before every forward pass.
    """
    raise NotImplementedError()


prompt = "When I look at myself in the mirror, I see"
latent_idx = 12082

no_steering_output = gemma_2_2b.generate(prompt, max_new_tokens=50, **GENERATE_KWARGS)

table = Table(show_header=False, show_lines=True, title="Steering Output")
table.add_row("Normal", no_steering_output)
for i in tqdm(range(3), "Generating steered examples..."):
    table.add_row(
        f"Steered #{i}",
        generate_with_steering(
            gemma_2_2b,
            gemma_2_2b_sae,
            prompt,
            latent_idx,
            steering_coefficient=240.0,  # roughly 1.5-2x the latent's max activation
        ).replace("\n", "↵"),
    )
rprint(table)

<details><summary>솔루션</summary>

```python
GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=2.0, verbose=False)


def generate_with_steering(
    model: HookedSAETransformer,
    sae: SAE,
    prompt: str,
    latent_idx: int,
    steering_coefficient: float = 1.0,
    max_new_tokens: int = 50,
):
    """
    Generates text with steering. A multiple of the steering vector (the decoder weight for this
    latent) is added to the last sequence position before every forward pass.
    """
    _steering_hook = partial(
        steering_hook,
        sae=sae,
        latent_idx=latent_idx,
        steering_coefficient=steering_coefficient,
    )

    with model.hooks(fwd_hooks=[(sae.cfg.metadata.hook_name, _steering_hook)]):
        output = model.generate(prompt, max_new_tokens=max_new_tokens, **GENERATE_KWARGS)

    return output
```
</details>

### Neuronpedia를 이용한 Steering

Neuronpedia에는 steering 인터페이스가 실제로 구현되어 있어, 코드를 전혀 작성하지 않고도 특정 latent에 steering을 적용했을 때의 효과를 확인할 수 있습니다! 직접 체험해 보시려면 관련 [Neuronpedia page](https://www.neuronpedia.org/steer) 를 방문해 보시기 바랍니다. "How it works" 버튼에 마우스를 올리면 steering API에서 각 계수(coefficient)가 어떻게 해석되는지 확인할 수 있습니다 (우리가 실험에서 사용한 방식과 매우 유사합니다).

이 latent와 다른 latent들을 사용하여 steering API를 실험해 보시기 바랍니다. 또한 DeepMind의 instruction-tuned Gemma 모델과 같은 다른 모델들도 시도해 볼 수 있습니다. finetuned 모델에 이르면, latent가 무엇에 반응(firing)하는지와 해당 latent를 steering 했을 때의 downstream 효과 사이에 괴리가 발생하는 것과 같은 흥미로운 패턴들이 나타나기 시작합니다. 예를 들어, 특정 종류의 유해하거나 공격적인 언어에 활성화되지만, steering을 적용하면 거절(refusal) 행동을 유도하는 latent를 발견할 수도 있습니다. 아마도 이러한 latent들은 non-finetuned 모델에 존재했으며 당시에는 steering 시 더 유해한 행동으로 유도했겠지만, finetuning 과정에서 그 출력 행동이 다시 학습되었을 가능성이 있습니다. 이는 latent interpretability를 수행할 때의 핵심 아이디어인, latent를 **representation**으로 보는 관점과 **function**으로 보는 관점 사이의 이중성(duality)과 연결됩니다 (이에 대한 자세한 내용은 circuit 섹션을 참조하시기 바랍니다).

## 다른 유형의 SAE들

> 이 섹션에서는 몇 가지 서로 다른 SAE 아키텍처를 소개하며, 그 중 일부는 이후 섹션에서 더 자세히 살펴볼 예정입니다. 여기에는 연습 문제가 없으며, 간단한 설명만 제공됩니다. 핵심 포인트는 다음과 같습니다:
>
> - **TopK**, **JumpReLU**, **Gated** 모델과 같은 서로 다른 activation 함수 / encoder 아키텍처는 feature suppression 문제나 표준 모델에서 SAE가 연속적이어야 한다는 압박과 같은 문제들을 해결할 수 있습니다.
> - **End-to-end SAEs**는 서로 다른 loss 함수로 학습되며, 단순히 MSE reconstruction error를 최소화하는 것이 아니라 모델의 출력에 기능적으로 유용한 feature를 학습하도록 유도합니다.
> - **Meta SAEs**는 SAE activation을 분해하도록 학습된 SAE입니다. (**feature absorption**과 같은 이유로 SAE latent가 항상 monosemantic일 것이라고 기대할 수 없기 때문입니다.)
> - **Transcoders**는 단순히 activation을 재구성하는 것이 아니라 모델의 계산(예: MLP input에서 MLP output으로의 sparse mapping)을 재구성하도록 학습하는 SAE의 한 유형입니다. 이는 때때로 더 쉬운 circuit 분석으로 이어질 수 있습니다.

이 섹션에서는 아직 다루지 않은 다른 종류의 SAE들에 대해 짧게 살펴보겠습니다. 이 중 일부는 이후의 실습에서 탐구하게 될 것입니다. 이 섹션에는 실습 문제가 포함되어 있지 않으며, 그 목적은 파트 1까지만 완료하고 이후의 실습을 계획하지 않는 분들에게 더 완전한 그림을 그려드리기 위함입니다.

여기에서 다루는 주제들은 복잡도 순으로 대략적으로 나열되어 있으며, 표준 SAE의 비교적 간단한 확장부터 시작하여 이후 실습에서 다시 다루게 될 더 복잡한 아이디어들로 마무리됩니다.

### TopK, JumpReLU, Gated

이들은 기본적인 SAE architecture에 대한 세 가지 서로 다른 작은 수정 사항들을 나타내며, 모두 표준 SAE보다 개선된 성능을 제공하는 것으로 보입니다. 각각에 대해 차례대로 설명하겠습니다:

**TopK** SAE는 다른 activation function을 사용합니다: $z = \operatorname{ReLU}(W_{enc}(x-b_{dec}) + b_{enc})$ 대신에 $z = \operatorname{TopK}(W_{enc}(x-b_{dec}))$로 계산하며, 여기서 $\operatorname{TopK}$는 입력 tensor의 상위 $K$개 요소를 반환하고 나머지는 0으로 설정합니다. 이는 $L_1$ penalty([feature suppression](https://www.lesswrong.com/posts/3JuSjTZyMzaSeTxKk/addressing-feature-suppression-in-saes)와 같은 문제를 해결함)의 필요성을 제거하며, $L_0$ 값을 특정 값으로 튜닝하는 대신 직접 설정할 수 있게 해줍니다. 또한 임의의 activation function과 결합하여 사용할 수 있습니다.

**JumpReLU** SAE는 일반적인 ReLU 대신 JumpReLU activation function을 사용합니다. JumpReLU는 추가적인 step과 (종종 학습 가능한) threshold parameter가 포함된 ReLU입니다. 즉, $\operatorname{JumpReLU}_\theta(x) = xH(x-\theta)$이며, 여기서 $\theta$은 학습 가능한 threshold parameter이고 $H$는 Heaviside step function(인수가 양수이면 1, 그렇지 않으면 0)입니다.

직관적으로, 왜 이것이 일반적인 ReLU SAE보다 더 나은 성능을 낼 것이라고 기대할 수 있을까요? 한 가지 이유는 경험적으로 feature들이 "binary가 되고 싶어 하는" 것처럼 보이기 때문입니다. 예를 들어, "이것이 농구에 관한 것인가"와 같은 feature들은 0에서 1 사이의 연속적인 범위를 차지하기보다 "off" 또는 "on"으로 생각하는 것이 더 적절한 경우가 많습니다. 실제로 정확한 coefficient를 재구성하는 것은 중요하며, 이는 종종 특정 feature가 존재한다는 모델의 confidence 등을 나타내는 데 중요한 것으로 보입니다. 하지만 그럼에도 불구하고, 우리는 이상적으로 이러한 불연속성을 학습할 수 있는 architecture를 원합니다.

안타깝게도 JumpReLU는 학습시키기가 어렵습니다(함수의 jump discontinuity로 인해 미분이 불가능하므로, 이를 우회하기 위한 교묘한 방법들을 사용해야 하기 때문입니다). 그 결과, 많은 그룹이 DeepMind의 [initial paper](https://arxiv.org/pdf/2407.14435)을 재현하는 데 실패한 것으로 보입니다.

**Gated** SAE는 (작성 시점 기준) 제안된 기본적인 SAE architecture의 가장 최신 변형이며, 또 다른 최근의 [DeepMind paper](https://deepmind.google/research/publications/88147/)에서 나왔습니다. 이들은 JumpReLU와 동일한 jump-discontinuity 이점을 제공하며(실제로 약간의 weight tying을 통해 Gated SAE가 JumpReLU와 동등함을 보일 수 있습니다), 한 가지 다른 장점도 제공합니다: 바로 **jump discontinuity를 magnitude와 분리(decouple)**한다는 점입니다. JumpReLU 함수에서는 변화시킬 수 있는 축이 하나뿐이지만, 이상적으로는 feature가 on 또는 off 되어야 하는지 여부와 on 되었을 때의 magnitude가 얼마여야 하는지를 독립적으로 결정할 수 있는 자유도가 필요합니다. Gated SAE는 magnitude 계산을 위한 것과 masking을 위한 것, 이렇게 2개의 별도 encoder weight matrix를 가짐으로써 이를 달성합니다. JumpReLU와 마찬가지로 이들 또한 불연속적이며 특별한 training objective function이 필요하지만, JumpReLU와 달리 일반적으로 학습시키기가 훨씬 쉽다는 것이 증명되었습니다. Neel의 [Extremely Opinionated Annotated List of My Favourite Mechanistic Interpretability Papers](https://www.alignmentforum.org/posts/NfFST5Mio7BCAQHPA/an-extremely-opinionated-annotated-list-of-my-favourite#Sparse_Autoencoders)에서:

> "저는 (매우 편향되게) [[the DeepMind paper](https://deepmind.google/research/publications/88147/)]이 SAE의 변경 사항이 개선되었는지를 엄격하게 평가하는 방법의 좋은 예시로서 읽어볼 가치가 있다고 생각하며, 가능하다면 Gated SAE를 사용할 것을 권장합니다."

### End-to-End SAEs

논문 [Identifying Functionally Important Features with End-to-End Sparse Dictionary Learning](https://www.lesswrong.com/posts/xzJK3nENopiLmo77H/identifying-functionally-important-features-with-end-to-end)에서 저자들은 표준 SAE를 훈련시키는 다른 방법을 제안합니다. activation의 평균 제곱 재구성 오차(MSE)를 훈련 목표로 사용하는 대신, 원래의 output logit과 SAE 출력을 네트워크의 나머지 부분에 통과시켜 얻은 output logit 사이의 KL divergence를 사용합니다. 여기서의 직관은 훈련 분포에서 모델의 동작을 설명하는 데 실제로 중요한 **기능적으로 중요한 feature**들을 식별하고자 하는 것입니다. MSE를 최소화하는 것이 이에 대한 좋은 휴리스틱이 될 수 있지만(중요한 feature들은 종종 높은 magnitude로 표현되어야 하기 때문입니다), 이는 우리가 측정하고자 하는 것을 직접적으로 겨냥하는 것이 아니며, **Goodhart의 법칙**("측정 지표가 목표가 되면, 더 이상 좋은 지표가 되지 않는다")에 취약하다고 볼 수 있습니다.

전체 논문에는 표준(local) SAE와 비교하여 end-to-end (e2e) SAE로 실행한 실험 결과가 포함되어 있습니다. 저자들은 e2e SAE가 동일한 수준의 모델 성능을 포착하면서도 더 작은 L0를 요구하는 경향이 있음을 발견했습니다. 비록 레이어당 MSE는 훨씬 더 크지만 말입니다 (저자들은 이를 완화하는 몇 가지 방법을 제안하며, local 목표와 end-to-end 목표 사이의 균형을 찾았습니다).

### Meta SAEs

Meta SAEs는 일반적인 SAE의 decoder direction을 재구성하도록 훈련된 특수한 형태의 SAE입니다. 이를 통해 SAE latent가 monosemantic하지 않은 상황에서도 base SAE latent의 sparse한 재구성을 찾을 수 있습니다. SAE latent가 monosemantic하지 않을 수 있는 한 가지 이유는 **feature absorption** 때문입니다. 예를 들어, SAE가 "e로 시작함"과 같은 feature를 학습했지만, "elephant"라는 단어에 대해 이미 "elephant"를 위한 feature가 학습되어 "e로 시작함"이라는 정보를 흡수해 버렸다면, 해당 feature는 "elephant"에서 활성화되지 않을 수 있습니다. 이는 모델의 sparsity 측면에서는 더 유리하지만( "e로 시작함" feature가 활성화되는 단어가 하나 줄어들기 때문입니다), 불행히도 feature가 monosemantic해지는 것을 방해하며, SAE가 인과적 매개체(causal mediators)의 sparse한 집합으로 분해하는 것을 어렵게 만듭니다.

Meta-SAEs에 관한 [paper](https://www.lesswrong.com/posts/TMAmHh4DdMr4nCSr5/showing-sae-latents-are-not-atomic-using-meta-saes) 에서는 다음과 같은 핵심 결과들을 발견했습니다:

> - SAE latent는 더 원자적이고 해석 가능한 meta-latent로 분해될 수 있습니다.
> - 더 큰 SAE의 latent가 더 작은 SAE의 latent로부터 분리되었을 때, 더 큰 SAE로 훈련된 meta SAE가 종종 이러한 구조를 복원한다는 것을 보여줍니다.
> - 특정 지식 편집(knowledge editing) 작업에서 meta-latent가 SAE latent보다 모델 동작에 대해 더 정밀한 인과적 개입(causal interventions)을 가능하게 함을 입증합니다.

저자들이 구축한 [dashboard](https://metasae.streamlit.app/?page=Feature+Explorer&feature=11329) 을 방문하여 meta-SAE latent를 탐색해 볼 수 있습니다.

### Transcoders

우리가 살펴본 MLP-layer SAE들은 activation을 feature 벡터들의 희소 선형 결합(sparse linear combination)으로 표현하려고 시도합니다. 중요한 점은, 이들이 **모델의 단일 지점**에 있는 activation에 대해서만 작동한다는 것입니다. 이들은 실제로 MLP layer의 연산을 수행하는 법을 배우는 것이 아니라, 그 연산의 결과를 재구성(reconstruct)하는 법을 배웁니다. 표준 SAE를 사용해서는 superposition 상태인 MLP layer에 대해 가중치 기반 분석을 수행하기가 매우 어렵습니다. 많은 feature들이 neuron basis에서 매우 조밀(dense)하게 나타나며, 이는 neuron들을 분해하기 어렵다는 것을 의미하기 때문입니다.

이와 대조적으로, **transcoder**는 MLP layer 이전의 activation(즉, 정규화되었을 가능성이 있는 residual stream 값들)을 입력으로 받아, 해당 MLP layer의 post-MLP activation을 다시 feature 벡터들의 희소 선형 결합으로 표현하는 것을 목표로 합니다. transcoder라는 용어가 가장 일반적이지만, 이들은 **input-output SAE**(기본 모델 layer의 입력을 받아 출력을 학습하려고 하기 때문) 또는 **predicting future activations**(명백한 이유로)라고도 불립니다. 엄밀히 말하면 transcoder는 재구성이 아닌 매핑을 학습하는 것이므로 autoencoder가 아니지만, SAE에서 얻은 많은 직관들이 transcoder에도 그대로 적용됩니다.

왜 transcoder가 표준 SAE보다 개선된 방식일 수 있을까요? 주로, 모델 layer의 기능에 대해 훨씬 더 명확한 통찰을 제공하기 때문입니다. [Transcoders LessWrong post](https://www.lesswrong.com/posts/YmkjnWtZGLbHRbzrP/transcoders-enable-fine-grained-interpretable-circuit)에서 다음과 같이 설명합니다:

> transcoder의 강점 중 하나는 MLP layer의 기능을 희소하고, 독립적으로 변화하며, 의미 있는 단위들(superposition이 발견되기 전 neuron들이 원래 의도되었던 모습과 같은)로 분해한다는 점입니다. 이는 circuit 분석을 상당히 단순화합니다.

직관적으로는 transcoder가 단순히 출력을 재현하는 것이 아니라 MLP의 연산을 모방하려고 시도하므로, 더 다른(더 복잡한) 종류의 최적화 문제를 해결하는 것처럼 보일 수 있으며, 따라서 표준 SAE에 비해 성능상의 트레이드오프가 있을 것이라고 생각할 수 있습니다. 하지만 여러 증거들은 그렇지 않을 수 있음을 시사하며, transcoder가 표준 SAE보다 파레토 개선(pareto improvement)을 제공할 수 있음을 보여줍니다.

연습 문제 세트 **1.4.2 SAE Circuits**에서 transcoder에 대해 더 깊이 파고들며, 이것들이 어떻게 그리고 왜 그렇게 잘 작동하는지 살펴보겠습니다.

## 보너스

이제 이 섹션의 주요 내용을 모두 마쳤습니다! 이 시점에서 여러분이 가장 관심 있는 이후 섹션으로 바로 넘어가시는 것을 추천하지만, 이번 연습 세트에서 다룬 일부 주제를 더 깊게 파고들고 싶다면 아래의 보너스 섹션들을 시도해 보실 수도 있습니다.

### [Not all Language Model Features are Linear](https://arxiv.org/abs/2405.14860)의 원형 서브스페이스 기하학(circular subspace geometry) 재현하기

latent dashboard를 재현하면서, 우리는 [Not All Language Model Features are Linear](https://arxiv.org/abs/2405.14860)의 일부 분석을 재현하는 데 필요한 코드와 매우 유사한 코드를 작성했습니다. 이 논문에서 저자들은 GPT2 Small에서 요일을 나타내는 latent들의 멋진 원형 표현(circular representation)을 보여줍니다.

이제 이미 작성한 코드를 대부분 사용하여 원형 기하학 결과를 재현할 수 있습니다. 최종 목표는 [the paper](https://arxiv.org/abs/2405.14860) 첫 페이지의 Figure 1과 같은 플롯을 생성하는 것입니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/circular-days.png" width="700">

가이드로서, 다음 단계를 수행해야 합니다:

1. 저자들이 식별한 모든 요일 latent의 activation을 가져옵니다 (아래 리스트에 요일뿐만 아니라 월 및 20세기 연도에 대한 latent도 포함해 두었습니다). 이 latent들은 우리가 작업해 온 SAE, 즉 release `"gpt2-small-res-jb"` 및 id `"blocks.7.hook_resid_pre"`에 해당한다는 점에 유의하십시오.
2. 해당 latent 중 적어도 하나가 활성화된 모든 token에 대해, 해당 latent 서브셋만으로 SAE reconstruction을 계산합니다 (다시 말해, latent 리스트에서 0이 아닌 latent들의 activation을 SAE decoder를 통해 매핑합니다). 또한 token을 저장하십시오. token들을 7일의 요일 중 하나 또는 "Other"의 8개 그룹 중 하나로 그룹화하여 단순화할 수 있습니다 (Figure 1의 예시를 참조하십시오).
3. 모든 SAE reconstruction에 대해 PCA를 수행하고, 두 번째와 세 번째 주성분(principal components)을 플롯합니다. 2D 평면에서 요일들이 원을 형성하는 원형 기하학을 관찰할 수 있을 것입니다 (월이나 20세기 연도에 대해서도 시도한다면 마찬가지 결과가 나옵니다).

또한 요일과 같은 latent를 표현할 때 왜 원형 기하학이 유용할 수 있는지 생각해보시기를 권장합니다. 이 분야에 관심이 있다면 논문 전체를 읽어보시는 것을 강력히 추천합니다!

In [ ]:
from sklearn.decomposition import PCA

day_of_the_week_latents = [2592, 4445, 4663, 4733, 6531, 8179, 9566, 20927, 24185]
# months_of_the_year = [3977, 4140, 5993, 7299, 9104, 9401, 10449, 11196, 12661, 14715, 17068, 17528, 19589, 21033, 22043, 23304]
# years_of_20th_century = [1052, 2753, 4427, 6382, 8314, 9576, 9606, 13551, 19734, 20349]

# YOUR CODE HERE - replicate circular subspace geometry, for `day_of_the_week_latents`

<details>
<summary>여기서 막히신다면, 이 드롭다운을 사용하여 PCA 코드를 확인하시기 바랍니다.</summary>

`all_reconstructions`이 `(n_datapoints, d_model)` 모양의 tensor라고 가정할 때, 이 코드는 reconstruction의 처음 3개 principal components를 포함하는 `(n_datapoints, 3)` 모양의 tensor를 생성합니다:

```python
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
pca_embedding = pca.fit_transform(all_reconstructions.detach().cpu().numpy())
```

</details>

<details>
<summary>여기서 막히신다면, 이 드롭다운을 사용하여 결과 시각화를 위한 코드를 확인하시기 바랍니다.</summary>

이 코드는 `pca_df`이 다음과 같은 컬럼을 가진 dataframe이라고 가정할 때 작동합니다:

- `PC2` 및 `PC3`: SAE reconstruction의 2번째와 3번째 principal components를 포함합니다 (요일 feature로 제한됨)
- `token`: reconstruction이 추출된 token을 포함합니다
- `token_group`: `token`과 동일하지만, 요일이 아닌 모든 token은 "Other"로 대체되었습니다
- `context`: 각 token 주변의 context 문자열입니다 (이는 선택 사항이며, 원치 않으시면 제거할 수 있습니다)

```python
px.scatter(
    pca_df,
    x="PC2",
    y="PC3",
    hover_data=["context"],
    hover_name="token",
    height=700,
    width=1000,
    color="token_group",
    color_discrete_sequence=px.colors.sample_colorscale("Viridis", 7) + ["#aaa"],
    title="PCA Subspace Reconstructions",
    labels={"token_group": "Activating token"},
    category_orders={"token_group": days_of_the_week + ["Other"]},
).show()
```

</details>


<details><summary>정답</summary>

```python
from sklearn.decomposition import PCA

day_of_the_week_latents = [2592, 4445, 4663, 4733, 6531, 8179, 9566, 20927, 24185]
# months_of_the_year = [3977, 4140, 5993, 7299, 9104, 9401, 10449, 11196, 12661, 14715, 17068, 17528, 19589, 21033, 22043, 23304]
# years_of_20th_century = [1052, 2753, 4427, 6382, 8314, 9576, 9606, 13551, 19734, 20349]

days_of_the_week = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
buffer = 5
seq_len = gpt2_act_store.context_size
sae_acts_post_hook_name = f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"

all_data = {"recons": [], "context": [], "token": [], "token_group": []}
total_batches = 400

for i in tqdm(range(total_batches), desc="Computing activations data for PCA, over all batches"):
    _, cache = gpt2.run_with_cache_with_saes(
        tokens := gpt2_act_store.get_batch_tokens(),
        saes=[gpt2_sae],
        stop_at_layer=_get_hook_layer(gpt2_sae) + 1,
        names_filter=[sae_acts_post_hook_name],
    )
    acts = cache[sae_acts_post_hook_name][..., day_of_the_week_latents].flatten(0, 1)

    any_latent_fired = (acts > 0).any(dim=1)
    acts = acts[any_latent_fired]
    reconstructions = acts @ gpt2_sae.W_dec[day_of_the_week_latents]

    all_data["recons"].append(reconstructions)

    for batch_seq_flat_idx in t.nonzero(any_latent_fired).squeeze(-1).tolist():
        batch, seq = divmod(batch_seq_flat_idx, seq_len)  # type: ignore

        token = gpt2.tokenizer.decode(tokens[batch, seq])  # type: ignore
        token_group = token.strip() if token.strip() in days_of_the_week else "Other"

        context = gpt2.tokenizer.decode(  # type: ignore
            tokens[batch, max(seq - buffer, 0) : min(seq + buffer + 1, seq_len)]
        )

        all_data["context"].append(context)
        all_data["token"].append(token)
        all_data["token_group"].append(token_group)

pca = PCA(n_components=3)
pca_embedding = pca.fit_transform(t.concat(all_data.pop("recons")).detach().cpu().numpy())

px.scatter(
    pd.DataFrame(all_data | {"PC2": pca_embedding[:, 1], "PC3": pca_embedding[:, 2]}),
    x="PC2",
    y="PC3",
    hover_data=["context"],
    hover_name="token",
    height=700,
    width=1000,
    color="token_group",
    color_discrete_sequence=px.colors.sample_colorscale("Viridis", 7) + ["#aaa"],
    title="PCA Subspace Reconstructions",
    labels={"token_group": "Activating token"},
    category_orders={"token_group": days_of_the_week + ["Other"]},
).show()
```
</details>

### 긴 접두사와 짧은 접두사 induction

저자들은 [LessWrong post](https://www.lesswrong.com/posts/xmegeW5mqiBsvoaim/we-inspected-every-head-in-gpt-2-small-using-saes-so-you-don#Case_Study__Long_Prefix_Induction_Head)에서 GPT2-Small의 induction feature를 발견했을 뿐만 아니라, SAE latent가 두 개의 L5 induction head의 서로 다른 역할을 유의미하게 구분함으로써 head의 역할에 대해 의미 있는 정보를 제공할 수 있다는 점을 발견했습니다. 그들은 이 레이어의 2개 induction head(5.1 및 5.5) 중, 하나의 head는 "long prefix induction"에 특화되어 있고 다른 하나는 주로 "standard induction"을 수행한다는 것을 발견했습니다.

예를 들어, 저자들은 각각 head 5.1과 5.5에 주로 기여하는 두 개의 서로 다른 latent를 발견했습니다. 이 latent들은 모두 `"-"`로 연결된 표현들에 attention을 보내지만, 적용 방식에는 약간의 차이가 있습니다. 그중 하나의 latent는 주로 "short prefix induction"을 수행합니다. 즉, induction 패턴에 있다는 것을 추론할 수 있는 긴 접두사가 없는 induction입니다. 예를 들면 다음과 같습니다:

- `"Indo-German ... Indo-"` → `"German"`
- `"center-left ... center-"` → `"left"`

다른 하나는 주로 "long prefix induction"을 수행합니다. 즉, induction 패턴의 두 번째 절반이 한동안 지속된 경우의 induction입니다. 예를 들면 다음과 같습니다:

- `"Ways To Prevent Computer-Related Eye Strain ... Ways To Prevent Computer-"` → `"Related"`
- `"shooting, a number of NRA-supported legislators ... a number of NRA-"` → `"supported"`

해당하는 두 latent를 찾고, 어떤 head가 각각 어디에 해당하는지 알아낼 수 있습니까? "finding latents for features" 섹션의 여러 가지 서로 다른 기법들을 사용하여 이 latent들을 찾을 수 있습니까?

In [ ]:
# YOUR CODE HERE - replicate long and short-prefix induction results

<details><summary>솔루션</summary>

```python
induction_prompts = {
    "long_form": [
        "To reduce the risk of computer-related injuries, it's important to maintain proper posture and take regular breaks. To reduce the risk of computer",
        "observed that many people suffer from stress-induced headaches, which can be alleviated through relaxation techniques. And because many people suffer from stress",
        "Experts are increasingly worried about the impact of technology-driven automation on jobs. Experts are increasingly worried about the impact of technology",
    ],
    "short_form": [
        "A lot of NRA-supported legislation has been controversial. Furthermore, NRA",
        "The company is pursuing technology-driven solutions. This is because technology",
        "Humanity is part-angel, part",
    ],
}

layer = 5
sae_acts_post_hook_name = f"{attn_saes[layer].cfg.metadata.hook_name}.hook_sae_acts_post"

logit_dir = gpt2.W_U[:, gpt2.to_single_token("-")]

for induction_type in ["long_form", "short_form"]:
    prompts = induction_prompts[induction_type]
    _, cache = gpt2.run_with_cache_with_saes(
        prompts, saes=[attn_saes[layer]], names_filter=[sae_acts_post_hook_name]
    )
    sae_acts_post = cache[sae_acts_post_hook_name][:, -1, :].mean(0)
    alive_latents = sae_acts_post.nonzero().squeeze().tolist()

    sae_attribution = sae_acts_post * (attn_saes[layer].W_dec @ gpt2.W_O[layer].flatten(0, 1) @ logit_dir)

    ind = sae_attribution.argmax().item()
    latent_dir = attn_saes[layer].W_dec[ind]
    norm_per_head = latent_dir.reshape(gpt2.cfg.n_heads, gpt2.cfg.d_head).pow(2).sum(-1).sqrt()
    norm_frac_per_head = norm_per_head / norm_per_head.sum(-1, keepdim=True)
    top_head_values, top_heads = norm_frac_per_head.topk(2, dim=-1)

    print(
        f"Top latent ({induction_type})\n"
        + tabulate(
            [
                ["Latent idx", ind],
                ["Attribution", f"{sae_attribution[ind]:.3f}"],
                ["Activation", f"{sae_acts_post[ind]:.3f}"],
                ["Top head", f"5.{top_heads[0]} ({top_head_values[0]:.2%})"],
                ["Second head", f"5.{top_heads[1]} ({top_head_values[1]:.2%})"],
            ],
            tablefmt="simple_outline",
        ),
    )

    # Line chart of latent attributions
    px.line(
        sae_attribution.cpu().numpy(),
        title=f"Attributions for correct token ({induction_type} induction) at final token position ({len(alive_latents)} non-zero attribution)",
        labels={"index": "Latent", "value": "Attribution"},
        template="ggplot2",
        width=1000,
    ).update_layout(showlegend=False).show()

    # Display dashboard
    display_dashboard(
        sae_release="gpt2-small-hook-z-kk",
        sae_id=f"blocks.{layer}.hook_z",
        latent_idx=int(ind),
    )
```
</details>

# 2️⃣ latent 이해하기: 심층 분석

> ##### 학습 목표
>
> - feature splitting을 학습하고, 이것이 SAE training에 어떤 의미를 갖는지 공부합니다.
> - UMAP 및 기타 차원 축소 기법을 사용하여 SAE latent geometry를 더 잘 이해합니다.
> - feature absorption을 이해하고, meta-SAE가 이 문제를 해결하는 데 어떻게 도움이 될 수 있는지 학습합니다 (아직 구현되지 않음).
> - logit lens 및 token enrichment analysis와 같은 기법을 사용하여 SAE latent를 더 잘 이해하고 특성화합니다 (아직 구현되지 않음).
> - autointerp 기반의 eval 및 patch scoping을 포함하여 automated interpretability를 심층적으로 분석합니다 (아직 구현되지 않음).

용어 참고 - 이 문맥에서는 "feature splitting"과 "feature absorption"이라는 표현을 사용합니다. 이는 데이터의 기저 feature들이 서로 다른 SAE latent들에 어떻게 나뉘어 분포하는지를 다루고 있기 때문입니다. 즉, latent가 아니라 feature 자체가 분리되는 것입니다. 마찬가지로, "feature absorption"은 기저 feature들이 서로 다른 latent들에 어떻게 분포하는지에 대한 현상을 설명하므로, 이를 "latent absorption"이라고 부르지 않습니다.

## 서론

이 섹션에서는 첫 번째 섹션에서 다루지 않았으며 circuit과 직접적인 관련이 없는 몇 가지 SAE 주제들을 더 깊이 있게 살펴보겠습니다. 일반적으로 이러한 주제들은 다음 두 가지 카테고리 중 하나에 속합니다:

- SAE latent 간의 관계 및 그 기하학적 구조 이해 (예: feature splitting 및 absorption)
- 개별 latent의 역할을 더 잘 이해하기 위해 세부 분석 (예: logit lens 및 automated interpretability와 같은 도구 사용)

이 두 가지 유형의 분석 모두 SAE를 더 잘 이해하는 데 도움이 됩니다. 하지만 SAE를 다룰 때는 다양한 증거들을 함께 고려하는 것이 중요합니다. 개별적인 증거는 종종 오해의 소지가 있거나 다른 증거보다 유용성이 떨어질 수 있기 때문입니다. 예를 들어, 어떤 latent들은 더 좁은(narrower) SAE의 단일 latent에서 분리된 그룹의 일부로 볼 때 훨씬 더 잘 이해되는 경우가 있는 반면, 어떤 latent들은 너무 중요하지 않아서 좁은 SAE에는 나타나지 않을 수도 있습니다. 또한, automated interpretability를 적용했을 때 매우 명확하고 해석 가능해 보이는 latent가 있더라도, 더 깊이 파고들면 그 설명이 false negative를 생성하거나 중요한 세부 사항을 놓치고 있다는 것이 드러나기도 합니다.

이 섹션의 마지막에는, 특정 SAE의 단일 latent를 선택하여 앞서 언급한 모든 다양한 증거들을 사용해 가능한 한 깊게 이해해 보는 개방형 연습 문제를 제안하겠습니다. 이 연습은 다양한 도구와 방법론을 사용하는 기술을 연습하는 좋은 방법일 뿐만 아니라, SAE interpretability(또는 일반적인 interpretability)를 수행할 때 더 신중하고 회의적인 연구 방법론을 개발하는 방법이므로 강력히 추천합니다.

## Feature Splitting

Feature splitting은 SAE 연구 초기 단계에서 발견된 특히 흥미로운 모티프 중 하나입니다. Anthropic의 "Towards Monosemanticity" 논문에서 다음과 같이 설명합니다:

> 우리가 발견한 feature들의 한 가지 놀라운 점은 그것들이 클러스터 형태로 나타난다는 것입니다. 예를 들어, 위에서 여러 개의 base64 feature, 여러 개의 아랍어 스크립트 feature 등을 관찰했습니다. 학습된 sparse feature의 총 개수를 늘릴수록 이러한 feature들이 더 많이 발견되는데, 우리는 이 현상을 feature splitting이라고 부릅니다. A/0의 512개 feature에서 A/1의 4,096개, 그리고 A/2의 16,384개 feature로 늘어남에 따라, base64 컨텍스트에 특화된 feature의 수는 1개에서 3개, 그리고 훨씬 더 많은 수로 증가합니다.

그들은 서로 다른 SAE들에서 얻은 latent들의 결합 집합에 대해 **2D UMAP**을 생성함으로써 feature splitting 현상을 더 자세히 분석합니다. [UMAP](https://umap-learn.readthedocs.io/en/latest/) (Uniform Manifold Approximation and Projection)은 t-SNE와 유사하게 시각화에 사용될 수 있는 차원 축소 기법이며, 일반적인 비선형 차원 축소에도 사용될 수 있습니다. Anthropic은 UMAP에서 흥미로운 기하학적 구조를 관찰했으며, 이는 모델의 latent space 내의 의미론적 구조와 일치하는 것으로 보입니다 (다시 말해, 유사한 의미를 가진 latent들은 그 dictionary vector 사이의 각도가 작습니다).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/feature-splitting.png" width="700">

다음 실습에서는 이러한 정성적인 UMAP 결과 중 일부를 재현하고, latent space의 기하학적 구조가 latent 해석의 유사성과 어떻게 연결되는지에 대한 직관을 쌓아보겠습니다.

먼저, 모델과 SAE를 로드하겠습니다. GPT2-Small과, feature splitting을 탐색하기 위해 다양한 width로 학습된 Joseph Bloom의 모델 세트를 사용하겠습니다.

In [ ]:
sae_release = "gpt2-small-res-jb-feature-splitting"

widths = [768 * (2**n) for n in range(7)]  # Note, you can increase to 8 if it fits on your GPU
sae_ids = [f"blocks.8.hook_resid_pre_{width}" for width in widths]

splitting_saes = {
    width: SAE.from_pretrained(sae_release, sae_id, device=str(device)) for width, sae_id in zip(widths, sae_ids)
}

gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

UMAP 결과를 쉽게 이해하실 수 있도록, 각 SAE에 대한 autointerp 설명을 함께 불러오겠습니다. 중복 항목을 제거하는 함수를 제공해 드렸으므로, latent당 하나의 설명만 가져오게 된다는 점에 유의하시기 바랍니다.

In [ ]:
def load_and_process_autointerp_dfs(width: int):
    # Load in dataframe
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[f"blocks.8.hook_resid_pre_{width}"]
    url = "https://www.neuronpedia.org/api/explanation/export?modelId={}&saeId={}".format(*neuronpedia_id.split("/"))
    headers = {"Content-Type": "application/json"}
    data = requests.get(url, headers=headers).json()
    df = pd.DataFrame(data)

    # Drop duplicate latent descriptions
    df["index"] = df["index"].astype(int)
    df = df.drop_duplicates(subset=["index"], keep="first").sort_values("index", ignore_index=True)

    # Fill in missing latent descriptions with empty strings
    full_index = pd.DataFrame({"index": range(width)})
    df = full_index.merge(df, on="index", how="left")
    df["description"] = df["description"].fillna("")
    print(f"Loaded autointerp df for {width=}")
    if (n_missing := (df["description"] == "").sum()) > 0:
        print(f"  Warning: {n_missing}/{len(df)} latents missing descriptions")

    return df


autointerp_dfs = {width: load_and_process_autointerp_dfs(width) for width in widths}
display(autointerp_dfs[768].head())

### 연습 문제 - 이 SAE들을 학습해 보세요

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

neuronpedia 페이지를 사용하거나(또는 `display_dashboard`을 사용하여 인라인 latent 대시보드를 표시하여) 이 SAE들의 latent를 학습해 보세요. 이 세트 내의 SAE 쌍 사이에서 latent splitting의 사례를 찾아보시기 바랍니다. 이것이 Anthropic의 "Towards Monosemanticity" 포스트에 나온 latent splitting 사례들과 비슷해 보이나요?

<details>
<summary>이 연습 문제에 접근하는 몇 가지 아이디어를 확인하려면 이 드롭다운을 사용하세요</summary>

- narrow SAE의 latent에서 시작하여, narrow-SAE latent를 가장 강하게 활성화하는 예시들에서도 함께 활성화되는 wide SAE의 latent를 찾아볼 수 있습니다.
- 특정 sequence를 가져와 모델과 SAE에 통과시킨 후, 양쪽 모두에서 활성화되는 latent를 찾을 수 있습니다.
- autointerp 설명을 사용할 수 있습니다. wide SAE와 small SAE 모두에서 특정 설명과 일치하는 것으로 보이는 latent의 교집합을 검색합니다. (예를 들어 [OpenAI's text embedding models](https://platform.openai.com/docs/guides/embeddings)의 embedding similarity를 사용하여 주어진 설명과 유사한 autointerp 설명 세트를 찾을 수 있음에 유의하세요)

</details>

### 연습 문제 - UMAP 생성하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 30-45 minutes on this exercise.
> ```

이제 아래 함수를 완성하여 서로 다른 SAE들의 UMAP을 생성해야 합니다. 그 후 셀을 실행하여 UMAP 결과를 플롯할 수 있습니다.

UMAP을 지정하고 fitting하는 코드는 다음과 같습니다:

```python
umap = UMAP(
    n_components=n_components,
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    metric="cosine",
)
umap_embedding = umap.fit_transform(values)
```

여기서 `values`는 `n_data`개의 개별 벡터로 구성된 `(n_data, d)` 모양의 tensor입니다 (cuda가 아닌 cpu에 있는지 확인하십시오). 출력값은 각 벡터의 UMAP 좌표를 포함하는 `(n_data, n_components)` 모양의 array입니다. 실행하는 데 시간이 다소 걸릴 수 있으며(최대 약 1분), 즉시 결과가 나오지 않더라도 걱정하지 마십시오. 원하신다면 데이터의 작은 부분 집합(예: 100개의 latent)에 대해서만 실행하도록 함수를 수정하여 출력이 적절한지 테스트해 볼 수 있습니다. 하지만 우리의 특수한 상황에서는 각 UMAP에 소요되는 시간이 `O(n_data ** 2)`보다 훨씬 느리게 증가하므로, 이 섹션에서 로드할 모든 SAE에 대해 맵을 생성하는 것이 가능할 것입니다.

2D UMAP을 생성하는 경우 `n_components=2`로 설정해야 합니다. `n_neighbors` 및 `min_dist` 인자는 UMAP 알고리즘의 하이퍼파라미터로, 지역적 구조(각 점이 가지는 이웃의 수)와 전역적 구조(점들 사이의 거리) 사이의 트레이드오프를 제어합니다. 이에 대한 자세한 내용은 `UMAP` docstring에서 확인할 수 있습니다. 2D UMAP의 경우, 이 인자들로 `n_neighbors_visual` 및 `min_dist_visual`를 사용할 수 있습니다. `n_neighbors_cluster` 등 다른 인자들이 어떻게 사용되는지는 다음 연습 문제를 참조하십시오.

마지막 참고 사항으로, 플롯을 더 쉽게 해석할 수 있도록 hoverdata에 `"top_token_strs"` 및 `"description"` 필드를 포함했습니다. 전자는 각 latent에 대해 가장 많이 boosted된 상위 10개 token(즉, decoder matrix를 모델의 unembedding에 통과시켰을 때 가장 큰 값들)이어야 합니다. 후자는 위 코드에서 정의된 `autointerp_dfs`의 해당 dataframe에서 가져온 autointerp 설명이어야 합니다. 이 두 가지 모두 권장되지만 선택 사항입니다. 이를 처리하고 싶지 않다면 플로팅 함수의 `hoverdata` 인자에서 해당 필드를 주석 처리하면 됩니다.

In [ ]:
import hdbscan
from umap import UMAP


def compute_sae_umap_data(
    saes: dict[int, SAE],
    autointerp_dfs: dict[int, pd.DataFrame],
    sae_widths: list[int],
    model: HookedSAETransformer,
    n_neighbors_visual: int = 15,
    min_dist_visual: float = 0.05,
    find_clusters: bool = False,
    n_neighbors_cluster: float = 15,
    min_dist_cluster: float = 0.1,
    min_cluster_size: int = 3,
    batch_size: int = 1000,
) -> pd.DataFrame:
    """
    This function will return a dataframe containing umap coordinates & other data (you can then use
    this to create a plot using the code immediately below). The UMAP calculation is done over
    multiple SAEs simultaneously, for comparison.

    Expected dataframe columns:
        sae_width: int
            The width of the SAE that this latent belongs to
        latent_idx: int
            The index of the latent
        umap_x: float
            The x-coordinate of the latent in the UMAP embedding
        umap_y: float
            The y-coordinate of the latent in the UMAP embedding
        autointerp: str
            The autointerp description of this latent
        top_token_strs_formatted: str
            The top 10 tokens that the latent is activated by

    Args:
        saes: dict[int, SAE]
            List of SAEs to use for the UMAP calculation
        autointerp_dfs: dict[int, pd.DataFrame]
            Dataframes containing autointerp descriptions for each SAE
        sae_widths: list[int]
            The widths of SAEs we'll be using for the UMAP calculation
        model: HookedSAETransformer
            The model which all the SAEs should be attached to
        n_neighbors_visual: int
            The number of neighbors to consider for the UMAP embedding for the visual plot
        min_dist_visual: float
            The minimum distance between points in the UMAP embedding for the visual plot
        n_neighbors_cluster: int
            The number of neighbors to consider for the UMAP embedding for the cluster plot
        min_dist_cluster: float
            The minimum distance between points in the UMAP embedding for the cluster plot
        min_cluster_size: int
            The minimum number of points in a cluster.
        batch_size: int
            Number of latents to process at once, for logits
    """
    raise NotImplementedError()


# This took about 40s to run for me in Colab Pro+, 80s on my VastAI A100 remote machine
expansion_factors = [1, 4, 16]
umap_df = compute_sae_umap_data(splitting_saes, autointerp_dfs, [768 * ex for ex in expansion_factors], gpt2)
display(umap_df.head())

<details>
<summary>솔루션</summary>

HDBSCAN clustering을 사용하지 않은 버전은 다음과 같습니다:

```python
def compute_sae_umap_data(
    saes: dict[int, SAE],
    autointerp_dfs: dict[int, pd.DataFrame],
    sae_widths: list[int],
    model: HookedSAETransformer,
    n_neighbors_visual: int = 15,
    min_dist_visual: float = 0.05,
    find_clusters: bool = False,
    n_neighbors_cluster: float = 15,
    min_dist_cluster: float = 0.1,
    min_cluster_size: int = 3,
    batch_size: int = 1000,
) -> pd.DataFrame:
    """
    This function will return a dataframe containing umap coordinates & other data (you can then use
    this to create a plot using the code immediately below). The UMAP calculation is done over
    multiple SAEs simultaneously, for comparison.

    Expected dataframe columns:
        sae_width: int
            The width of the SAE that this feature belongs to
        feature_idx: int
            The index of the feature
        umap_x: float
            The x-coordinate of the feature in the UMAP embedding
        umap_y: float
            The y-coordinate of the feature in the UMAP embedding
        autointerp: str
            The autointerp description of this feature
        top_token_strs_formatted: str
            The top 10 tokens that the feature is activated by

    Args:
        saes: dict[int, SAE]
            List of SAEs to use for the UMAP calculation
        autointerp_dfs: dict[int, pd.DataFrame]
            Dataframes containing autointerp descriptions for each SAE
        sae_widths: list[int]
            The widths of SAEs we'll be using for the UMAP calculation
        model: HookedSAETransformer
            The model which all the SAEs should be attached to
        n_neighbors_visual: int
            The number of neighbors to consider for the UMAP embedding for the visual plot
        min_dist_visual: float
            The minimum distance between points in the UMAP embedding for the visual plot
        n_neighbors_cluster: int
            The number of neighbors to consider for the UMAP embedding for the cluster plot
        min_dist_cluster: float
            The minimum distance between points in the UMAP embedding for the cluster plot
        min_cluster_size: int
            The minimum number of points in a cluster.
        batch_size: int
            Number of features to process at once, for logits
    """
    assert not find_clusters, "Not implemented yet"

    # Get initial dataframe by concatenating across SAEs (and autointerp descriptions)
    sae_dfs = []
    for width in sae_widths:
        df = autointerp_dfs[width].copy()
        df["sae_width"] = width
        df["feature_idx"] = list(range(width))
        sae_dfs.append(df)
    feature_df = pd.concat(sae_dfs)

    # Get concatenated decoder matrix
    W_dec = t.cat([saes[width].W_dec for width in sae_widths])

    # Get all the top boosted tokens for each feature, processing in batches
    top_token_ids = []
    print("Computing top logits")
    for start_idx in range(0, len(feature_df), batch_size):
        end_idx = min(start_idx + batch_size, len(feature_df))
        batch_result = W_dec[start_idx:end_idx] @ model.W_U
        top_token_ids.append(batch_result.topk(10).indices)

    # Combine results from all batches, and get them into the dataframe
    token_factors_inds = t.cat(top_token_ids)
    feature_df["tok_token_ids"] = token_factors_inds.tolist()
    feature_df["top_token_strs"] = [
        ", ".join(map(repr, model.to_str_tokens(tokens)))
        for tokens in token_factors_inds
    ]

    print("Calculating 2D UMAP")
    visual_umap = UMAP(
        n_components=2,
        n_neighbors=n_neighbors_visual,
        min_dist=min_dist_visual,
        metric="cosine",
    )
    visual_umap_embedding = visual_umap.fit_transform(W_dec.cpu())

    feature_df[["umap_x", "umap_y"]] = visual_umap_embedding[:, :2]

    return feature_df
```

</details>

데이터를 플롯하려면 아래 코드를 사용하십시오.

In [ ]:
# For the color scale
custom_grey_green_color_scale = lambda n: (
    ["rgba(170,170,170,0.5)"] + px.colors.n_colors("rgb(0,120,0)", "rgb(144,238,144)", n - 1, colortype="rgb")
)

# Make sure the points for wider SAEs are on top
umap_df = umap_df.sort_values("sae_width", ascending=False)

# Get marker size (larger for narrower SAEs)
umap_df["marker_size"] = 4 * umap_df["sae_width"] / umap_df["sae_width"].max()

px.scatter(
    umap_df,
    x="umap_x",
    y="umap_y",
    color=umap_df["sae_width"].astype(str),  # for discrete colors
    size="marker_size",
    height=900,
    width=1200,
    hover_data=["description", "top_token_strs"],
    labels={"umap_x": "UMAP 1", "umap_y": "UMAP 2", "color": "SAE Width"},
    color_discrete_sequence=custom_grey_green_color_scale(len(expansion_factors)),
    template="simple_white",
    title="Feature Splitting in SAEs",
).update_traces(marker=dict(line=dict(width=0))).show()

### 연습 문제 - 클러스터링 추가하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

기하학적 구조를 더 자세히 탐색하기 위해 embedding에 클러스터링 알고리즘을 적용할 수도 있습니다. 이 연습 문제는 서로 다른 SAE 간의 관계보다는, 단일 SAE의 기하학적 구조를 탐색하는 것에 더 중점을 둡니다.

**HDBSCAN**은 밀도에 따라 공간을 변환한 다음, 변환된 공간의 최소 신장 트리(minimum spanning tree)를 기반으로 클러스터를 구축하는 계층적 클러스터링 알고리즘입니다. 이에 대한 더 자세한 내용은 [here](https://hdbscan.readthedocs.io/en/latest/how_hdbscan_works.html)에서 읽어보실 수 있습니다. 이러한 클러스터링 알고리즘을 적용하는 표준적인 방법은 다음과 같습니다:

- 고차원 Umap을 생성합니다 (최소 2차원보다 높게, 예를 들어 10과 같은 숫자).
- 다음 코드를 사용하여 clusterer를 학습시킵니다:

```python
clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
clusterer.fit(clustering_umap_embedding)
```

- `clusterer.labels_`을 통해 label을 추출하고, 이를 dataframe에 넣습니다.

`find_clusters`가 True인 경우 이 로직을 포함하도록 이전 함수를 수정할 수 있습니다.

In [ ]:
# This took about 50s to run for me in Colab Pro+, 90s on my VastAI A100 remote machine
umap_df = compute_sae_umap_data(splitting_saes, autointerp_dfs, [widths[5]], gpt2, find_clusters=True)
display(umap_df.head())

px.scatter(
    umap_df,
    x="umap_x",
    y="umap_y",
    color="cluster",
    height=900,
    width=1200,
    hover_data=["description", "top_token_strs"],
    labels={"umap_x": "UMAP 1", "umap_y": "UMAP 2"},
    template="simple_white",
    title=f"2D UMAP for SAE width = {widths[5]}, clustering algorithm = HDBSCAN from 10D UMAP embedding",
).update_traces(marker=dict(size=4, line=dict(width=0))).update_layout(showlegend=False)

<details>
<summary>정답</summary>

`find_clusters`이 True인 경우, 함수의 끝에 삽입할 코드는 다음과 같습니다:

```python
print("Calculating 10D UMAP")
clustering_umap = UMAP(
    n_components=10,
    n_neighbors=n_neighbors_cluster,
    min_dist=min_dist_cluster,
    metric="cosine",
)
clustering_umap_embedding = clustering_umap.fit_transform(W_dec.cpu())
clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
clusterer.fit(clustering_umap_embedding)

feature_df["cluster"] = clusterer.labels_
feature_df.sort_values("cluster", inplace=True)
feature_df["cluster"] = feature_df["cluster"].astype(str)
```

</details>

## Feature Absorption

> 참고 - 이 섹션은 아직 완성되지 않았습니다. [feature absorption paper](https://www.lesswrong.com/posts/3zBsxeZzd3cvuueMJ/paper-a-is-for-absorption-studying-feature-splitting-and)의 결과를 재현하는 것을 바탕으로 향후 약 한 달 동안 연습 문제가 추가될 예정입니다.

Feature absorption은 매우 흥미로운 주제이며, 이 자료를 작성하는 시점에서 가장 최근에 발표된 연구입니다 (관련 포스트가 [this week](https://www.lesswrong.com/posts/3zBsxeZzd3cvuueMJ/paper-a-is-for-absorption-studying-feature-splitting-and)에 게시되었습니다). 논문을 인용하자면, feature absorption 현상은 다음과 같습니다:

> - 특정 SAE latent가 인간이 해석 가능한 개념(예: "E로 시작함")을 추적하는 것처럼 보입니다.
> - 해당 SAE latent가 겉보기에 임의적인 예시(예: "Elephant")에서는 활성화되지 않습니다.
> - 우리는 해당 feature 방향으로 약하게 투영되며, 메인 latent를 대신하여 인과적으로 매개하는 "absorbing" latent들을 발견합니다 (예: "elephants" latent가 "E로 시작함" feature 방향을 흡수하면, SAE는 더 이상 "Elephant" token에 대해 "E로 시작함" latent를 활성화하지 않습니다. 대신 "elephants" latent가 다른 코끼리 관련 semantic feature들과 함께 해당 정보를 인코딩하게 됩니다).

feature absorption은 위에서 설명한 feature splitting의 그림을 더 복잡하게 만든다는 점에 유의하십시오. feature splitting의 관점에서는, 좁은 SAE의 단일 latent가 더 넓은 SAE에서는 여러 개의 더 구체적인 latent로 분리될 수 있습니다. 이 관점에서는 SAE activation을 예측하는 능력이 여전히 좋으며, 모델에서 sparse circuit을 찾을 수 있을 것으로 기대할 수 있습니다. 하지만 feature absorption은 SAE width가 커지더라도 여전히 지속되는 문제이며, 이는 SAE activation을 예측하는 능력뿐만 아니라 sparse circuit을 찾는 능력까지 저해할 수 있습니다 (SAE가 더 이상 희소한 causal mediator 세트를 가진 분해를 제공하지 않기 때문입니다). 더욱이, feature absorption은 sparsity penalty에서 직접적으로 기인하는 문제로 보이며 ("E로 시작함" feature의 활성화를 줄이면서 "Elephant" feature의 활성화를 더 높이지 않아도 되기 때문입니다), 이로 인해 해결하기가 매우 어렵습니다.

다음은 feature absorption을 설명하는 다이어그램입니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/feature-absorption-last.png" width="600">

이 논문에서 수행된 연구는 주로 feature absorption 문제를 식별하고 특성화하는 것입니다. 이 문제에 대한 일반적인 해결책은 아직 제안되지 않았으며, 이는 이 분야를 특히 흥미로운 연구 영역으로 만듭니다! 하지만, 해결책의 일부는 다음과 같을 수 있습니다...

## Meta-SAEs

> 참고 - 이 섹션은 아직 완성되지 않았습니다. [Meta-SAEs paper](https://www.lesswrong.com/posts/TMAmHh4DdMr4nCSr5/showing-sae-latents-are-not-atomic-using-meta-saes)의 결과를 재현하는 내용을 바탕으로 향후 약 한 달 동안 연습 문제가 추가될 예정입니다.

Meta SAE는 일반적인 SAE의 decoder direction을 재구성하도록 훈련된 특수한 형태의 SAE입니다. 이를 통해 SAE latent가 monosemantic하지 않은 상황에서도 base SAE latent의 sparse한 재구성을 찾을 수 있습니다 (SAE latent가 항상 monosemantic할 것이라고 기대할 수 없는 이유에 대해서는 이전의 feature absorption 섹션을 참조하십시오). Meta-SAE에 관한 [paper](https://www.lesswrong.com/posts/TMAmHh4DdMr4nCSr5/showing-sae-latents-are-not-atomic-using-meta-saes)에서는 다음과 같은 핵심 결과들을 발견했습니다:

> - SAE latent는 더 원자적이고 해석 가능한 meta-latent로 분해될 수 있습니다.
> - 더 큰 SAE의 latent가 더 작은 SAE의 latent로부터 분리되었을 때, 더 큰 SAE로 훈련된 meta SAE가 종종 이러한 구조를 복구한다는 것을 보여줍니다.
> - 특정 지식 편집 작업(knowledge editing task)에서 meta-latent가 SAE latent보다 모델 동작에 대해 더 정밀한 causal intervention을 가능하게 함을 입증합니다.

저자들이 구축한 [dashboard](https://metasae.streamlit.app/?page=Feature+Explorer&feature=11329)를 방문하여 meta-SAE latent를 탐색해 볼 수 있습니다.

## Logit Lens

> 참고 - 이 섹션은 아직 완성되지 않았습니다. LessWrong 포스트 [Understanding SAE Features with the Logit Lens](https://www.lesswrong.com/posts/qykrYY6rXXM7EEs8Q/understanding-sae-features-with-the-logit-lens)의 결과를 재현하는 것을 바탕으로 향후 약 한 달 동안 연습 문제가 추가될 예정이며, universal features 및 token enrichment 분석 주제를 다룰 것입니다.

## Autointerp

> 참고 - 이 섹션으로 넘어가기 전에 1️⃣ Intro to SAE Interpretability에 있는 자동화된 interpretability 관련 내용을 먼저 학습하시기를 강력히 권장합니다.

이 섹션에서는 자동화된 interpretability에 대해 더 깊이 있게 다룹니다. 특히 다음 두 가지 주제를 다룰 예정입니다:

- 자동화된 interpretability를 사용하여 feature 설명의 점수를 매기는 방법 (이를 SAE를 평가하는 방법으로 사용합니다),
- autointerp의 생성 단계에서 계산 속도를 크게 높일 수 있는 잠재적 방법으로 patch scoping을 사용하는 방법.

### autointerp 설명 점수 매기기

이전에 다루었던 autointerp 및 SAE evals에 대해 짧게 논의했을 때의 내용을 복습해 보겠습니다:

> SAE evaluations는 "우리의 SAE가 얼마나 좋은지"를 다양한 방식으로 측정하는 방법입니다. 하지만 우리가 선택하는 어떤 metric이든 **Goodhearting**에 취약하며, 우리가 SAE를 통해 얻고자 하는 바를 반드시 대표하지는 않기 때문에 이는 매우 어려운 작업임이 드러났습니다. 예를 들어, (reconstruction loss가 일정하다면) 더 sparse한 latent가 종종 더 interpretable한 경향이 있으며, 이것이 SAE를 평가하는 일반적인 방법이 **latent sparsity와 reconstruction loss의 Pareto frontier**를 따라 평가하는 이유입니다. 하지만 만약 sparsity가 더 interpretable한 latent로 이어지지 않는다면 어떻게 될까요 (예: [feature absorption](https://www.lesswrong.com/posts/3zBsxeZzd3cvuueMJ/paper-a-is-for-absorption-studying-feature-splitting-and) 때문)? **Autointerp는 SAE interpretability를 평가하는 대안적인 방법을 제공합니다. 왜냐하면 latent 설명이 얼마나 좋은지를 직접적으로 정량화할 수 있기 때문입니다!** 아이디어는 latent 설명을 일부 prompt 테스트 세트에 대한 예측 세트로 변환한 다음, 해당 예측의 정확도를 점수 매기는 것입니다. 더 interpretable한 latent는 더 나은 예측으로 이어져야 합니다. 왜냐하면 latent가 주어진 설명으로부터 예측 가능한 monosemantic하고 인간이 해석 가능한 패턴을 갖는 경향이 있기 때문입니다.

점수를 매길 때, 우리는 **recall** (latent가 activate되는 텍스트를 식별하는 것)과 **precision** (false positive를 피하는 것) 사이의 균형을 맞추고자 합니다. 예를 들어, "don't stop"이나 "won't stop"과 같은 구절에서 "stop"이라는 단어에 반응하는 latent가 있다고 가정해 보겠습니다. 다음과 같은 두 가지 유형의 잘못된 설명이 있을 수 있습니다:

| 설명 | Recall | Precision | 해설 |
|-----------------------------------------------------------------------------|--------|-----------|-------------|
| *'The latent activates on the word "stop"'* | 높음 | 낮음 | 모든 positive example을 식별하지만, false positive도 많습니다. |
| *'The latent activates on the word "stop" in the phrase "don't stop"'* | 낮음 | 높음 | positive example의 절반만 식별하지만, false positive는 없습니다. |

이러한 문제들은 해결하기 꽤 어려울 수 있습니다. 예를 들어, 낮은 precision 문제를 해결하려면 "don't stop"이나 "won't stop" 이외의 문맥에서 "stop"이라는 단어가 포함된 예시 시퀀스를 제시하여, 여기서의 activation이 0임을 모델에게 보여주어야 합니다. 불행히도, 이러한 false positive 예시를 생성하는 방법은 명확하지 않습니다. 왜냐하면 autointerp에서 사용하는 시퀀스들은 보통 latent의 top activating sequences에서 샘플링되거나(즉, "don't stop"이나 "won't stop" 구절 내의 "stop"만 포함하게 됨), 전체 데이터셋에서 무작위로 추출되기 때문입니다(이 경우 "stop"이라는 단어가 특히 높은 빈도로 포함될 것이라고 기대하기 어렵습니다). 이러한 문제들은 더 큰 SAE와 더 sparse하고 구체적인 latent로 확장함에 따라 더욱 악화될 수 있습니다.

이전 섹션에서 다음 모델과 SAE를 아직 로드하지 않았다면, 아래 코드 블록을 실행하여 로드하시기 바랍니다:

In [ ]:
gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)

gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_sae,
    dataset="NeelNanda/pile-10k",
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=str(device),
)

### 연습 문제 - autointerp scoring 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 30-45 minutes on this exercise.
> ```

문헌에서는 여러 가지 서로 다른 scoring 방법들이 탐구되어 왔습니다. 예를 들어, [simulation scoring](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html#sec-algorithm-explain) (OpenAI의 원본 "Language models can explain neurons in language models" 논문에서 제안된 방법)은 텍스트의 각 token에 activation을 할당한 다음, 예측된 activation과 실제 activation 사이의 상관관계를 측정합니다. 하지만 여기서는 구현하기 조금 더 쉬운 방법인 **detection**을 사용할 것입니다. 즉, 모델에게 무작위 시퀀스 샘플(일부는 top activating sequences에서 추출하고, 나머지는 데이터셋에서 무작위로 추출)을 제공하고, 각 시퀀스가 해당 latent를 포함하고 있는지 분류하도록 요청하는 방식입니다. 그런 다음 예측의 정확도를 측정할 수 있습니다. 예를 들어, 5개의 무작위 시퀀스와 3개의 top activating sequences( #1, #3, #7이 top activating sequences가 되도록 정렬됨)를 선택하고, 모델이 #1, #3, #6으로 예측했다면, #6과 #7을 제외한 모든 시퀀스가 올바르게 분류되었으므로 score는 6/8 = 75%가 됩니다.

autointerp scoring을 시작하는 데 도움이 되도록 아래에 몇 가지 인프라를 설정해 두었습니다. 우선, forward pass를 수행하여 얻은 반환 데이터를 더 쉽게 정리할 수 있도록 도와주는 `Example` 클래스가 있습니다. 이 클래스는 `act_threshold`으로 초기화된다는 점에 유의하십시오. 이는 token이 "active"하다고 간주할 activation 기준값입니다. 특정 시퀀스에서 단순히 max-activation token만 active한 것이 아닌 경우가 많기 때문에 이 설정은 중요합니다.

In [ ]:
class Example:
    """
    Data for a single example sequence.
    """

    def __init__(self, toks: list[int], acts: list[float], act_threshold: float, model: HookedSAETransformer):
        self.toks = toks
        self.str_toks = model.to_str_tokens(t.tensor(self.toks))
        self.acts = acts
        self.act_threshold = act_threshold
        self.toks_are_active = [act > act_threshold for act in self.acts]
        self.is_active = any(self.toks_are_active)  # this is what we predict in the scoring phase

    def to_str(self, mark_toks: bool = False) -> str:
        return (
            "".join(
                f"<<{tok}>>" if (mark_toks and is_active) else tok
                for tok, is_active in zip(self.str_toks, self.toks_are_active)
            )
            .replace(">><<", "")
            .replace("�", "")
            .replace("\n", "↵")
        )


ex = Example(
    toks=[1212, 1276, 307, 3635, 13, 314, 1239, 714, 651, 262, 8181, 286, 48971, 12545, 13],
    acts=[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    act_threshold=0.5,
    model=gpt2,
)

print(ex.str_toks)

print(ex.to_str(mark_toks=True))

다음으로, autointerp 생성 및 scoring에 설정해야 할 모든 파라미터를 포함하는 `AutoInterpConfig` 클래스를 제공해 드렸습니다. 여기의 docstring을 살펴보고 각 파라미터가 어떤 역할을 하는지 반드시 이해하시기 바랍니다. 주의 깊게 보셔야 할 몇 가지 사항은 다음과 같습니다:

- 이 autointerp 구현에서는 한 번에 하나씩이 아니라, 여러 latent에 대해 동시에 병렬화하여 처리할 수 있습니다.
- 데이터 수집 과정에는 입력 데이터에서 `n_top_ex_for_generation + n_top_ex_for_scoring` 개의 top activation을 수집하는 과정(이를 위해 `n_top_ex` property를 제공했습니다)과 `n_random_ex_for_scoring` 개의 random sequence를 수집하는 과정이 포함됩니다. 생성 단계에서는 무작위로 선택된 `n_top_ex_for_generation` 개의 top example을 사용하여 설명을 생성하며, scoring 단계에서는 `n_top_ex_for_scoring` 개의 top example과 `n_random_ex_for_scoring` 개의 random example을 함께 섞은 뒤, 모델에게 각각 어느 것인지 분류하도록 요청합니다.

In [ ]:
@dataclass
class AutoInterpConfig:
    """
    Controls all parameters for how autointerp will work.

    Arguments:
        latents:                    The latent indices we'll be studying
        buffer:                     The size of the buffer to use for scoring
        no_overlap:                 Whether to allow overlapping sequences for scoring
        act_threshold_frac:         The fraction of the maximum act to use as the act threshold
        total_tokens:               The total number of tokens we'll gather data for.
        scoring:                    Whether to perform the scoring phase, or just return explanation
        max_tokens_in_explanation:  The maximum number of tokens to allow in an explanation
        n_top_ex_for_generation:    The number of top activating sequences to use for generation
        n_top_ex_for_scoring:       The number of top sequences to use for scoring
        n_random_ex_for_scoring:    The number of random sequences to use for scoring
    """

    latents: list[int]
    buffer: int = 10
    no_overlap: bool = False
    act_threshold_frac: float = 0.1
    total_tokens: int = 500_000
    scoring: bool = False
    max_tokens_in_explanation: int = 25
    use_examples_in_explanation_prompt: bool = True
    n_top_ex_for_generation: int = 10
    n_top_ex_for_scoring: int = 4
    n_random_ex_for_scoring: int = 8

    @property
    def n_top_ex(self):
        """When fetching data, we get the top examples for generation & scoring simultaneously."""
        return self.n_top_ex_for_generation + self.n_top_ex_for_scoring

    @property
    def max_tokens_in_prediction(self) -> int:
        """Predictions take the form of comma-separated numbers, which should all be single toks."""
        return 2 * self.n_ex_for_scoring + 5

    @property
    def n_ex_for_scoring(self) -> int:
        """For scoring phase, we use a randomly shuffled mix of top-k activations and random seqs."""
        return self.n_top_ex_for_scoring + self.n_random_ex_for_scoring

    @property
    def n_latents(self) -> int:
        return len(self.latents)

마지막으로, `AutoInterp` 클래스가 있습니다. 이 클래스는 config로 초기화되며, 다음과 같은 중요한 메서드들을 포함합니다:

- `gather_data`: generation 단계에서 사용할 데이터를 수집합니다. 모든 batch를 처리하며 top k를 계산하고, 이를 길이 `n_top_ex_for_generation * n_batches`의 단일 tensor로 연결한 다음, 여기서 top `n_top_ex_for_generation`을 추출하는 방식으로 작동합니다.
- `get_generation_prompts`: `gather_data`의 데이터를 사용하여 generation 단계의 prompt를 반환합니다.
- `get_response`: OpenAI에 일반적인 API 호출을 수행하고 응답을 반환합니다.
- `run`: 전체 autointerp 파이프라인을 실행합니다 (현재는 generation만 구현되어 있습니다).

먼저 아래의 코드 블록을 실행하고, generation 단계가 어떻게 작동하는지 이해해야 합니다. 이해가 되었다면 scoring 단계를 구현해 보시기 바랍니다. 이를 위해 다음 메서드들을 채우거나 추가해야 합니다:

#### `get_scoring_prompts`

이 메서드는 scoring 단계에서 사용될 prompt를 반환해야 합니다. `get_generation_prompts` 메서드와 유사한 구조를 따를 수 있지만, 설명 대신 특정 예시에 대한 예측을 요청해야 합니다.

몇 가지 가이드라인입니다:

- 예시의 `to_str` 메서드에서 `mark_toks=True`을 사용하지 않도록 주의하십시오. 모델에게 어떤 sequence가 active한지 알려주고 싶지 않기 때문입니다!
- 모델이 예측값을 쉽게 파싱할 수 있도록 반환 형식을 정확하게 지정해야 합니다. 예를 들어, `1, 4, 7`와 같이 쉼표로 구분된 숫자 리스트를 요청할 수 있습니다 (또한 활성화되는 sequence가 없다고 생각되면 `None`이라고 답하도록 지정하십시오).
- 모델의 응답을 받아 다시 정수 리스트로 파싱하는 `parse_predictions` 메서드를 작성하는 것도 좋습니다.

<details>
<summary>scoring 단계 prompt를 위한 권장 구조입니다 (하지만 이를 읽기 전에 직접 시도해 보시는 것을 추천합니다!)</summary>

assistant prompt 없이 system & user prompt만으로도 충분할 것입니다.

```python
{
    "system": f"""We're studying neurons in a neural network. Each neuron activates on some particular word or concept in a short document. You will be given a short explanation of what this neuron activates for, and then be shown {n_ex_for_scoring} example sequences. You will have to return the examples where you think the neuron should activate at least once, in the form of a comma-separated list. For example, your response might look like "1, 4, 7". If you think there are no examples where the neuron should activate, you should just respond with "None". You should include nothing else in your response other than comma-separated numbers or the word "None" - this is important.""",

    "user": f"Here is the explanation: this neuron fires on {explanation}.\n\nHere are the examples:\n\n{examples_as_str}",
}
```

</details>

#### `gather_data`

scoring 단계에 필요한 데이터를 수집하도록 이 메서드를 다시 작성해야 합니다. 단순히 generation 단계를 위한 `n_top_ex_for_generation` 예시만 반환하는 것이 아니라, scoring 단계를 위한 `n_top_ex_for_scoring` top 예시(`n_random_ex_for_scoring` random 예시와 함께 섞임)도 함께 반환합니다. 이 random 예시들은 이 함수에서 반복하는 데이터셋의 모든 batch에서 선택하고, 각 latent마다 서로 다르게 선택하는 것을 권장합니다. (참고 - 실제로는 무작위로 선택된 example sequence의 activation을 반환하여 이것이 activating으로 분류되어야 하는지 확인하고 싶을 수 있습니다. 하지만 SAE가 충분히 sparse하다면 무작위로 선택된 sequence가 활성화될 확률은 매우 낮으므로, 이 실습에서는 무작위로 선택된 sequence는 활성화되지 않는다고 가정해도 무방합니다.)

#### `run`

`run` 메서드에 다음 코드를 추가해야 합니다: (1) `n_top_ex_for_scoring` top 예시와 `n_random_ex_for_scoring` random sequence로부터 scoring prompt를 가져오고, (2) 해당 prompt로부터 예측값을 반환 및 파싱하며, (3) 예측값을 score로 계산합니다 (여기서 score는 전체 `n_top_ex_for_scoring + n_random_ex_for_scoring` 중 정답 분류의 비율로 정의됩니다).

마지막 팁들입니다:

- `run` 메서드에 `debug` 플래그를 제공했습니다. 이 값이 `True`로 설정되면, generation 및 scoring 단계에서 prompt와 raw response에 대한 유용한 정보가 출력됩니다. 코드가 성공적으로 실행되지만 score가 낮게 나오는 경우(명확하게 interpretable해 보이는 latent에 대해 scoring 단계에서 75% 미만이라면 상당히 좋지 않은 결과입니다) 이를 사용하여 디버깅하십시오.
- `display_dashboard` 함수를 사용하여 선택한 latent가 interpretable하며 좋은 score를 얻어야 하는 것이 맞는지 sanity check를 할 수 있습니다.
- 디버깅 피드백 루프의 속도를 높이려면 `self.gather_data()` 메서드를 수동으로 실행하고, `run`를 수정하여 이 메서드의 출력을 선택적으로 인자로 받을 수 있게 하십시오. 그렇게 하면 run 함수를 수정하고 다시 호출할 때, 데이터 수집을 다시 기다리지 않고 바로 API 쿼리 섹션으로 넘어갈 수 있습니다.
- off-by-one 인덱싱 오류를 피하십시오. 예를 들어, 예시에는 0부터 n-1까지 레이블을 붙였는데 분류 결과는 1부터 n까지 저장하는 경우입니다. (제가 이 함수를 디버깅하는 데 30분을 쓰고 나서야 이 실수를 깨달았기 때문에 드리는 말씀이 절대 아닙니다. 정말 아닙니다.)

In [ ]:
Messages: TypeAlias = list[dict[Literal["role", "content"], str]]


def display_messages(messages: Messages):
    print(tabulate([m.values() for m in messages], tablefmt="simple_grid", maxcolwidths=[None, 120]))


class AutoInterp:
    """
    This is a start-to-end class for generating explanations and optionally scores. It's easiest to
    implement it as a single class for the time being because there's data we'll need to fetch
    that'll be used in both the generation and scoring phases.
    """

    def __init__(
        self,
        cfg: AutoInterpConfig,
        model: HookedSAETransformer,
        sae: SAE,
        act_store: ActivationsStore,
        api_key: str,
    ):
        self.cfg = cfg
        self.model = model
        self.sae = sae
        self.act_store = act_store
        self.api_key = api_key

    def run(self, debug: bool = False) -> dict[int, dict[str, Any]]:
        """Runs both generation & scoring phases, and returns the results in a dictionary."""
        generation_examples, scoring_examples = self.gather_data()
        results = {}

        for latent in tqdm(self.cfg.latents, desc="Querying OpenAI api"):
            gen_prompts = self.get_generation_prompts(generation_examples[latent])
            explanation_raw = self.get_response(
                gen_prompts,
                max_tokens=self.cfg.max_tokens_in_explanation,
                debug=debug and (latent == self.cfg.latents[0]),
            )[0]
            explanation = self.parse_explanation(explanation_raw)
            results[latent] = {"explanation": explanation}

            if self.cfg.scoring:
                raise NotImplementedError()

        return results

    def parse_explanation(self, explanation: str) -> str:
        return explanation.split("activates on")[-1].rstrip(".").strip()


    def get_response(self, messages: list[dict], max_tokens: int, n_completions: int = 1, debug: bool = False) -> str:
        """Generic API usage function for OpenAI"""
        for message in messages:
            assert message.keys() == {"content", "role"}
            assert message["role"] in ["system", "user", "assistant"]

        client = OpenAI(api_key=self.api_key)

        result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            n=n_completions,
            max_tokens=max_tokens,
            stream=False,
        )
        if debug:
            display_messages(messages + [{"role": "assistant", "content": result.choices[0].message.content}])

        return [choice.message.content.strip() for choice in result.choices]

    def get_generation_prompts(self, generation_examples: list[Example]) -> Messages:
        assert len(generation_examples) > 0, "No generation examples found"

        examples_as_str = "\n".join(
            [f"{i + 1}. {ex.to_str(mark_toks=True)}" for i, ex in enumerate(generation_examples)]
        )

        SYSTEM_PROMPT = """We're studying neurons in a neural network. Each neuron activates on some particular word/words or concept in a short document. The activating words in each document are indicated with << ... >>. Look at the parts of the document the neuron activates for and summarize in a single sentence what the neuron is activating on. Try to be specific in your explanations, although don't be so specific that you exclude some of the examples from matching your explanation. Pay attention to things like the capitalization and punctuation of the activating words or concepts, if that seems relevant. Keep the explanation as short and simple as possible, limited to 20 words or less. Omit punctuation and formatting. You should avoid giving long lists of words."""
        if self.cfg.use_examples_in_explanation_prompt:
            SYSTEM_PROMPT += """ Some examples: "This neuron activates on the word 'knows' in rhetorical questions like 'Who knows ... ?'", and "This neuron activates on verbs related to decision-making and preferences", and "This neuron activates on the substring 'Ent' at the start of words like 'Entrepreneur' or 'Entire'."""
        else:
            SYSTEM_PROMPT += """Your response should be in the form "This neuron activates on..."."""
        USER_PROMPT = f"""The activating documents are given below:\n\n{examples_as_str}"""

        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ]

    def get_scoring_prompts(self, explanation: str, scoring_examples: list[Example]) -> Messages:
        assert len(scoring_examples) > 0, "No scoring examples found"

        raise NotImplementedError()

    def gather_data(self) -> tuple[dict[int, list[Example]], dict[int, list[Example]]]:
        """
        Stores top acts / random seqs data, which is used for generation & scoring respectively.
        """
        sae_acts_post_hook_name = f"{self.sae.cfg.metadata.hook_name}.hook_sae_acts_post"
        total_batches = self.cfg.total_tokens // (self.act_store.store_batch_size * self.act_store.context_size)

        # Dictionary to store data for each latent
        latent_data = {
            latent: {
                "top_toks": t.empty(0, 1 + 2 * self.cfg.buffer, dtype=t.int64, device=device),
                "top_values": t.empty(0, dtype=t.float32, device=device),
            }
            for latent in self.cfg.latents
        }

        for batch in tqdm(range(total_batches), desc="Collecting activations data"):
            _, cache = self.model.run_with_cache_with_saes(
                tokens := self.act_store.get_batch_tokens().to(device),
                saes=[self.sae],
                stop_at_layer=_get_hook_layer(self.sae) + 1,
                names_filter=[sae_acts_post_hook_name],
            )
            acts = cache[sae_acts_post_hook_name][..., self.cfg.latents]
            del cache

            for i, latent in enumerate(self.cfg.latents):
                # Get top activations from this batch, and filter down to the data we'll actually include
                top_indices = get_k_largest_indices(
                    acts[..., i],
                    k=self.cfg.n_top_ex_for_generation,
                    buffer=self.cfg.buffer,
                    no_overlap=self.cfg.no_overlap,
                )
                top_toks = index_with_buffer(tokens, top_indices, buffer=self.cfg.buffer)
                top_values = index_with_buffer(acts[..., i], top_indices, buffer=self.cfg.buffer)
                latent_data[latent]["top_toks"] = t.cat((latent_data[latent]["top_toks"], top_toks), dim=0)
                latent_data[latent]["top_values"] = t.cat((latent_data[latent]["top_values"], top_values), dim=0)


        # Dicts to store all generation & scoring examples for each latent
        generation_examples = {}
        scoring_examples = {}

        for i, latent in enumerate(self.cfg.latents):
            top_toks = latent_data[latent]["top_toks"]
            top_values = latent_data[latent]["top_values"]
            topk = top_values[:, self.cfg.buffer].topk(self.cfg.n_top_ex_for_generation).indices
            act_threshold = self.cfg.act_threshold_frac * latent_data[latent]["max_act"]
            generation_examples[latent] = [
                Example(
                    toks=top_toks[topk[j]].tolist(),
                    acts=top_values[topk[j]].tolist(),
                    act_threshold=act_threshold,
                    model=self.model,
                )
                for j in range(len(topk))
            ]

        return generation_examples, scoring_examples

<details><summary>솔루션</summary>

```python
Messages: TypeAlias = list[dict[Literal["role", "content"], str]]


def display_messages(messages: Messages):
    print(tabulate([m.values() for m in messages], tablefmt="simple_grid", maxcolwidths=[None, 120]))


class AutoInterp:
    """
    This is a start-to-end class for generating explanations and optionally scores. It's easiest to
    implement it as a single class for the time being because there's data we'll need to fetch
    that'll be used in both the generation and scoring phases.
    """

    def __init__(
        self,
        cfg: AutoInterpConfig,
        model: HookedSAETransformer,
        sae: SAE,
        act_store: ActivationsStore,
        api_key: str,
    ):
        self.cfg = cfg
        self.model = model
        self.sae = sae
        self.act_store = act_store
        self.api_key = api_key

    def run(self, debug: bool = False) -> dict[int, dict[str, Any]]:
        """Runs both generation & scoring phases, and returns the results in a dictionary."""
        generation_examples, scoring_examples = self.gather_data()
        results = {}

        for latent in tqdm(self.cfg.latents, desc="Querying OpenAI api"):
            gen_prompts = self.get_generation_prompts(generation_examples[latent])
            explanation_raw = self.get_response(
                gen_prompts,
                max_tokens=self.cfg.max_tokens_in_explanation,
                debug=debug and (latent == self.cfg.latents[0]),
            )[0]
            explanation = self.parse_explanation(explanation_raw)
            results[latent] = {"explanation": explanation}

            if self.cfg.scoring:
                scoring_prompts = self.get_scoring_prompts(explanation, scoring_examples[latent])
                predictions = self.get_response(
                    scoring_prompts,
                    max_tokens=self.cfg.max_tokens_in_prediction,
                    debug=debug and (latent == self.cfg.latents[0]),
                )[0]
                predictions_parsed = self.parse_predictions(predictions)
                score = self.score_predictions(predictions_parsed, scoring_examples[latent])
                results[latent] |= {
                    "predictions": predictions_parsed,
                    "correct seqs": [i for i, ex in enumerate(scoring_examples[latent], start=1) if ex.is_active],
                    "score": score,
                }

        return results

    def parse_explanation(self, explanation: str) -> str:
        return explanation.split("activates on")[-1].rstrip(".").strip()

    def parse_predictions(self, predictions: str) -> list[int]:
        predictions_split = predictions.strip().rstrip(".").replace("and", ",").split(",")
        predictions_list = [i.strip() for i in predictions_split if i.strip() != ""]
        if predictions_list == ["None"]:
            return []
        assert all(pred.strip().isdigit() for pred in predictions_list), (
            f"Prediction parsing error: predictions should be comma-separated numbers, found {predictions!r}"
        )
        predictions = [int(pred.strip()) for pred in predictions_list]
        return predictions

    def score_predictions(self, predictions: list[str], scoring_examples: list[Example]) -> float:
        classifications = [i in predictions for i in range(1, len(scoring_examples) + 1)]
        correct_classifications = [ex.is_active for ex in scoring_examples]
        return sum([c == cc for c, cc in zip(classifications, correct_classifications)]) / len(classifications)


    def get_response(self, messages: list[dict], max_tokens: int, n_completions: int = 1, debug: bool = False) -> str:
        """Generic API usage function for OpenAI"""
        for message in messages:
            assert message.keys() == {"content", "role"}
            assert message["role"] in ["system", "user", "assistant"]

        client = OpenAI(api_key=self.api_key)

        result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            n=n_completions,
            max_tokens=max_tokens,
            stream=False,
        )
        if debug:
            display_messages(messages + [{"role": "assistant", "content": result.choices[0].message.content}])

        return [choice.message.content.strip() for choice in result.choices]

    def get_generation_prompts(self, generation_examples: list[Example]) -> Messages:
        assert len(generation_examples) > 0, "No generation examples found"

        examples_as_str = "\n".join(
            [f"{i + 1}. {ex.to_str(mark_toks=True)}" for i, ex in enumerate(generation_examples)]
        )

        SYSTEM_PROMPT = """We're studying neurons in a neural network. Each neuron activates on some particular word/words or concept in a short document. The activating words in each document are indicated with << ... >>. Look at the parts of the document the neuron activates for and summarize in a single sentence what the neuron is activating on. Try to be specific in your explanations, although don't be so specific that you exclude some of the examples from matching your explanation. Pay attention to things like the capitalization and punctuation of the activating words or concepts, if that seems relevant. Keep the explanation as short and simple as possible, limited to 20 words or less. Omit punctuation and formatting. You should avoid giving long lists of words."""
        if self.cfg.use_examples_in_explanation_prompt:
            SYSTEM_PROMPT += """ Some examples: "This neuron activates on the word 'knows' in rhetorical questions like 'Who knows ... ?'", and "This neuron activates on verbs related to decision-making and preferences", and "This neuron activates on the substring 'Ent' at the start of words like 'Entrepreneur' or 'Entire'."""
        else:
            SYSTEM_PROMPT += """Your response should be in the form "This neuron activates on..."."""
        USER_PROMPT = f"""The activating documents are given below:\n\n{examples_as_str}"""

        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ]

    def get_scoring_prompts(self, explanation: str, scoring_examples: list[Example]) -> Messages:
        assert len(scoring_examples) > 0, "No scoring examples found"

        examples_as_str = "\n".join([f"{i + 1}. {ex.to_str(mark_toks=False)}" for i, ex in enumerate(scoring_examples)])

        SYSTEM_PROMPT = f"""We're studying neurons in a neural network. Each neuron activates on some particular word/words or concept in a short document. You will be given a short explanation of what this neuron activates for, and then be shown {self.cfg.n_ex_for_scoring} example sequences. You will have to return a comma-separated list of the examples where you think the neuron should activate at least once. For example, your response might look like "1, 4, 7, 8". If you think there are no examples where the neuron will activate, you should just respond with "None". You should include nothing else in your response other than comma-separated numbers or the word "None" - this is important."""
        USER_PROMPT = f"Here is the explanation: this neuron fires on {explanation}.\n\nHere are the examples:\n\n{examples_as_str}"

        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ]

    def gather_data(self) -> tuple[dict[int, list[Example]], dict[int, list[Example]]]:
        """
        Stores top acts / random seqs data, which is used for generation & scoring respectively.
        """
        sae_acts_post_hook_name = f"{self.sae.cfg.metadata.hook_name}.hook_sae_acts_post"
        batch_size, seq_len = self.act_store.store_batch_size_prompts, self.act_store.context_size
        total_seqs = self.cfg.total_tokens // seq_len
        total_batches = total_seqs // batch_size

        # Get indices we'll take our random examples from, over all batches (and over all latents)
        all_rand_indices_shape = (self.cfg.n_random_ex_for_scoring, self.cfg.n_latents)
        all_rand_indices = t.stack(
            [
                t.randint(0, total_batches, all_rand_indices_shape),  # which batch
                t.randint(0, batch_size, all_rand_indices_shape),  # which sequence in the batch
                t.randint(self.cfg.buffer, seq_len - self.cfg.buffer, all_rand_indices_shape),  # where in the sequence
            ],
            dim=-1,
        )  # shape [n_random_ex_for_scoring, n_latents, 3]

        # Dictionary to store data for each latent
        latent_data = {
            latent: {
                "rand_toks": t.empty(0, 1 + 2 * self.cfg.buffer, dtype=t.int64, device=device),
                "top_toks": t.empty(0, 1 + 2 * self.cfg.buffer, dtype=t.int64, device=device),
                "top_values": t.empty(0, dtype=t.float32, device=device),
            }
            for latent in self.cfg.latents
        }

        for batch in tqdm(range(total_batches), desc="Collecting activations data"):
            _, cache = self.model.run_with_cache_with_saes(
                tokens := self.act_store.get_batch_tokens().to(device),
                saes=[self.sae],
                stop_at_layer=_get_hook_layer(self.sae) + 1,
                names_filter=[sae_acts_post_hook_name],
            )
            acts = cache[sae_acts_post_hook_name][..., self.cfg.latents]
            del cache

            for i, latent in enumerate(self.cfg.latents):
                # Get top activations from this batch, and filter down to the data we'll actually include
                top_indices = get_k_largest_indices(
                    acts[..., i],
                    k=self.cfg.n_top_ex,
                    buffer=self.cfg.buffer,
                    no_overlap=self.cfg.no_overlap,
                )
                top_toks = index_with_buffer(tokens, top_indices, buffer=self.cfg.buffer)
                top_values = index_with_buffer(acts[..., i], top_indices, buffer=self.cfg.buffer)
                latent_data[latent]["top_toks"] = t.cat((latent_data[latent]["top_toks"], top_toks), dim=0)
                latent_data[latent]["top_values"] = t.cat((latent_data[latent]["top_values"], top_values), dim=0)

                # Get random activations (our `all_rand_indices` tensor tells us which random sequences to take)
                rand_indices = all_rand_indices[all_rand_indices[:, i, 0] == batch, i, 1:]
                random_toks = index_with_buffer(tokens, rand_indices, self.cfg.buffer)
                latent_data[latent]["rand_toks"] = t.cat((latent_data[latent]["rand_toks"], random_toks), dim=0)

        # Dicts to store all generation & scoring examples for each latent
        generation_examples = {}
        scoring_examples = {}

        for i, latent in enumerate(self.cfg.latents):
            top_toks = latent_data[latent]["top_toks"]
            top_values = latent_data[latent]["top_values"]
            # From our tensor of `n_top_examples * n_batches` top examples, get only the top
            # `n_top_examples` of them
            topk = top_values[:, self.cfg.buffer].topk(self.cfg.n_top_ex).indices
            act_threshold = self.cfg.act_threshold_frac * top_values.max().item()
            rand_split_indices = t.randperm(self.cfg.n_top_ex)

            # generation_examples[latent] = random sample of some of the top activating sequences
            generation_examples[latent] = [
                Example(
                    toks=top_toks[topk[j]].tolist(),
                    acts=top_values[topk[j]].tolist(),
                    act_threshold=act_threshold,
                    model=self.model,
                )
                for j in sorted(rand_split_indices[: self.cfg.n_top_ex_for_generation])
            ]

            # scoring_examples[latent] = random mix of the sampled top activating sequences & random
            # examples (with the top activating sequences chosen to have zero overlap with those
            # used in generation_examples)
            scoring_examples[latent] = random.sample(
                [
                    Example(
                        toks=top_toks[topk[j]].tolist(),
                        acts=top_values[topk[j]].tolist(),
                        act_threshold=act_threshold,
                        model=self.model,
                    )
                    for j in rand_split_indices[self.cfg.n_top_ex_for_generation :]
                ]
                + [
                    Example(
                        toks=random_toks.tolist(),
                        acts=[0.0 for _ in random_toks],
                        act_threshold=act_threshold,
                        model=self.model,
                    )
                    for random_toks in latent_data[latent]["rand_toks"]
                ],
                k=self.cfg.n_ex_for_scoring,
            )

        return generation_examples, scoring_examples
```
</details>

다음은 생성 단계만 실행하는 예시이며, 별도의 설정 없이 바로 작동할 것입니다:

In [ ]:
latents = [9, 11, 15, 16873]

API_KEY = os.environ.get("OPENAI_API_KEY", None)
assert API_KEY is not None, "Please set your own OpenAI key."

autointerp = AutoInterp(
    cfg=AutoInterpConfig(latents=latents, scoring=False),
    model=gpt2,
    sae=gpt2_sae,
    act_store=gpt2_act_store,
    api_key=API_KEY,
)

results = autointerp.run(debug=False)

print(
    tabulate(
        [[latent, *results[latent].values()] for latent in latents],
        headers=["Feature"] + list(results[latents[0]].keys()),
        tablefmt="simple_outline",
    )
)

그리고 여기에는 scoring 단계를 실행하는 코드가 있으며, 연습 문제를 해결하고 나면 정상적으로 작동할 것입니다:

In [ ]:
latents = [9, 11, 15, 16873]

autointerp = AutoInterp(
    cfg=AutoInterpConfig(latents=latents, scoring=True),
    model=gpt2,
    sae=gpt2_sae,
    act_store=gpt2_act_store,
    api_key=API_KEY,
)

results = autointerp.run(debug=False)

print(
    tabulate(
        [[latent, *results[latent].values()] for latent in latents],
        headers=["Feature"] + list(results[latents[0]].keys()),
        tablefmt="simple_outline",
        floatfmt=".2f",
    )
)

### 보너스 - autointerp 개선하기

이것은 autointerp의 매우 기본적인 구현이며, 여러 가지 방법으로 개선될 수 있습니다. 아래에 몇 가지 아이디어를 나열하였으며, 이 방법들이나 여러분이 생각하신 다른 아이디어들을 자유롭게 시도해 보시기 바랍니다!

- **asyncio를 사용하여 API 쿼리 속도 높이기.** 데이터 수집은 latent들에 대해 병렬화되어 있지만(대부분의 계산 시간은 forward pass에 소요되며, 최대 activation을 얻기 위한 latent for 루프에 소요되지 않습니다), OpenAI API 쿼리의 경우는 그렇지 않습니다. 현재 이는 for 루프로 구현되어 있어 매우 비효율적입니다. OpenAI API를 쿼리하는 작업은 I/O bottleneck이 발생하므로, `asyncio`와 같은 라이브러리를 사용하여 속도를 높일 수 있습니다. 이를 구현해 보시겠습니까? (참고로, 이 특정 확장 기능은 Colab이나 Jupyter notebook 환경에서는 적합하지 않을 수 있습니다.)
- **생성을 위해 explanation 사용하기.** generation 및 scoring과 더불어, generation 단계에서 생성된 explanation을 입력으로 받아, 모델이 해당 latent를 활성화할 가능성이 높다고 생각하는 시퀀스를 생성하는 세 번째 파이프라인을 구축할 수 있습니까 (EleutherAI [here](https://blog.eleuther.ai/autointerp/#:~:text=Generation)에서 설명한 방식입니다). 이 경우 모델이 false positive를 생성합니까? 만약 그렇다면, 이러한 예시들을 generation 단계에서 사용하여 explanation의 precision을 높이고, false positive 분류 횟수를 줄일 수 있습니까?
- **latent 이웃을 사용하여 precision 측정하기.** 다시 EleutherAI [here](https://blog.eleuther.ai/autointerp/#:~:text=Neighbors)에서 설명한 것처럼, 서로 decoder cosine similarity가 높은 latent들을 찾아, 해당 explanation이 특정 시퀀스가 어느 latent를 활성화하는지 구분하기에 충분한지 확인하여 precision을 측정할 수 있습니까? 다른 관점에서, 이웃한 latent들에서 가장 높게 활성화된 시퀀스들을 가져와 이를 generation 단계의 false positive로 사용하여 explanation의 precision을 개선할 수 있습니까?
- **neuron / random latent와 autointerp 벤치마크 비교하기.** Anthropic [found](https://transformer-circuits.pub/2023/monosemantic-features/index.html#appendix-automated-setup)에 따르면, SAE latent에 대한 autointerp가 transformer neuron이나 무작위로 초기화된 latent에 대한 autointerp보다 일관되게 우수한 성능을 보였습니다 (이는 우리의 SAE latent가 interpretable하고 monosemantic하다는 좋은 신호입니다!). 하지만 그들은 무작위 SAE latent에 대한 autointerp가 예상보다 더 좋은 성능을 보인다는 것을 발견했는데, 이는 매우 큰 데이터셋에서 top-k를 추출할 때 무작위 latent조차 특정 패턴을 보이기 때문입니다 (예: 항상 동일한 token에서 firing 하는 경우 등). 이 결과들을 재현할 수 있습니까? 이러한 결과가 autointerp와 SAE 전체에 대해 무엇을 시사한다고 생각하십니까?
- **더 많은 정보 통합하기.** EleutherAI가 제안한 방식으로 generation prompt를 추가할 수 있습니까? 예를 들어 (1) 이 latent에 의해 boost된 top logit들을 추가하거나, (2) 각 token에 대한 quantized activation 값을 제공하거나, (3) 마주칠 수 있는 다양한 종류의 latent(예: token-level, substring-level, 또는 높은 consistent activation heuristic을 가진 concept-level)를 보여주는 예시 explanation을 prompt에 포함하거나, (4) chain of thought를 사용하여 모델 explanation의 품질을 높이는 방법 등이 있습니다. 예를 들면 다음과 같습니다:

```plaintext
Step 1: List a couple activating and contextual tokens you find interesting. Search for patterns in these tokens, if there are any. Don't list more than 5 tokens. 
Step 2: Write down general shared latents of the text examples.
Step 3: List the tokens that the neuron boosts in the next token prediction.
Step 4: Write an explanation.
```

## Patch scoping

이제 **patch scoping**으로 넘어가겠습니다. 이는 autointerp를 위한 흥미로운 새로운 방법으로, 매우 많은 수의 SAE latent에 대해 autointerp를 실행할 때 발생하는 계산 비용을 크게 줄여줄 것입니다. 요약하자면, patch scoping은 `"The meaning of X is"`와 같은 prompt를 가져온 다음, SAE가 학습된 것과 동일한 모델로부터 출력을 생성하되, `X` token을 SAE latent의 방향으로 steering하는 과정을 포함합니다. 결과적으로, 우리는 단순히 다른 모델을 사용하여 activation을 기반으로 latent 평가를 자동화하는 대신, 모델 자체의 내부 representation을 활용하여 모델이 직접 latent를 정의하도록 만드는 것입니다.

이 실습에서는 instruction-tuned Gemma 2B 모델을 사용할 예정입니다. 따라서 실습을 시작하기 전에 instruction tuned 모델과 그 작동 방식에 대해 짧게 다루겠습니다. IT 모델의 기본 개념에 익숙하시다면 이 섹션은 자유롭게 건너뛰셔도 좋습니다.

### Instruction-tuned 모델

Instruction tuning은 지시어 프롬프트와 그에 대응하는 출력값으로 구성된 레이블 데이터셋을 통해 LLM을 미세 조정하는 기술입니다. 이는 특정 작업뿐만 아니라 일반적인 지시 사항을 따르는 모델의 성능을 향상시킵니다. Instruction tuning은 RLHF와 동일한 것이 아니라는 점에 유의하십시오. 최적화 과정에서 RL이 필요하지 않으며, ARENA 자료의 첫 번째 장(이미지 분류기를 학습했을 때)에서 수행했던 미세 조정 연습과 동일한 방식으로 진행되는 미세 조정입니다.

Instruction tuning의 학습 데이터는 보통 다음 3가지 요소로 구성됩니다:

- 지시어(Instruction): 주어진 작업을 지정하는 자연어 텍스트 입력입니다. 예를 들어, "이 문장을 영어에서 스페인어로 번역하세요"와 같습니다.
- 추가 정보(Additional information): 현재 작업과 관련된 문맥을 제공하는 선택적 보조 정보입니다. 예를 들어, 독해 작업의 입력에는 짧은 지문이 포함될 수 있으며, 이후 모델에게 해당 지문에 대한 질문에 답하도록 지시합니다.
- 원하는 출력(Desired output): 제공된 지시어와 문맥에 따른 프롬프트의 타겟 출력 응답입니다. 이는 모델의 예측값을 평가하고 최적화하기 위한 ground truth 역할을 합니다.

예를 들어, 다음은 [UltraChat dataset](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k?row=1)의 예시입니다 (간결함을 위해 일부 내용을 생략했습니다):

```python
[
    {
        "content": "Here is a piece of text: SEOUL, SOUTH KOREA — In the final game of their historic match, Google's artificially intelligent Go-playing computer system has defeated Korean grandmaster Lee Sedol, finishing the best-of-five series with four wins and one loss.\nThe win puts an exclamation point on a significant moment for artificial intelligence. Over the last twenty-five years, machines have beaten the best humans [...] There are holes in its education. But, able to draw on months of play with itself—on a corpus of moves that no human has even seen—it also has the ability to climb out of such a deep hole, even against one of the world's best players. AI is flawed. But it is here.\n\nBased on the text material above, generate the response to the following quesion or instruction: Assess the advantages and disadvantages of a machine like AlphaGo compared to a human player like Lee Sedol in a game of Go.",
        "role": "user",
    },
    {
        "content": "One advantage of AlphaGo over Lee Sedol is that it doesn't feel pressure or get tired, and it can play for hours on end without losing focus. Additionally, AlphaGo uses deep neural networks to learn tasks by analyzing massive amounts of digital data, which allows it to make moves that no human would make. On the other hand, Lee Sedol can adjust his strategy based on his opponent's previous moves, but AlphaGo can't do so until it is retrained by its creators. Additionally, Lee Sedol was able to draw on his intuition and adapt to what he saw from his opponent in previous games, something AlphaGo can't do yet. However, even with its flaws, AlphaGo has shown that AI is a force to be reckoned with in complex games like Go.",
        "role": "assistant"
    }
]
```

학습 과정은 위의 내용을 하나의 프롬프트로 포맷팅한 다음, 사용자 텍스트가 아닌 assistant 텍스트에 대한 모델 예측값의 loss를 통해 학습하는 과정을 포함합니다. 학습의 세부 사항은 다양하며, 때로는 이전 레이어들을 freeze하거나 LORA (low-rank adaptation)와 같은 기술을 사용하여 학습 중에 모델이 너무 많이 변하는 것을 방지합니다. 결국 우리는 모델이 알고 있는 지식을 대폭 변경하려는 것이 아니라, 응답하는 방식을 특정 방향으로 유도하려는 것이기 때문입니다.

이러한 모델을 사용할 때 까다로운 부분 중 하나는 프롬프트가 올바른 방식으로 포맷팅되었는지 확인해야 한다는 점입니다. 아래는 instruction-tuned Gemma 2B 모델을 로드하고 예상되는 포맷을 사용하여 응답을 생성하는 예시 코드입니다. 참고로, Gemma 모델을 사용해 본 적이 없다면 섹션 1의 GemmaScope 및 latent steering 섹션으로 돌아가시거나, 적어도 해당 코드를 훑어보며 gemma 모델이 무엇인지와 어떻게 로드하는지 이해하시기를 권장합니다 (모델을 다운로드하려면 먼저 HuggingFace 인증이 필요할 수 있습니다).

알림 - 저장 공간 제약이 발생하는 경우 `huggingface-cli delete-cache`를 사용하여 캐시를 비울 수 있습니다 (이에 대한 자세한 내용은 섹션 1의 GemmaScope 내용을 참조하십시오).

In [ ]:
gemma_2b_it = HookedSAETransformer.from_pretrained("google/gemma-2b-it", device=device)

prompt = "\n".join(
    [
        "<start_of_turn>user",
        "Write a hello world program in python<end_of_turn>",
        "<start_of_turn>model",
    ]
)

GENERATE_KWARGS = dict(temperature=0.5, freq_penalty=2.0)

output = gemma_2b_it.generate(prompt, max_new_tokens=150, **GENERATE_KWARGS)
print("\n" + output)

이제 SAE들도 로드해 보겠습니다. 우리가 사용하는 SAE는 `gemma-2b`에서 학습되었다는 점에 유의하십시오 (이는 이전에 다루었던 `gemma-2-2b` 모델과 동일하지 않습니다).

instruction tuned 모델이 아닌 base 모델에서 학습된 SAE를 사용해도 괜찮은 이유는 무엇일까요? 그 답은 Neel Nanda의 MATS stream의 일부로 진행된 연구에서 찾을 수 있으며, 해당 연구는 다음과 같은 점을 보여줍니다.

In [ ]:
# Display all SAEs trained on the base gemma-2b model
metadata_rows = [
    [data.model, data.release, data.repo_id, len(data.saes_map)]
    for data in get_pretrained_saes_directory().values()
    if data.model == "gemma-2b"
]
print(
    tabulate(
        metadata_rows,
        headers=["model", "release", "repo_id", "n_saes"],
        tablefmt="simple_outline",
    )
)

# Take a closer look at the SAE release we'll be using
sae_release = "gemma-2b-res-jb"
sae_id = "blocks.6.hook_resid_post"
release = get_pretrained_saes_directory()[sae_release]

print(
    tabulate(
        [[k, repr(v)] for k, v in release.__dict__.items() if k not in ["saes_map", "neuronpedia_id"]],
        headers=["Field", "Value"],
        tablefmt="simple_outline",
    )
)

layer 0과 6에서 학습된 SAE는 성능이 좋은 것으로 보이지만, 이후 layer의 SAE들은 그렇지 않은 것 같습니다. layer 6에서 학습된 SAE를 로드하여 살펴보겠습니다.

왜 base model에서 학습된 SAE를 사용하면서, forward pass는 instruction tuned model에서 실행할 수 있을까요? 즉, 왜 base model의 feature가 SAE의 feature일 것이라고 가정해도 되는 것일까요? 이에 대한 답은 포스트 [SAEs (usually) Transfer Between Base and Chat Models](https://www.lesswrong.com/posts/fmwk6qxrpW8d4jvbd/saes-usually-transfer-between-base-and-chat-models)에서 찾을 수 있으며, 여기서는 base model에서 학습된 SAE를 IT model에서 평가했을 때 여러 지표(예: 복구된 cross entropy loss 또는 설명된 분산의 비율)에서 여전히 강력한 성능을 보인다는 것을 보여줍니다.

In [ ]:
gemma_2b_sae = SAE.from_pretrained(sae_release, sae_id, device=str(device))

print(
    tabulate(
        list(gemma_2b_sae.cfg.__dict__.items()) + list(gemma_2b_sae.cfg.metadata.items()),
        headers=["name", "value"],
        tablefmt="simple_outline",
    )
)

### 연습 문제 - patch scoping 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

이제 patch scoping을 구현해 보겠습니다. 방법은 다음과 같습니다:

- 모델에게 특정 용어를 정의하도록 요청하는 prompt를 사용합니다 (아래에 prompt가 제공됩니다). 제안된 개선 사항 [here](https://www.lesswrong.com/posts/8ev6coxChSWcxCDy8/self-explaining-sae-features?commentId=TGzwH4qepiituxgAA)에 따라, latent의 대용으로 `"X"` 대신 unknown token `"<unk>"`을 사용합니다.
- SAE latent 방향으로 token `X`를 steering 하여 모델의 출력을 생성합니다. 이를 위해 layer `replacement_layer`의 residual stream 벡터 `resid_pre`를 L2 norm이 `scale`이 되도록 스케일링된 SAE latent 벡터로 교체합니다. 이 작업은 시퀀스 내에서 `X`이 나타나는 모든 위치에서 수행되어야 합니다.

예시로 다음 latent를 사용하겠습니다:

In [ ]:
latent_idx = 607
display_dashboard(sae_release, sae_id, latent_idx)

출력을 생성하는 동안 참고할 몇 가지 팁입니다:

- 위 셀에서 사용한 것과 동일한 `GENERATE_KWARGS` 기반 출력 코드를 사용할 수 있습니다 (`verbose=False` 또한 설정하고 싶을 수 있습니다).
- 이 모델의 caching 작동 방식에 따라, 처음 생성하는 token의 shape은 `(batch_size=1, seq_len, d_model)` 이지만, 이후의 모든 token에 대해서는 shape이 `(1, 1, d_model)` 이 됩니다. 이는 새로 생성된 token에 대해서만 residual stream 값을 계산하기 때문입니다 (출력을 완전히 결정하기 위해서는 이전 token 위치의 key 및 value 벡터만 필요합니다. 자세한 내용은 ARENA transformers 자료 첫째 날의 보너스 섹션에서 key-value caching에 대해 확인하시기 바랍니다). 따라서 `hook_fn_patch_scoping` 함수에 이 로직을 추가해야 합니다. 즉, activation의 sequence length 차원이 1보다 클 때만 activation을 latent vector로 교체하도록 합니다.

In [ ]:
def hook_fn_patch_scoping(
    activations: Float[Tensor, "batch pos d_model"],
    hook: HookPoint,
    seq_pos: list[int],
    latent_vector: Float[Tensor, " d_model"],
) -> None:
    """
    Steers the model by returning a modified activations tensor, with some multiple of the steering
    vector added to it.

    Note that because of caching, this will be (1, seq_pos, d_model) the first time, and for every
    subsequent token it will be (1, 1, d_model) - see previous exercises in this chapter to revisit
    how KV caching works and why this is the case. You should only replace the activation with the
    latent vector once, i.e. in the first forward pass.
    """
    raise NotImplementedError()


def generate_patch_scoping_explanation(
    model: HookedSAETransformer,
    sae: SAE,
    prompt: str,
    latent_idx: int,
    replacement_layer: int,
    scale: float,
    max_new_tokens: int = 50,
):
    """
    Generates text with steering.

    The steering vector is taken from the SAE's decoder weights for this particular latent. The
    steering magnitude is computed from the `steering_strength` parameter, as well as the maximum
    activation of this latent `max_act` (which has been computed from `find_max_activation`).
    """
    raise NotImplementedError()


scale_list = list(range(0, 60, 10))
replacement_layer = 2

prompt = "\n".join(
    [
        "<start_of_turn>user",
        f'What is the meaning of the word "{gemma_2b_it.tokenizer.unk_token}"?<end_of_turn>',
        "<start_of_turn>model",
        f'The meaning of the word "{gemma_2b_it.tokenizer.unk_token}" is "',
    ]
)

for scale in scale_list:
    output = generate_patch_scoping_explanation(
        gemma_2b_it,
        gemma_2b_sae,
        prompt,
        latent_idx,
        replacement_layer,
        scale,
        max_new_tokens=50,
    )
    output_split = output.removeprefix(prompt).split('"')[0].strip().rstrip(".")
    print(f"scale {scale:02} | {output_split!r}")

<details><summary>솔루션</summary>

```python
def hook_fn_patch_scoping(
    activations: Float[Tensor, "batch pos d_model"],
    hook: HookPoint,
    seq_pos: list[int],
    latent_vector: Float[Tensor, " d_model"],
) -> None:
    """
    Steers the model by returning a modified activations tensor, with some multiple of the steering
    vector added to it.

    Note that because of caching, this will be (1, seq_pos, d_model) the first time, and for every
    subsequent token it will be (1, 1, d_model) - see previous exercises in this chapter to revisit
    how KV caching works and why this is the case. You should only replace the activation with the
    latent vector once, i.e. in the first forward pass.
    """
    if activations.shape[1] > 1:
        activations[:, seq_pos] = latent_vector


def generate_patch_scoping_explanation(
    model: HookedSAETransformer,
    sae: SAE,
    prompt: str,
    latent_idx: int,
    replacement_layer: int,
    scale: float,
    max_new_tokens: int = 50,
):
    """
    Generates text with steering.

    The steering vector is taken from the SAE's decoder weights for this particular latent. The
    steering magnitude is computed from the `steering_strength` parameter, as well as the maximum
    activation of this latent `max_act` (which has been computed from `find_max_activation`).
    """
    positions = [
        i
        for i, a in enumerate(model.tokenizer.encode(prompt))
        if model.tokenizer.decode([a]) == model.tokenizer.unk_token
    ]

    latent_dir = sae.W_dec[latent_idx]
    latent_dir_scaled = (latent_dir / latent_dir.norm(dim=-1)) * scale

    steering_hook = partial(hook_fn_patch_scoping, latent_vector=latent_dir_scaled, seq_pos=positions)

    with model.hooks(fwd_hooks=[(get_act_name("resid_pre", replacement_layer), steering_hook)]):
        output = model.generate(prompt, max_new_tokens=max_new_tokens, **GENERATE_KWARGS)

    return output
```
</details>

### 연습 문제 - patch scoping scale tuning 결과 재현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 30-45 minutes on this exercise, if you choose to attempt it.
> ```

더 많은 patch scoping 실습이 필요하시다면, [`nnsight`](https://nnsight.net/) 라이브러리를 사용한 patch scoping의 [this Colab-based implementation](https://colab.research.google.com/drive/1_geSRb0oPFTsSoiEhUJoYjCQv3JmK4R_?usp=sharing#scrollTo=qNOk3Grmm3By)을 확인해 보시기 바랍니다. 이 노트북에 있는 다른 결과들, 예를 들어 이 feature에 대한 scale tuning 결과를 재현할 수 있습니까?

참고 - 아래의 솔루션은 해당 그래프를 재현하기 위해 미분(derivatives)을 사용하지만, `gemma-2b-it`은 상당히 큰 모델이므로 메모리 사용량이 많을 것입니다! 솔루션에서 미분 계산 부분을 자유롭게 생략하셔도 됩니다. 하지만 관심이 있으시다면, 코드 블록 아래의 드롭다운 메뉴에서 안내를 확인하실 수 있습니다.

In [ ]:
# Your code here - can you replicate the scale tuning plot?

<details>
<summary>1차 및 2차 도함수를 계산하는 방법 (또는 이 replication 과정에서 막혔다면 일반적으로 이 내용을 읽어보시기 바랍니다).</summary>

먼저 hook을 사용하여 모델을 실행해 `self_similarity = feature_dir_normalized @ resid_post_final_normalized`을(를) 얻을 수 있습니다. 이 코드는 이전 연습 문제에서 작성한 코드와 매우 유사하겠지만, (1) diagnostic layer에서 residual stream 값을 추출하기 위한 또 다른 hook을 추가해야 하며, (2) scale에 대한 gradient를 원한다면 `scale` 값을 직접 사용하는 대신 feature direction 벡터에 `t.tensor([float(scale)], device=device, requires_grad=True)`을(를) 곱하여 scale을 계산해야 합니다. 이는 `scale`에 대한 downstream 값들의 gradient를 계산하기 위함입니다.

이제 `scale`과(와) 동일한 computational graph의 일부이자 scalar인 `self_similarity`을(를) 얻었으므로, 다음과 같이 도함수를 계산할 수 있습니다:

```python
first_deriv = t.autograd.grad(self_similarity, scale_tensor, create_graph=True)[0]
second_deriv = t.autograd.grad(first_deriv, scale_tensor, create_graph=True)[0]
```

`torch.autograd.grad`의 처음 두 인자는 `output`과(와) `input`이며, 이는 이 요소들 사이의 gradient만 반환한다는 것을 의미합니다 (출력을 계산하는 데 사용된 모든 leaf node의 `.grad` attribute를 채우는 `.backward()` 메서드의 기본 작동 방식과는 다릅니다).

</details>

<details>
<summary>데이터로부터 plot을 생성하는 코드가 필요하시다면 이 드롭다운을 클릭하십시오 (이 부분이 연습 문제의 핵심이라고 생각하지 않으신다면 사용하시기 바랍니다).</summary>

`scale_list`가 사용 중인 scale의 리스트(즉, x축)이고, `self_similarity`, `self_similarity_first_deriv`, `self_similarity_second_deriv`가 `get_patch_scoping_self_similarity` 함수의 결과(각각 동일한 길이의 리스트 형태)라면 다음 코드가 작동할 것입니다.

```python
fig = px.scatter(
    template="ggplot2",
    width=800,
    height=500,
    title="Patch scoping: steering vector self-similarity",
    x=scale_list,
    y=self_similarity,
    labels={"x": "Scale", "y": "Self-similarity"},
).update_layout(
    yaxis_range=[0.0, 0.3],
)

# Add scatter plot for first & second order derivatives, on each point
for i, (x, ss, ssg, ssgg) in enumerate(
    zip(scale_list, self_similarity, self_similarity_first_deriv, self_similarity_second_deriv)
):
    half_step = scale_step / 2
    xrange = t.linspace(x - half_step, x + half_step, 100)
    y_first_order = ss + ssg * (xrange - x)
    y_second_order = ss + ssg * (xrange - x) + ssgg * (xrange - x) ** 2 / 2
    fig.add_scatter(
        x=xrange,
        y=y_first_order,
        mode="lines",
        opacity=0.5,
        line=dict(color="red", width=1),
        hoverinfo="skip",
        showlegend=i == 0,
        name="1st order approx.",
    )
    fig.add_scatter(
        x=xrange,
        y=y_second_order,
        mode="lines",
        opacity=0.5,
        line=dict(color="blue", width=1),
        hoverinfo="skip",
        showlegend=i == 0,
        name="2nd order approx.",
    )

fig.show()
```

</details>

<details>
<summary>솔루션 (전체 plot을 생성하는 코드)</summary>

```python
def hook_fn_store_value(activations: Tensor, hook: HookPoint):
    hook.ctx["value"] = activations


def get_patch_scoping_self_similarity(
    model: HookedSAETransformer,
    sae: SAE,
    prompt: str,
    latent_idx: int,
    replacement_layer: int,
    diagnostic_layer: int,
    scale: int,
) -> tuple[float, float, float]:
    t.cuda.empty_cache()
    replacement_hook_name = get_act_name("resid_pre", replacement_layer)
    diagnostic_hook_name = get_act_name("resid_pre", diagnostic_layer)

    positions = [i for i, a in enumerate(model.tokenizer.encode(prompt)) if a == model.tokenizer.unk_token_id]

    latent_dir = sae.W_dec[latent_idx]
    latent_dir_normalized = latent_dir / latent_dir.norm(dim=-1)

    scale_tensor = t.tensor([float(scale)], device=device, requires_grad=True)  # to get gradients correctly
    steering_hook = partial(
        hook_fn_patch_scoping, latent_vector=latent_dir_normalized * scale_tensor, seq_pos=positions
    )
    model.run_with_hooks(
        prompt,
        return_type=None,
        fwd_hooks=[
            (replacement_hook_name, steering_hook),
            (diagnostic_hook_name, hook_fn_store_value),
        ],
    )
    resid_post_final: Tensor = model.hook_dict[diagnostic_hook_name].ctx.pop("value")[0, -1]
    resid_post_final_normalized = resid_post_final / resid_post_final.norm(dim=-1)

    self_similarity = latent_dir_normalized @ resid_post_final_normalized
    first_deriv = t.autograd.grad(self_similarity, scale_tensor, create_graph=True)[0]
    second_deriv = t.autograd.grad(first_deriv, scale_tensor, create_graph=True)[0]

    return self_similarity.item(), first_deriv.item(), second_deriv.item()


scale_min, scale_max, n_datapoints = 5, 50, 20
scale_step = (scale_max - scale_min) / n_datapoints
scale_list = t.linspace(scale_min, scale_max, n_datapoints)
replacement_layer = 2
diagnostic_layer = 15

prompt = "\n".join(
    [
        "<start_of_turn>user",
        f'What is the meaning of the word "{gemma_2b_it.tokenizer.unk_token}"?<end_of_turn>',
        "<start_of_turn>model",
        f'The meaning of the word "{gemma_2b_it.tokenizer.unk_token}" is "',
    ]
)

t.set_grad_enabled(True)
self_similarity_results = [
    get_patch_scoping_self_similarity(
        gemma_2b_it,
        gemma_2b_sae,
        prompt,
        latent_idx,
        replacement_layer,
        diagnostic_layer,
        scale,
    )
    for scale in scale_list
]
self_similarity, self_similarity_first_deriv, self_similarity_second_deriv = zip(*self_similarity_results)
t.set_grad_enabled(False)

fig = px.scatter(
    template="ggplot2",
    width=800,
    height=500,
    title="Patch scoping: steering vector self-similarity",
    x=scale_list,
    y=self_similarity,
    labels={"x": "Scale", "y": "Self-similarity"},
).update_layout(yaxis_range=[0.0, 0.3])

# Add scatter plot for first & second order derivatives, on each point
for i, (x, ss, ssg, ssgg) in enumerate(
    zip(
        scale_list,
        self_similarity,
        self_similarity_first_deriv,
        self_similarity_second_deriv,
    )
):
    half_step = scale_step / 2
    xrange = t.linspace(x - half_step, x + half_step, 100)
    y_first_order = ss + ssg * (xrange - x)
    y_second_order = ss + ssg * (xrange - x) + ssgg * (xrange - x) ** 2 / 2
    for y_values, color, name in zip(
        [y_first_order, y_second_order],
        ["red", "blue"],
        ["1st order approx.", "2nd order approx."],
    ):
        fig.add_scatter(
            x=xrange,
            y=y_values,
            mode="lines",
            opacity=0.5,
            line=dict(color=color, width=1),
            hoverinfo="skip",
            showlegend=i == 0,
            name=name,
        )
fig.show()
```

</details>

# 3️⃣ SAE 학습 및 평가

> ##### 학습 목표
>
> - `SAELens`을 사용하여 SAE를 학습시키는 방법을 배웁니다.
> - 학습 중 다양한 metric을 해석하는 방법을 이해하고, SAE 학습이 해석 가능한 latent를 생성하는 데 실패하는 시점과 이유를 이해합니다.
> - TinyStories-1L의 MLP output, Gemma-2-2B의 residual stream, 2L 모델의 attention output 등 다양한 컨텍스트에서 SAE를 학습시키는 실습 경험을 쌓습니다.
> - SAE를 평가하는 방법과 단순한 metric이 왜 기만적일 수 있는지 이해합니다 (아직 구현되지 않음).

## 서론

SAE를 학습시키는 것은 매우 어려울 수 있으며, 새로운 통찰들이 빠르게 발견되고 있습니다. Joseph Bloom의 설명입니다:

> SAE는 activation sparsity를 유도함으로써 reconstruction 정확도와 interpretability 사이의 trade-off를 시도하는 unsupervised 방법입니다. interpretability나 reconstruction 품질에 대한 좋은 metric이 없기 때문에, 우리가 실제로 중요하게 생각하는 것을 최적화하고 있는지 알기 어렵습니다. 게다가, interpretability와 reconstruction 품질 사이의 pareto frontier에서 적절한 지점을 선택하려 노력하고 있는데, 이는 제대로 평가하기 어려운 일입니다. 주요 목표는 일부 dense latents(항상 activate되어 interpretability가 낮을 가능성이 큰 latents)나 너무 많은 dead latents(결코 fire되지 않는 latents) 없이, SAE가 (interpretability가 높을 가능성이 큰) sparse latents 집합을 학습하도록 하는 것입니다.

SAE 학습을 돕기 위해, 학습 중에 기록할 수 있는 수많은 metric을 개발했습니다. 이에 대해서는 나중에 더 자세히 논의하겠습니다. 이러한 metric 중 상당수는 **SAE evaluations**를 수행할 때도 관련이 있습니다. 즉, 학습 중의 성능 향상을 측정하는 것이 아니라, 새로운 SAE architecture나 학습 기법의 유익한 영향을 평가하기 위해 학습 후 SAE의 성능을 측정하는 것입니다. 하지만 학습 중 metric과 학습 후 eval을 위한 metric에는 서로 다른 고려 사항이 들어간다는 점을 인지해야 합니다. 특히 학습 후 metric은 한 번만 계산하면 되지만, 명확하고 상세한 이야기를 들려주는 것이 더 중요합니다. 예를 들어, autointerp scoring과 같은 기법은 SAE interpretability의 유망한 측정 방법이지만, 현재 형태로서는 학습 중에 수행하기에는 비용이 너무 많이 듭니다.

다음으로 넘어가기 전 중요한 참고 사항입니다. **level 1 thinking**과 **level 2 thinking**을 구분하는 것이 중요합니다. 이 문맥에서 level 1 thinking은 metric을 액면 그대로 해석하고, 최적의 trade-off 또는 Pareto frontier 상의 위치로 이끄는 학습 설정을 찾는 것입니다. Level 2는 이러한 proxy objective들이 실제로 우리가 원하는 SAE를 얻는 것과 일치하는지, 아니면 바람직하지 않은 방식으로 서로 어긋나게 될지를 묻는 것입니다. 예를 들어, **feature absorption**은 SAE의 잠재적인 문제 중 하나이며, 이는 sparsity penalty의 효과에 의문을 제기합니다. 하지만 이러한 주제들은 주로 이 자료의 3장(latent absorption, autointerp, latent splitting 등의 주제를 깊게 다룹니다)과 이 장의 후반부(학습 중 eval이 아닌 학습 후 eval을 논의합니다)에서 탐구됩니다. 이 장의 전반부에서는 대부분의 시간 동안 level 1에서 생각하시면 됩니다.

## SAELens를 이용한 학습

많은 다른 것들과 마찬가지로, `SAELens` 덕분에 SAE를 학습시키는 과정이 비교적 간단해집니다. 학습을 위한 코드는 기본적으로 다음과 같습니다:

```python
from sae_lens import LanguageModelSAERunnerConfig, SAETrainingRunner

runner_cfg = LanguageModelSAERunnerConfig(...)
runner = SAETrainingRunner(runner_cfg)
sae = runner.run()
```

### Training config

`LanguageModelSAERunnerConfig` 클래스는 SAE가 어떻게 학습되는지를 지정하는 데 필요한 모든 파라미터를 포함하고 있습니다. 여기에는 SAE config에 들어갔던 많은 파라미터가 포함됩니다 (실제로 이 클래스는 연관된 `SAEConfig` 객체를 생성하는 데 사용할 수 있는 딕셔너리를 반환하는 `get_base_sae_cfg_dict()` 메서드를 포함하고 있습니다). 하지만 학습 과정 자체에 특화된 다른 많은 인자들도 포함되어 있습니다. `LanguageModelSAERunnerConfig`의 소스 코드에서 전체 config 파라미터 세트를 확인할 수 있지만, 여기서는 편의상 약 7가지 주요 카테고리로 그룹화하여 설명하겠습니다:

1. **Data generation** - SAE 학습에 사용하는 데이터가 어떻게 생성되고 batch로 묶이는지에 관한 모든 것입니다. SAE는 기본 TransformerLens 모델의 activation을 통해 학습된다는 점을 기억하십시오. 따라서 여기에는 모델 이름([supported by TransformerLens](https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html) 모델을 가리켜야 합니다), 모델 내의 hook point, dataset 경로(HuggingFace dataset을 참조해야 합니다) 등이 포함됩니다. 또한 `d_in`는 학습 중인 hook point에 의해 고유하게 결정되므로 이 카테고리에 포함시켰습니다.
2. **SAE architecture** - data generation 파라미터로 암시되지 않는 SAE architecture에 관한 모든 것입니다. 여기에는 `d_sae` (사용 가능하지만, 실제로는 보통 `expansion_factor`을 지정하고 `d_sae`을 `d_in * expansion_factor`로 정의합니다), 어떤 activation function을 사용할지, weight를 어떻게 초기화할지, 입력에서 decoder bias `b_dec`를 뺄지 여부 등이 포함됩니다.
3. **Activations store** - 학습 중에 activation이 어떻게 저장되고 처리되는지에 관한 모든 것입니다. 이전에 모델의 latent dashboard를 생성할 때 `ActivationsStore` 클래스를 사용했던 것을 기억하십시오. 학습 중에도 이 클래스의 인스턴스를 사용하여 SAE에 공급할 activation batch들을 저장하고 처리합니다. 여기에는 얼마나 많은 batch를 저장할지, 한 번에 얼마나 많은 prompt를 처리할지 등을 지정해야 합니다.
4. **Training hyperparameters (standard)** - 이는 다른 ML 학습 루프에서 흔히 볼 수 있는 표준 파라미터들입니다: learning rate 및 learning rate scheduling, Adam optimizer를 위한 betas 등이 이에 해당합니다.
5. **Training hyperparameters (SAE-specific)** - SAE 학습에 특화된 모든 파라미터입니다. 이는 $L_1$ penalty와 같은 다양한 계수(및 이러한 계수들을 위한 warmup)와 resampling protocol 같은 것들을 의미하며, 이에 대해서는 나중에 더 자세히 다루겠습니다. 특정 다른 architecture(예: gated)의 경우 지정이 필요한 추가 파라미터가 있을 수 있습니다.
6. **Logging / evals** - Weights & Biases에 데이터를 얼마나 자주 기록할지와 SAE에 대해 얼마나 자주 evaluation을 수행할지를 제어합니다 (evaluation에 대해서는 이 연습 문제 세트의 후반부에서 더 자세히 다룹니다!). 학습 중의 evaluation은 서로 다른 SAE를 비교하기 위해 학습 후에 수행하는 evaluation과는 다른 종류인 경우가 많다는 점을 기억하십시오 (물론 겹치는 부분이 많습니다).
7. **Misc** - 모델 checkpoint를 얼마나 자주 저장할지, random seed, device 및 dtype 등 그 외에 지정하고 싶은 모든 것을 위한 항목입니다.

### SAE 로깅, 체크포인팅 및 저장하기

실제 학습을 진행할 때는 반드시 **Weights and Biases (WandB)로 로깅**을 해야 합니다. 이를 통해 학습 진행 상황을 추적하고 서로 다른 실행 결과를 비교할 수 있습니다. WandB를 활성화하려면 `log_to_wandb=True`를 설정하십시오. 설정의 `wandb_project` 파라미터는 WandB의 프로젝트 이름을 제어합니다. 또한 `wandb_log_frequency`와 `eval_every_n_wandb_logs`을 통해 로깅 빈도를 조절할 수 있습니다. SAE의 sparsity, SAE의 mean squared error (MSE), dead latents, 그리고 explained variance를 포함한 여러 유용한 지표들이 WandB에 기록됩니다. 이러한 지표들은 학습 진행 상황을 모니터링하고 학습 파라미터를 조정하는 데 사용될 수 있습니다. 이 지표들에 대해서는 이후 섹션에서 더 자세히 다루겠습니다.

**Checkpoints**를 사용하면 학습 중에 SAE의 스냅샷과 sparsity 통계량을 저장할 수 있습니다. 체크포인팅을 활성화하려면 `n_checkpoints`를 0보다 큰 값으로 설정하십시오. WandB 로깅이 활성화되어 있다면, 체크포인트는 WandB artifacts로 업로드됩니다. 체크포인트를 로컬에 저장하려면 `checkpoint_path` 파라미터를 로컬 디렉토리로 설정할 수 있습니다.

만족스러운 SAE 세트를 얻었다면, 다음 단계는 이를 세상과 공유하는 것입니다! SAELens에는 이를 쉽게 수행할 수 있게 해주는 `upload_saes_to_huggingface()` 함수가 있습니다. 키는 SAE id이고 값은 `SAE` 객체이거나 `sae.save_model()` 메서드를 사용하여 저장된 SAE의 경로인 딕셔너리를 업로드해야 합니다 (딕셔너리에 두 가지 방식을 혼합해서 사용할 수 있습니다). 터미널에서 `huggingface-cli login`를 실행하거나 `HF_TOKEN` 환경 변수를 API 토큰(레포지토리에 대한 쓰기 권한이 있어야 합니다)으로 설정하여 huggingface 계정에 로그인해야 한다는 점에 유의하십시오.

```python
from sae_lens import SAE, upload_saes_to_huggingface

# Create a dictionary of SAEs (keys = SAE ids (can be hook points but don't have to be), values = SAEs)
saes_dict = {
    "blocks.0.hook_resid_pre": layer_0_sae,
    "blocks.1.hook_resid_pre": layer_1_sae,
}

# Upload SAEs to HuggingFace, in your chosen repo (if it doesn't exist, running this code will create it for you)
upload_saes_to_huggingface(
    saes_dict,
    hf_repo_id="your-username/your-sae-repo",
)

# Load all the SAEs back in
uploaded_saes = {
    layer: SAE.from_pretrained(
        release="your-username/your-sae-repo",
        sae_id=f"blocks.{layer}.hook_resid_pre",
        device=str(device)
    )
    for layer in [0, 1]
}
```

## 학습 조언

이 섹션에서는 몇 가지 일반적인 학습 조언을 다루며, 대략적으로 여러 섹션으로 나누어 설명합니다. 이 섹션의 조언 중 상당 부분은 서로 다른 SAE 모델에도 적용될 수 있음에 유의하시기 바랍니다. 하지만 모든 서로 다른 아키텍처는 각각 고유한 고려 사항을 가지고 있으며, SAE를 학습시킬 때 이러한 사항들이 무엇인지 이해하는 것이 중요합니다.

### Metrics

#### Reconstruction vs sparsity metrics

우리가 논의했듯이, WandB에 기록하는 대부분의 metric은 다양한 방식으로 reconstruction loss 또는 sparsity를 측정하려는 시도입니다. 목표는 이 두 가지 서로 다른 목적이 어떻게 균형을 이루고 있는지 모니터링하고, 가급적이면 이들에 대한 pareto-improvement를 찾는 것입니다! reconstruction loss의 경우, **MSE loss**, **CE loss recovered**, 그리고 **explained variance**에 특히 주의를 기울여야 합니다. sparsity의 경우, L0 및 L1 통계와 activation 히스토그램을 살펴봐야 합니다 (이는 단순히 "선이 올라가거나 내려가는 것"보다 더 미묘하므로 아래에서 더 자세히 설명하겠습니다!).

L1 계수는 정확한 reconstruction과 sparsity 사이의 트레이드오프를 관리하는 주요 레버입니다. 이 값이 너무 높으면 많은 dead latent가 발생하며 (스케줄러를 사용하는 경우 L1 warmup에 의해 완화됩니다 - 아래 참조), 너무 낮으면 latent가 sparse하고 interpretable하기보다는 dense하고 polysemantic하게 됩니다.

이 두 metric 사이의 트레이드오프를 고려할 때 정말 중요한 또 다른 점은, 멋진 sparse하고 interpretable한 latent를 얻기 위해 매우 높은 L1 계수를 사용하고 싶은 유혹이 생길 수 있다는 것입니다. 하지만 이것이 높은 reconstruction loss라는 비용을 치러야 한다면, **실제로 모델의 진정한 동작을 학습하지 못하고 있을** 실질적인 위험이 있습니다. SAE는 모델의 representation이 실제로 무엇인지에 대한 진정한 통찰을 줄 때만 가치가 있으며, 이것 없이 interpretability를 수행하는 것은 시간 낭비가 될 위험이 있습니다. 이 점에 대한 더 자세한 내용은 Neel Nanda, Ryan Greenblat & Buck Schlegris 사이의 토론 [here](https://www.lesswrong.com/posts/tEPHGZAb63dfq2v8n/how-useful-is-mechanistic-interpretability) 을 참조하십시오 (제가 이 포스트의 모든 점에 동의하는 것은 아니지만, SAE 연구자들이 명심하면 좋을 매우 가치 있는 아이디어들을 제시하고 있습니다).

#### ...하지만 metric이 때로는 오해를 불러일으킬 수 있습니다

이 두 그룹 중 하나의 metric들이 종종 비슷한 이야기를 들려주지만 (예: MSE loss가 작을 때 보통 explained variance가 높습니다), 때로는 서로 동떨어질 수 있으며 왜 이런 일이 발생하는지 이해하는 것이 중요합니다. 몇 가지 예시는 다음과 같습니다:

- L0와 L1 모두 sparsity에 대해 알려주지만, latent shrinkage가 발생하면 두 값이 서로 동떨어집니다 (L0가 아니라 L1을 더 작게 만듭니다).
- MSE loss와 KL div / downstream CE 모두 reconstruction에 대해 알려주지만, 하나는 근시안적이고 다른 하나는 그렇지 않기 때문에 서로 동떨어집니다.

이러한 구체적인 예시들을 이해하는 것도 유용하지만, 회의적인 사고방식을 갖고 왜 이런 종류의 문제들이 발생할 수 있는지 이해하는 것이 가치 있습니다. 새로운 metric들이 계속해서 개발되고 있으며, 그중 일부는 현재의 것보다 개선된 것일 수 있지만, 다른 것들은 완전히 새롭고 예측하지 못한 함정을 가지고 있을 수 있습니다!

#### Dead latents, resampling & ghost gradients

**Dead latents**는 어떤 input에서도 전혀 활성화되지 않는 latent들입니다. 이는 학습 중에 큰 문제가 될 수 있는데, gradient를 전혀 받지 못하므로 SAE에서 영구적으로 손실된 capacity를 의미하기 때문입니다. Anthropic이 여러 논문과 업데이트 포스트에서 설명한 dead latent를 처리하는 두 가지 방법은 다음과 같습니다:

1. **Ghost gradients** - 이는 loss에 추가 항을 더하는 방법으로, 기본적으로 dead latent에 autoencoder의 residual을 더 많이 설명하도록 밀어내는 gradient signal을 줍니다. 자세한 기술적 내용은 [here](https://transformer-circuits.pub/2024/jan-update/index.html#dict-learning-resampling) 을 참조하십시오.
2. **Resampling** - 다양한 timestep에서 모든 dead latent를 가져와 residual을 더 잘 설명하는 값으로 무작위로 재초기화합니다 (구체적으로, SAE가 reconstruction에 실패한 input들을 무작위로 선택한 다음, dead feature를 해당 input에 대응하는 SAE hidden state로 설정합니다).

이 기술들은 모두 유용하지만, 처음 도입된 이후 재평가되었으며 이제는 예전만큼 결정적이라고 여겨지지 않습니다 (특히 [ghost grads](https://transformer-circuits.pub/2024/march-update/index.html#dl-update)). 대신, 우리는 dead latent를 피하기 위해 더 표준적인 기술, 구체적으로 적절하게 작은 learning rate와 L1 warmup의 조합을 사용합니다. 일반적으로 resampling은 사용하되 ghost gradients는 사용하지 않는 것을 권장하며, resampling에 의존하지 않도록 LR과 L1 warmup에 충분한 주의를 기울이는 것을 권장합니다.

ghost gradients를 사용하지 않는다고 가정할 때, resampling은 다음 3가지 파라미터에 의해 제어됩니다:

- `feature_sampling_window`: 뉴런을 얼마나 자주 resample 하는지 결정합니다.
- `dead_feature_window`: resample 할 때마다 dead latent를 계산하는 윈도우의 크기입니다. 이는 `feature_sampling_window` 보다 작아야 합니다.
- `dead_feature_threshold`: latent가 dead 되었다고 판단하여 resample 하는 임계값입니다.

#### Dense latents & learning rates

Dense latent는 dead latent와 반대되는 문제입니다: learning rate가 너무 작거나 L1 penalty가 너무 작으면, latent가 sparse해질 때까지 학습시키는 데 실패하게 됩니다. dense latent는 너무 빈번하게 활성화되는 latent입니다 (예: 1/100 또는 심지어 1/10 token 이상에서 활성화). 이러한 latent들은 일반적으로 uninterpretable해 보이지만, reconstruction loss를 낮추는 데는 엄청난 도움이 될 수 있습니다. 본질적으로 이는 SAE가 특별히 sparse하지 않은, 아마도 nonlinear한 계산을 SAE 내부로 몰래 들여오는 방법입니다.

학습 중에 dense latent와 dead latent의 균형을 맞추는 것은 어려울 수 있습니다. 일반적으로는 latent가 dense해지거나 학습 속도가 너무 느려지지 않는 선에서 learning rate를 최대한 낮추는 것이 좋습니다.

참고 - 어떤 상황에서 dense 또는 dead latent의 적절한 수가 얼마여야 하는지는(만약 있다면) 매우 열린 질문이며, 해당 latent distribution의 기저에 무엇이 있다고 믿느냐에 따라 달라집니다. 이 질문을 조사하는 한 가지 방법은 기저의 latent를 추측하기 더 쉬운 단순한 도메인(예: OthelloGPT 또는 TinyStories - 둘 다 이 장의 뒷부분에서 논의하겠습니다)에서 SAE를 학습시켜 보는 것입니다.

#### Interpreting the latent density histogram

이 섹션은 SAE 학습에 관한 Joseph Bloom의 [excellent post](https://www.lesswrong.com/posts/f9EgfLSurAiqRJySD/open-source-sparse-autoencoders-for-all-residual-stream#Why_can_training_Sparse_AutoEncoders_be_difficult__) 에서 직접 인용한 내용입니다:

> Latent density 히스토그램은 SAE 품질의 좋은 척도입니다. 우리는 모든 latent에 대해 log10 latent sparsity (얼마나 자주 활성화되는지)를 플롯합니다. 이를 더 쉽게 실행에 옮기기 위해, 이러한 히스토그램이 진단하는 데 도움을 주는 문제들에 대한 제 생각을 담은 다이어그램을 그렸습니다. Latent density 히스토그램은 다음과 같이 나눌 수 있습니다:
> - **Too Dense**: dense latent는 1 / 100보다 높은 빈도로 발생합니다. 일부 dense-ish한 latent는 괜찮을 가능성이 높지만 (예: token이 공백으로 시작한다는 것을 나타내는 latent), 너무 많으면 문제가 될 가능성이 큽니다.
> - **Too Sparse**: Dead latent는 샘플링되지 않으므로, 0을 기록하는 것을 피하기 위해 추가된 epsilon 값인 log10(epsilon)에서 나타납니다. 이것이 너무 많다는 것은 L1 penalty를 너무 과하게 주고 있다는 뜻입니다.
> - **Just-Right**: dead 또는 dense latent가 너무 많지 않다면, 대부분의 질량이 -5 또는 -4와 -3 log10 latent sparsity 사이에 있는 분포를 보게 됩니다. 정확한 범위는 모델 / SAE 크기에 따라 다를 수 있지만, dense 또는 dead latent는 눈에 띄게 도드라지는 경향이 있습니다.
>
> <img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/f9EgfLSurAiqRJySD/loz4zhrj3ps0cue7uike" width="450">

### Architecture

#### Width

더 넓은 SAE(즉, 더 큰 expansion factor / 더 큰 `d_sae`를 가진 SAE)는 학습 시간이 더 오래 걸리지만, explained variance와 같은 지표에서 더 나은 성능을 보이는 경우가 많습니다. 이전 지표 섹션에서 언급했듯이, 모델의 실제 성능 대부분을 설명하지 못하는 SAE를 과도하게 해석하지 않는 것이 중요합니다. 왜냐하면 그 시점에서는 모델이 아니라 SAE에 대해 배우고 있는 것이기 때문입니다!

width를 선택할 때 **feature splitting**과 같은 개념을 인지하는 것도 좋습니다. 다양한 width에서도 해석 가능한 latent를 얻을 수 있지만, 이러한 latent들은 종종 일종의 feature splitting을 통해 서로 연관되어 있을 수 있습니다. **Feature absorption**은 이와 다른 (아마도 더 심각한) 종류의 문제로, 더 넓은 SAE에서 더 자주 발생할 수 있지만 이론적으로는 어떤 width에서도 발생할 수 있습니다.

#### Gated models

이전 섹션에서 standard 또는 topk와 같은 다른 architecture와 비교하여 gated models에 대해 논의했습니다. 대부분의 경우, 학습 시 gated models를 선택하는 것을 권장합니다. 현재로서는 (transcoders와 같이 패러다임을 바꾸는 중대한 SAE 유사 architecture를 제외하고) 다른 단순한 architecture보다 성능이 더 뛰어난 것으로 보이기 때문입니다. Neel Nanda의 말에 따르면 다음과 같습니다:

> "저는 [[the DeepMind Gated SAEs paper](https://deepmind.google/research/publications/88147/)]이 SAE의 변경 사항이 개선되었는지 엄격하게 평가하는 방법의 좋은 예시로서 읽어볼 가치가 있다고 생각합니다."

gated models를 사용하지 않는다면, 최소한 topk와 같은 방식을 권장합니다. topk는 gated models와 유사한 이점(예: shrinkage 문제 해결)을 제공하기 때문입니다.

### 성능 최적화

#### Datasets & streaming

```python
cfg = LanguageModelSAERunnerConfig(
    dataset_path="apollo-research/roneneldan-TinyStories-tokenizer-gpt2",
    is_dataset_tokenized=True,
    prepend_bos=True,
    streaming=True,
    train_batch_size_tokens=4096,
    context_size=512,
)
```

데이터셋이 pre-tokenized 된 경우 `is_dataset_tokenized` 인자는 `True` 여야 하며, tokenized 되지 않은 경우 `False` 여야 합니다. pre-tokenized 데이터셋은 이후의 모든 학습 실행을 위해 이미 tokenized 및 batched 처리가 완료된 데이터셋을 의미합니다. 이렇게 하면 학습 중에 실시간으로 데이터셋을 tokenize 할 필요가 없으므로 SAE 학습 속도가 빨라집니다. 사용 중인 데이터셋이 pre-tokenized 되지 않은 경우, 직접 pre-tokenize 하는 방법에 대한 정보는 [this tutorial](https://github.com/jbloomAus/SAELens/blob/main/tutorials/pretokenizing_datasets.ipynb) 를 참조하십시오. 하지만 현재 우리가 사용하는 데이터셋은 pre-tokenized 되어 있으므로 이에 대해 걱정할 필요는 없습니다. pre-tokenized 데이터셋의 전체 목록은 아니지만 일부 리스트를 [here](https://github.com/jbloomAus/SAELens/blob/main/docs/training_saes.md#list-of-pretokenized-datasets) 에서 확인하실 수 있습니다.

tokenization 여부와 관계없이, SAELens가 사용하는 데이터셋은 종종 매우 크며 HuggingFace에서 다운로드하는 데 많은 시간과 디스크 공간이 소요됩니다. 이를 가속화하기 위해 config에서 `streaming=True` 을 설정할 수 있습니다. 이렇게 하면 학습 중에 Huggingface에서 데이터셋을 stream 하게 되며, 이를 통해 학습을 즉시 시작할 수 있고 디스크 공간을 절약할 수 있습니다.

#### Context size

`context_size` 파라미터는 모델에 입력되는 prompt의 길이를 제어합니다. context size가 클수록 SAE 성능은 향상되지만, 학습 속도는 느려집니다. 각 학습 batch는 `train_batch_size_tokens * context_size` 크기의 token들로 구성됩니다.

### 기타 팁

- [Anthropic's update](https://transformer-circuits.pub/2024/march-update/index.html#dl-update)에 따라, 기본 beta 값인 `(0.9, 0.999)`을 그대로 사용하는 것을 권장합니다.
- 얼마나 오래 학습시켜야 할까요? 일반적인 ML 조언이 여기에도 적용됩니다. loss 곡선에서 더 이상 개선이 보이지 않는다면, 그때가 아마 학습을 중단할 시점입니다! [this post](https://www.lesswrong.com/posts/f9EgfLSurAiqRJySD/open-source-sparse-autoencoders-for-all-residual-stream#Architecture_and_Hyperparameters)에 설명된 사례와 같이 성공적인 다른 실행 결과들을 참고로 사용할 수도 있습니다. 링크된 학습 실행 결과 [here](https://www.lesswrong.com/posts/fmwk6qxrpW8d4jvbd/saes-usually-transfer-between-base-and-chat-models) 또한 좋은 참고가 될 것입니다.
- 학습 데이터는 모델이 학습된 데이터와 일치해야 합니다. IT 모델 / SAE의 경우 chat 데이터가 좋겠지만 찾기 어려울 수 있으며, DM은 Open Web Text를 사용합니다.

이 장의 나머지 부분(evals 이전)은 세 가지 섹션으로 나뉩니다:

1. TinyStories-1L 모델의 마지막 MLP layer 출력값으로 SAE를 학습시키는 예시 학습 실행 과정을 제시합니다. 이를 통해 처음부터 끝까지의 학습 과정이 실제로 어떻게 진행되는지 확인하고, WandB 페이지에서 몇 가지 예시 metric을 살펴볼 수 있는 기회가 될 것입니다.
2. 실습으로, 동일한 모델에 대해 각각 문제가 있는 여러 가지 학습 실행 사례를 제시합니다. 여러분의 과제는 다양한 metric을 사용하여 이러한 실행 과정에서 무엇이 잘못되었는지 진단하고, 원인이 되었을 가능성이 있는 trainer config 설정을 찾는 것입니다.
3. 마지막으로, SAE 학습의 다양한 사례를 보여주는 일련의 케이스 스터디를 제시합니다. 여기에는 다양한 base 모델(1L 모델부터 Gemma-2B까지), 다양한 hook point(residual stream부터 MLP output, attention output까지), 그리고 다양한 SAE architecture(gated, transcoder 등)가 사용됩니다. 각 사례에 대해 학습 팁과 학습 대상의 특성에 따라 어떻게 다르게 접근하는 것이 좋은지에 대한 권장 사항을 제공합니다. 막히는 부분이 있을 경우를 대비해 샘플 코드도 제공하지만, 이 샘플 코드는 최적이 아닐 수 있으며 더 나은 방법을 찾는 것은 여러분의 몫입니다!

In [ ]:
# We start by emptying memory of all large tensors & objects (since we'll be loading in a lot of different models in the coming sections)
THRESHOLD = 0.1  # GB
for obj in gc.get_objects():
    try:
        if isinstance(obj, t.nn.Module) and utils.get_tensors_size(obj) / 1024**3 > THRESHOLD:
            if hasattr(obj, "cuda"):
                obj.cpu()
            if hasattr(obj, "reset"):
                obj.reset()
    except:
        pass

## 학습 사례 연구: TinyStories-1L, MLP-out

첫 번째 학습 사례 연구에서는 TinyStories 모델의 마지막(유일한) MLP layer 출력값에 대해 SAE를 학습시켜 보겠습니다. [TinyStories](https://arxiv.org/abs/2305.07759)은 짧은 이야기들로 구성된 합성 데이터셋으로, 약 1500개의 단어(주로 일반적인 3~4세 아이들이 이해할 수 있는 흔한 단어들)로 이루어진 vocabulary를 가지고 있습니다. 각 이야기는 비교적 짧고 독립적이며, 이전 문맥을 통해 인과적으로 추론 가능한 기본적인 사건 순서를 포함하고 있습니다. 예시 시퀀스는 다음과 같습니다:

> 옛날 옛적에 릴리라는 작은 소녀가 살았습니다. 릴리는 자신이 인기 있는 공주인 것처럼 행동하는 것을 좋아했습니다. 릴리는 가장 친한 친구인 고양이, 강아지와 함께 큰 성에서 살았습니다. 어느 날 성에서 놀던 중 릴리는 커다란 거미줄을 발견했습니다. 거미줄이 즐거운 놀이를 방해하고 있었습니다. 릴리는 거미줄을 없애고 싶었지만, 그곳에 사는 거미가 무서웠습니다. 릴리는 친구들인 고양이와 강아지에게 도와달라고 부탁했습니다. 그들은 모두 함께 힘을 합쳐 거미줄을 치웠습니다. 거미는 슬펐지만 성 밖에서 새로운 집을 찾았습니다. 릴리와 고양이, 강아지는 거미줄 없이 놀 수 있게 되어 행복했습니다. 그리고 그들은 모두 오래오래 행복하게 살았습니다.

이 데이터셋은 interpretability 분석을 위한 유용한 놀이터를 제공합니다. 왜냐하면 이 데이터셋에서 predictive loss를 최소화하기 위해 모델이 학습해야 하는 feature의 종류가 더 복잡한 자연어 데이터셋으로 학습된 모델보다 훨씬 좁고 단순하기 때문입니다.

이제 SAE를 학습시킬 모델을 로드하고, 텍스트를 생성하여 이 데이터셋으로 학습된 모델이 어떻게 동작하는지 살펴보겠습니다. 이는 모델이 어떤 feature를 학습했을 가능성이 높은지 생각할 때 유용한 첫 번째 단계입니다.

In [ ]:
tinystories_model = HookedSAETransformer.from_pretrained("tiny-stories-1L-21M")

completions = [(i, tinystories_model.generate("Once upon a time", temperature=1, max_new_tokens=50)) for i in range(5)]

print(tabulate(completions, tablefmt="simple_grid", maxcolwidths=[None, 100]))

TransformerLens 라이브러리의 `test_prompt`을 사용하여 모델의 능력을 부분적으로 점검할 수 있습니다:

In [ ]:
test_prompt(
    "Once upon a time, there was a little girl named Lily. She lived in a big, happy little girl. On her big adventure,",
    [" Lily", " she", " he"],
    tinystories_model,
)

위의 출력에서, 모델은 `" she"`이 다음 token이 될 확률을 약 70%로 할당하고(`" he"`은 .01%로 훨씬 낮게 순위가 매겨졌습니다), `" Lily"`가 다음 token이 될 확률을 13%로 할당하는 것을 볼 수 있습니다. Lucy나 Anna와 같은 다른 이름들은 높은 순위를 차지하지 않았습니다.

transformerlens의 `test_prompt` 함수가 제공하는 것보다 더 자세한 뷰를 위해, `circuitsvis` 라이브러리를 사용하여 시각화를 생성할 수 있습니다. 다음 코드에서는 생성된 시퀀스의 모든 token에 대해 다음 token의 logprobs를 시각화합니다. token의 색이 어두울수록 모델이 실제 다음 token에 더 높은 확률을 할당했음을 나타내며, token 위에 마우스를 올리면 logprob 기준 상위 10개 예측값을 확인할 수 있습니다.

In [ ]:
completion = tinystories_model.generate("Once upon a time", temperature=1, verbose=False, max_new_tokens=200)

cv.logits.token_log_probs(
    tinystories_model.to_tokens(completion),
    tinystories_model(completion).squeeze(0).log_softmax(dim=-1),
    tinystories_model.to_string,
)

모델 학습을 시작하기 전에, 이 코드를 직접 실행하며 살펴보시는 것을 권장합니다. 다음과 같은 점들을 탐색해 보십시오:

- 모델이 어떤 token에 높은 확률을 할당합니까? 모델이 다음에 어떤 단어가 올지 어떻게 알 수 있는지 확인해 보시겠습니까?
- token의 순위가 합리적으로 보입니까? 모델이 실제로 다음에 온 token에 높은 확률을 할당하지 않은 경우는 어떠합니까?
- 생성된 completion의 temperature를 변경하여, 모델이 더 가능성이 높거나 낮은 trajectory를 샘플링하도록 시도해 보십시오. 이것이 확률에 어떤 영향을 줍니까?

이제 SAE를 학습시킬 준비가 되었습니다. runner config를 만들고 runner를 인스턴스화하면 나머지는 자동으로 처리됩니다!

학습 과정에서 weights and biases를 사용하여 우리가 중요하게 생각하는 변수들을 얼마나 잘 최적화하고 있는지 나타내는 핵심 metric들을 확인합니다. WandB 대시보드를 재구성하여 L0, CE loss score, explained variance와 같은 중요한 metric들을 상단 한 섹션에 모아둘 수 있습니다. 또한, 여러 번의 학습 실행(예: hyperparameter sweep)을 수행할 때는 각 실행마다 [run comparer](https://docs.wandb.ai/guides/app/features/panels/run-comparer/)를 만드는 것을 권장합니다.

만약 gradient를 비활성화했다면, 학습 전에 `t.set_grad_enabled(True)`를 사용하여 다시 활성화하는 것을 잊지 마십시오.

In [ ]:
total_training_steps = 30_000  # probably we should do more
batch_size = 4096
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = l1_warm_up_steps = total_training_steps // 10  # 10% of training
lr_decay_steps = total_training_steps // 5  # 20% of training

cfg = LanguageModelSAERunnerConfig(
    #
    # SAE architecture
    sae=GatedTrainingSAEConfig(
        d_in=tinystories_model.cfg.d_model,
        d_sae=tinystories_model.cfg.d_model * 16,
        apply_b_dec_to_input=True,
        l1_coefficient=4,
        l1_warm_up_steps=l1_warm_up_steps,
    ),
    #
    # Data generation
    model_name="tiny-stories-1L-21M",  # our model (more options here: https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html)
    hook_name="blocks.0.hook_mlp_out",
    dataset_path="apollo-research/roneneldan-TinyStories-tokenizer-gpt2",  # tokenized language dataset on HF for the Tiny Stories corpus.
    is_dataset_tokenized=True,
    prepend_bos=True,  # you should use whatever the base model was trained with
    streaming=True,  # we could pre-download the token dataset if it was small.
    train_batch_size_tokens=batch_size,
    context_size=512,  # larger is better but takes longer (for tutorial we'll use a short one)
    #
    # Activations store
    n_batches_in_buffer=64,
    training_tokens=total_training_tokens,
    store_batch_size_prompts=16,
    #
    # Training hyperparameters (standard)
    lr=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",  # controls how the LR warmup / decay works
    lr_warm_up_steps=lr_warm_up_steps,  # avoids large number of initial dead features
    lr_decay_steps=lr_decay_steps,  # helps avoid overfitting
    #
    # Training hyperparameters (resampling)
    feature_sampling_window=2000,  # how often we resample dead features
    dead_feature_window=1000,  # size of window to assess whether a feature is dead
    dead_feature_threshold=1e-4,  # threshold for classifying feature as dead, over window
    #
    # Logging / evals
    logger=LoggingConfig(
        log_to_wandb=True,  # always use wandb unless you are just testing code.
        wandb_project="arena-demos-tinystories",
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=20,
    ),
    #
    # Misc.
    device=str(device),
    seed=42,
    n_checkpoints=5,
    checkpoint_path="checkpoints",
    dtype="float32",
)

print("Comment this code out to train! Otherwise, it will load in the already trained model.")
# t.set_grad_enabled(True)
# runner = SAETrainingRunner(cfg)
# sae = runner.run()

hf_repo_id = "callummcdougall/arena-demos-tinystories"
sae_id = cfg.hook_name

# upload_saes_to_huggingface({sae_id: sae}, hf_repo_id=hf_repo_id)

tinystories_sae = SAE.from_pretrained(release=hf_repo_id, sae_id=sae_id, device=str(device))

SAE 학습을 마치셨다면, `sae_vis` 라이브러리의 다음 코드를 사용하여 SAE의 latent를 시각화해 볼 수 있습니다.

(참고 - 이 코드는 `sae_vis` 라이브러리의 브랜치에서 가져온 것으로, 곧 main에 병합될 예정이며 `SAELens`의 나머지 부분과 더 긴밀하게 통합될 것입니다. 예를 들어, 현재 이 메서드는 token 배치를 직접 입력받아 작동하지만, 향후에는 편의를 위해 `ActivationsStore` 객체를 입력받게 될 가능성이 높습니다.)

먼저, 데이터셋에서 token 배치를 가져옵니다:

In [ ]:
dataset = load_dataset(cfg.dataset_path, streaming=True)
batch_size = 1024
tokens = t.tensor(
    [x["input_ids"] for i, x in zip(range(batch_size), dataset["train"])],
    device=str(device),
)
print(tokens.shape)

다음으로, 시각화 자료를 생성하고 저장합니다 (파일을 다운로드하여 브라우저에서 열어 확인해야 합니다). 만약 OOM 에러가 발생한다면, 시각화할 feature의 수를 줄이거나 batch size 또는 context length를 줄이십시오.

In [ ]:
sae_vis_data = SaeVisData.create(
    sae=tinystories_sae,
    model=tinystories_model,
    tokens=tokens,
    cfg=SaeVisConfig(features=range(16)),
    verbose=True,
)
sae_vis_data.save_feature_centric_vis(
    filename=str(section_dir / "feature_vis.html"),
    verbose=True,
)

display_vis_inline(section_dir / "feature_vis.html")

### 연습 문제 - 좋은 학습 곡선과 나쁜 학습 곡선 식별하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 25-40 minutes on this exercise.
> ```

[Here](https://wandb.ai/callum-mcdougall/arena-demos-tinystories-v3/workspace?nw=nwusercallummcdougall)은 7개의 학습 실행(run)이 포함된 WandB 프로젝트 페이지 링크입니다. 첫 번째 실행(Run #0)은 (적어도 다른 실행들과 비교했을 때) "좋은" 학습 실행이며, 베이스라인으로 생각할 수 있습니다. 나머지 6개의 실행(Run #1 - Run #6으로 표시됨)은 각각 특정한 문제가 있습니다. 여러분의 과제는 (하나 이상의 metric 플롯을 통해) 문제를 식별하고, config를 살펴보고 문제의 근본 원인을 찾는 것입니다 ("runs" 탭에서 config들을 서로 비교할 수 있습니다). 즉시 config로 넘어가서 좋은 실행과의 차이점을 찾기보다는, 먼저 플롯을 통해 문제를 식별하려고 시도하는 것을 권장합니다. 각 실행을 평가할 때 다음 패턴을 사용하면 이 연습 문제에서 가장 많은 가치를 얻을 수 있습니다:

1. metric을 살펴보고, SAE 품질이 낮음을 나타내는 지표를 찾습니다.
2. 이러한 metric을 바탕으로 config에서 무엇이 잘못되었을지 추측합니다.
3. config를 확인하여 추측이 맞았는지 테스트합니다.

또한, 실행의 density histogram 플롯을 확인할 수 있다는 점을 기억하십시오. 다만 이는 프로젝트 페이지가 아닌 개별 실행 페이지를 볼 때만 가능합니다.

아래의 드롭다운을 사용하여 각 실행에 대한 정답을 확인하십시오. 처음 몇 개를 정확히 맞히는 것이 더 중요합니다. 마지막 몇 개는 더 어렵고 근본 원인이 항상 명확하지는 않기 때문입니다.

<details>
<summary>Run #1</summary>

이 실행은 L1 계수가 너무 작아서 sparse한 솔루션을 학습하지 못했습니다. 여기서 힌트는 feature sparsity 통계, 예를 들어 L0가 매우 높다는 점이었어야 합니다. 이상적인 L0 값은 모델과 hook 포인트마다 다르며 (또한 SAE width와 같은 요소에도 영향을 받습니다 - 이에 대한 자세한 내용은 앞의 feature splitting 섹션을 참조하십시오), 참고로 canonical GemmaScope SAE들은 L0가 100에 가장 가까운 것들로 선택되었습니다. 이보다 훨씬 큰 값(예: 200+)은 특히 기본적으로 매우 단순한 데이터셋(1L 모델을 사용한 TinyStories)을 다루고 있다면 거의 확실히 나쁜 징후입니다.

</details>

<details>
<summary>Run #2</summary>

이 실행은 dead latent가 너무 많았습니다. 이는 베이스라인 실행에서 사용된 16배의 expansion factor 대신, 불필요하게 큰 32배의 expansion factor를 선택한 결과였습니다. expansion factor가 더 크고 dead feature가 많다고 해서 반드시 나쁜 것은 아니지만, 이는 많은 용량이 낭비되고 있음을 의미합니다. resampling을 통해서도 dead latent의 수를 줄일 수 없었다는 사실은 우리의 `d_sae`가 필요 이상으로 컸다는 신호입니다.

</details>

<details>
<summary>Run #3</summary>

이 실행은 학습률(learning rate)이 매우 낮았습니다: 베이스라인 값인 `5e-5`에 비해 `1e-5`였습니다. 학습 시간을 더 길게 잡는다면 이렇게 낮은 학습률이 본질적으로 나쁜 것은 아니지만 (실제로 시간이 충분하다면 더 작은 학습률과 더 긴 학습 기간이 일반적으로 더 좋습니다), 동일한 수의 training token이 주어진 상황에서 작은 학습률은 학습 종료 시점의 성능 저하로 이어질 수 있습니다. 이 경우, 학습이 끝날 때 대부분의 loss 곡선이 여전히 하강하고 있는 것을 볼 수 있으며, 이는 이 모델이 undertrained 되었음을 시사합니다.

</details>

<details>
<summary>Run #4</summary>

이 실행은 expansion factor가 1이었습니다. 이는 학습된 feature의 수가 MLP output의 차원보다 클 수 없음을 의미합니다. 이는 분명히 좋지 않으며, loss 곡선(sparsity와 reconstruction loss 모두에서)에서 보이는 것처럼 저조한 성능으로 이어집니다.

</details>

<details>
<summary>Run #5</summary>

이 실행은 dead feature의 수가 많았습니다. Run #2와 달리, 원인은 불필요하게 큰 expansion factor가 아니라 다음과 같은 요소들의 조합이었습니다:

- 높은 학습률(learning rate)
- warmup step 없음
- feature resampling 없음

따라서 Run #2와 달리 live feature의 수도 많지 않았으며, 이는 성능이 훨씬 더 낮았음을 의미합니다.

</details>

<details>
<summary>Run #6 (힌트)</summary>

여기서의 실패 모드는 다른 다섯 가지와는 종류가 다릅니다. `metrics/ce_loss_without_sae` 플롯을 살펴보십시오 - 이것이 무엇을 말해주나요? (이 metric이 무엇을 의미하는지 알아내려면 SAELens 소스 코드를 확인하십시오).

</details>

<details>
<summary>Run #6</summary>

이 실행의 실패 모드는 이 섹션의 다른 실행들과 다릅니다. SAE 자체는 완벽하게 학습되었지만, 잘못된 input distribution에서 생성된 activation으로 학습되었습니다! 모델 activation을 생성하는 데 사용된 데이터셋은 `apollo-research/monology-pile-uncopyrighted-tokenizer-gpt2`였는데, 이 데이터셋은 GPT2와 같은 모델을 위해 설계된 것이지 우리가 사용 중인 tinystories 모델을 위한 것이 아니었습니다.

여기서의 힌트는 몇 가지 다른 플롯에서 얻을 수 있었겠지만, 특히 `"metrics/ce_loss_without_sae"` 플롯에서 확인할 수 있습니다 - SAE가 관여하기도 전에 모델의 성능이 프로젝트의 다른 어떤 실행보다 훨씬 더 낮게 나타납니다. 이 metric 플롯은 모델에 적절한 데이터가 입력되고 있는지 확인하는 유용한 sanity check 방법입니다!

</details>

## 더 많은 사례 연구

### 2L 모델의 attn output으로 학습하기

이 섹션에서는 2개 층의 attention-only SAE를 직접 학습해 보시기를 권장합니다. TransformerLens 모델의 이름은 `"attn-only-2l-demo"` 입니다. 모델을 로드하여 구조가 어떻게 생겼는지 살펴볼 수 있습니다.

이 섹션의 적절한 목표는 layer 0과 1 모두의 attention output(즉, `hook_z`)에 대해 SAE를 학습시키고, induction circuit를 형성하는 feature 쌍을 찾을 수 있는지 확인하는 것입니다. 이를 수행하는 방법에 대한 가이드는 이전 섹션들을 다시 참고하시기 바랍니다 (예: 이 연습 세트의 1️⃣ 섹션에 있는 "finding features" 또는 연습 세트 1.4.2의 latent gradient 연습 문제).

<details>
<summary>질문 - 이 모델은 어떤 유형의 positional embeddings를 가지고 있습니까? 이것이 induction circuits에 어떤 영향을 줄까요?</summary>

이 모델은 **shortformer positional embeddings**를 가지고 있습니다. 이는 attention layer에서 value 벡터를 계산하기 전에 residual stream에서 위치 정보를 뺍니다. 따라서 SAE feature가 위치 정보를 직접적으로 포함하지는 않게 됩니다 (하지만 previous-token feature 같은 것들을 찾는 것을 방해하지는 않습니다. 왜냐하면 위치 정보를 한 token에서 다음 token으로 실제로 이동시키는 pointer arithmetic을 수행하지 않더라도 이러한 feature들은 여전히 존재하기 때문입니다).

induction의 경우, shortformer positional embeddings는 induction head가 Q-composition으로는 형성될 수 없고 K-composition으로만 형성될 수 있음을 의미합니다. 이는 induction circuit를 찾을 때 탐색 범위를 좁히는 데 도움이 될 것입니다.

</details>

몇 가지 팁입니다:

- 학습시키는 모델과 살펴보고 있는 hook point에 따라 적절한 expansion factor가 다르므로, attention SAE에 대해 다양한 expansion factor를 실험해 보시는 것이 좋습니다.
- pretokenized 되지 않았거나, 학습 중인 모델의 tokenizer와 tokenization이 일치하는 다른 데이터셋이 필요합니다. 후자의 경우 `model.cfg.tokenizer_name`으로 확인할 수 있으며, pretokenized 데이터셋 [here](https://github.com/jbloomAus/SAELens/blob/main/docs/training_saes.md#list-of-pretokenized-datasets) 중 이 tokenizer를 지원하는 것이 있는지 확인해 보십시오.

시작을 돕기 위해 아래에 합리적인 기본 파라미터들을 제공했습니다. 이를 수정하거나 베이스라인으로 사용하여 hyperparameter sweep를 수행할 수 있습니다. 더 도전적으로 해보고 싶다면, 이전 섹션에서 제공한 config를 바탕으로 처음부터 직접 config를 작성해 보십시오.

In [ ]:
attn_model = HookedSAETransformer.from_pretrained("attn-only-2l-demo")

total_training_steps = 30_000  # probably we should do more
batch_size = 4096
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = l1_warm_up_steps = total_training_steps // 10  # 10% of training
lr_decay_steps = total_training_steps // 5  # 20% of training

layer = 0

d_in_attn = attn_model.cfg.d_head * attn_model.cfg.n_heads

cfg = LanguageModelSAERunnerConfig(
    #
    # SAE architecture
    sae=GatedTrainingSAEConfig(
        d_in=d_in_attn,
        d_sae=d_in_attn * 16,
        apply_b_dec_to_input=True,
        reshape_activations="hook_z",
        l1_coefficient=2,
        l1_warm_up_steps=l1_warm_up_steps,
    ),
    #
    # Data generation
    model_name="attn-only-2l-demo",
    hook_name=f"blocks.{layer}.attn.hook_z",
    dataset_path="apollo-research/Skylion007-openwebtext-tokenizer-EleutherAI-gpt-neox-20b",
    is_dataset_tokenized=True,
    prepend_bos=True,  # you should use whatever the base model was trained with
    streaming=True,  # we could pre-download the token dataset if it was small.
    train_batch_size_tokens=batch_size,
    context_size=attn_model.cfg.n_ctx,
    #
    # Activations store
    n_batches_in_buffer=64,
    training_tokens=total_training_tokens,
    store_batch_size_prompts=16,
    #
    # Training hyperparameters (standard)
    lr=1e-4,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",
    lr_warm_up_steps=lr_warm_up_steps,  # avoids large number of initial dead features
    lr_decay_steps=lr_decay_steps,
    #
    # Training hyperparameters (resampling)
    feature_sampling_window=1000,  # how often we resample dead features
    dead_feature_window=500,  # size of window to assess whether a feature is dead
    dead_feature_threshold=1e-4,  # threshold for classifying feature as dead, over window
    #
    # Logging / evals
    logger=LoggingConfig(
        log_to_wandb=True,  # always use wandb unless you are just testing code.
        wandb_project="arena-demos-attn2l",
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=20,
    ),
    #
    # Misc.
    device=str(device),
    seed=42,
    n_checkpoints=5,
    checkpoint_path="checkpoints",
    dtype="float32",
)

print("Comment this code out to train! Otherwise, it will load in the already trained model.")
# t.set_grad_enabled(True)
# runner = SAETrainingRunner(cfg)
# sae = runner.run()

hf_repo_id = "callummcdougall/arena-demos-attn2l"
sae_id = f"{cfg.hook_name}-v2"

# upload_saes_to_huggingface({sae_id: sae}, hf_repo_id=hf_repo_id)

attn_sae = SAE.from_pretrained(release=hf_repo_id, sae_id=sae_id, device=str(device))

In [ ]:
# Get batch of tokens
dataset = load_dataset(cfg.dataset_path, streaming=True)
batch_size = 1024
seq_len = 256
tokens = t.tensor(
    [x["input_ids"][: seq_len - 1] for i, x in zip(range(batch_size), dataset["train"])],
    device=str(device),
)
bos_token = t.tensor([attn_model.tokenizer.bos_token_id for _ in range(batch_size)], device=device)
tokens = t.cat([bos_token.unsqueeze(1), tokens], dim=1)
assert tokens.shape == (batch_size, seq_len)

# Get a subset of live latents (probably not getting all of them, with only 100 seqs)
acts_post_hook_name = f"{attn_sae.cfg.metadata.hook_name}.hook_sae_acts_post"
_, cache = attn_model.run_with_cache_with_saes(tokens[:100], saes=[attn_sae], names_filter=acts_post_hook_name)
acts = cache[acts_post_hook_name]
alive_feats = (acts.flatten(0, 1) > 1e-8).any(dim=0).nonzero().squeeze().tolist()
print(f"Alive latents: {len(alive_feats)}/{attn_sae.cfg.d_sae}\n")
del cache

# Create vis from live latents
sae_vis_data = SaeVisData.create(
    sae=attn_sae,
    model=attn_model,
    tokens=tokens,
    cfg=SaeVisConfig(features=alive_feats[:32]),
    verbose=True,
    clear_memory_between_batches=True,
)
sae_vis_data.save_feature_centric_vis(filename=str(section_dir / "sae_vis_attn.html"))

display_vis_inline(section_dir / "sae_vis_attn.html")

### Gemma-2B residual stream으로 학습하기

이 섹션에서는 `gemma-2-2b`의 residual stream을 대상으로 학습을 시도해야 합니다. Gemma 모델 시리즈가 무엇인지 복습하고 GemmaScope architecture(본인의 architecture 선택에 도움이 될 것입니다)에 대한 감을 잡기 위해, GemmaScope 섹션(feature steering 직전 섹션)으로 돌아가 확인하시기 바랍니다. 다양한 모델에 적합한 pretokenized 데이터셋 목록은 [here](https://github.com/jbloomAus/SAELens/blob/main/docs/training_saes.md#list-of-pretokenized-datasets)에서 찾을 수 있음을 알려드립니다.

시작하는 데 도움이 될 수 있도록, 합리적인 기본 파라미터들이 포함된 예시 config를 아래에 제공합니다.

> `gemma-2-2b`에서의 학습은 연산 및 메모리 소모가 매우 클 수 있으며, 이러한 이유로 Gemma로 넘어가기 전에 이 섹션의 다른 학습 연습 문제들을 먼저 진행하는 것을 권장합니다. A100(예: Colab Pro+)을 사용하더라도, 품질 좋은 feature를 학습시키는 데는 몇 시간이 아니라 며칠이 걸릴 수 있습니다.

In [ ]:
total_training_steps = 300_000  # Calculated from training_tokens / batch_size
batch_size = 4096
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = l1_warm_up_steps = total_training_steps // 10  # 10% of training
lr_decay_steps = total_training_steps // 5  # 20% of training

layer = 12

cfg = LanguageModelSAERunnerConfig(
    #
    # SAE architecture
    sae=GatedTrainingSAEConfig(
        d_in=2304,
        d_sae=2304 * 8,
        apply_b_dec_to_input=True,
        l1_coefficient=2,
        l1_warm_up_steps=l1_warm_up_steps,
    ),
    #
    # Data generation
    model_name="gemma-2-2b",
    hook_name=f"blocks.{layer}.hook_resid_post",
    dataset_path="chanind/openwebtext-gemma",
    is_dataset_tokenized=True,
    # dataset_path="HuggingFaceFW/fineweb",
    # is_dataset_tokenized=False,
    prepend_bos=True,
    streaming=True,
    train_batch_size_tokens=batch_size,
    context_size=1024,
    #
    # Activations store
    n_batches_in_buffer=16,
    training_tokens=total_training_tokens,
    store_batch_size_prompts=8,
    #
    # Training hyperparameters (standard)
    lr=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",
    lr_warm_up_steps=lr_warm_up_steps,
    lr_decay_steps=lr_decay_steps,
    #
    # Training hyperparameters (resampling)
    feature_sampling_window=5000,
    dead_feature_window=5000,
    dead_feature_threshold=1e-6,
    #
    # Logging / evals
    logger=LoggingConfig(
        log_to_wandb=True,
        wandb_project="arena-demos-gemma2b",
        wandb_log_frequency=50,
        eval_every_n_wandb_logs=20,
    ),
    #
    # Misc.
    device=str(device),
    seed=42,
    n_checkpoints=5,
    checkpoint_path="checkpoints",
    dtype="float32",
)


print("This model hasn't been trained yet!")
# t.set_grad_enabled(True)
# runner = SAETrainingRunner(cfg)
# sae = runner.run()

# hf_repo_id = "callummcdougall/arena-demos-gemma2b"
# sae_id = cfg.hook_name

# upload_saes_to_huggingface({sae_id: sae}, hf_repo_id=hf_repo_id)

# gemma_sae = SAE.from_pretrained(
#     release=hf_repo_id, sae_id=sae_id, device=str(device)
# )

### OthelloGPT로 학습하기

[OthelloGPT](https://arena-chapter1-transformer-interp.streamlit.app/[1.5.3]_OthelloGPT)은 Othello의 유효한 수를 예측하도록 학습된 모델입니다. 이는 SAE를 연구하기에 흥미로운 도메인인데, 대부분의 자연어 데이터셋보다 단순하면서도, 대다수의 토이 문제보다는 더 복잡하기 때문입니다 (어떤 칸이 유효한지 추적하려면 수많은 가능한 캡처와 재캡처를 계속 추적해야 하기 때문입니다). 또한, [research by Neel Nanda](https://www.neelnanda.io/mechanistic-interpretability/othello)은 OthelloGPT가 선형 보드 상태 모델을 포함하고 있음을 강력하게 시사하며, 이는 다음을 의미합니다:

- residual stream에서 학습된 SAE가 이러한 보드 상태 표현을 추출할 수 있을 것으로 기대할 수 있습니다.
- MLP 레이어나 attention 출력에서 학습된 SAE가 이러한 선형 표현을 생성하거나 이를 이용해 수행되는 일부 연산을 포착할 것으로 기대할 수 있습니다.

OthelloGPT에서 SAE를 학습하고 탐색하는 것은 매우 흥미로운 프로젝트가 될 것입니다. 왜냐하면 이는 SAE interpretability의 많은 새로운 기법들(이전 연습 문제에서 논의한 많은 기법들을 포함하여)을 적용해 볼 수 있는 좋은 테스트베드 역할을 하기 때문입니다. 아래에 모델 학습을 위한 샘플 코드를 포함했습니다. 이 코드는 최적화가 덜 되어 있을 가능성이 높으므로, 이를 개선하기 위한 다양한 방법을 시도하거나 (또는 다른 레이어 / 베이스 모델의 다른 부분에 적용해 보는 것을) 권장합니다.

In [ ]:
model_name = "othello-gpt"
othellogpt = HookedSAETransformer.from_pretrained(model_name)

layer = 5
training_tokens = int(1e8)
train_batch_size_tokens = 2048
n_steps = int(training_tokens / train_batch_size_tokens)

cfg = LanguageModelSAERunnerConfig(
    #
    # SAE architecture
    sae=GatedTrainingSAEConfig(
        d_in=othellogpt.cfg.d_mlp,
        d_sae=othellogpt.cfg.d_mlp * 8,
        apply_b_dec_to_input=True,
        l1_coefficient=5,
        l1_warm_up_steps=int(0.2 * n_steps),
    ),
    #
    # Data generation
    model_name=model_name,
    hook_name=f"blocks.{layer}.mlp.hook_post",
    dataset_path="taufeeque/othellogpt",
    is_dataset_tokenized=True,
    prepend_bos=False,
    streaming=True,
    train_batch_size_tokens=train_batch_size_tokens,
    context_size=othellogpt.cfg.n_ctx,  # = 59, we only train on tokens up to (not including) the last one
    seqpos_slice=(5, -5),  # we don't train on the first or last 5 sequence positions
    #
    # Activations store
    n_batches_in_buffer=32,
    store_batch_size_prompts=16,
    training_tokens=training_tokens,
    #
    # Training hyperparameters (standard)
    lr=2e-4,
    adam_beta1=0.9,
    adam_beta2=0.999,
    lr_scheduler_name="constant",
    lr_warm_up_steps=int(0.2 * n_steps),
    lr_decay_steps=int(0.2 * n_steps),
    #
    # Training hyperparameters (resampling)
    feature_sampling_window=1000,
    dead_feature_window=500,
    dead_feature_threshold=1e-5,
    #
    # Logging / evals
    logger=LoggingConfig(
        log_to_wandb=True,
        wandb_project="othello_gpt_sae_16_09",
        wandb_log_frequency=30,
        eval_every_n_wandb_logs=10,
    ),
    #
    # Misc.
    device=str(device),
    seed=42,
    n_checkpoints=5,
    checkpoint_path="checkpoints",
    dtype="float32",
)

# t.set_grad_enabled(True)
# runner = SAETrainingRunner(cfg, override_dataset=override_dataset)
# sae = runner.run()

hf_repo_id = "callummcdougall/arena-demos-othellogpt"
sae_id = f"{cfg.hook_name}-v1"

# upload_saes_to_huggingface({sae_id: sae}, hf_repo_id=hf_repo_id)

othellogpt_sae = SAE.from_pretrained(release=hf_repo_id, sae_id=sae_id, device=str(device))

이제, 이 SAE를 위한 시각화를 생성해 보겠습니다:

In [ ]:
def hf_othello_load(filename):
    path = hf_hub_download(repo_id=hf_repo_id, filename=filename)
    return t.load(path, weights_only=True, map_location=device)


def load_othello_vocab():
    all_squares = [r + c for r in "ABCDEFGH" for c in "01234567"]
    legal_squares = [sq for sq in all_squares if sq not in ["D3", "D4", "E3", "E4"]]
    # Model's vocabulary = all legal squares (plus "pass")
    vocab_dict = {token_id: str_token for token_id, str_token in enumerate(["pass"] + legal_squares)}
    # Probe vocabulary = all squares on the board
    vocab_dict_probes = {token_id: str_token for token_id, str_token in enumerate(all_squares)}
    return {
        "embed": vocab_dict,
        "unembed": vocab_dict,
        "probes": vocab_dict_probes,
    }


othello_tokens = hf_othello_load("tokens.pt")
othello_target_logits = hf_othello_load("target_logits.pt")
othello_linear_probes = hf_othello_load("linear_probes.pt")
print(f"{othello_tokens.shape=}")

# Get live features
acts_post_hook_name = f"{othellogpt_sae.cfg.metadata.hook_name}.hook_sae_acts_post"
_, cache = othellogpt.run_with_cache_with_saes(
    othello_tokens[:500], saes=[othellogpt_sae], names_filter=acts_post_hook_name
)
acts = cache[acts_post_hook_name]
alive_feats = (acts[:, 5:-5].flatten(0, 1) > 1e-8).any(dim=0).nonzero().squeeze().tolist()
print(f"Alive features: {len(alive_feats)}/{othellogpt_sae.cfg.d_sae}\n")
del cache

sae_vis_data = SaeVisData.create(
    sae=othellogpt_sae,
    model=othellogpt,
    linear_probes=[
        ("input", "theirs vs mine", othello_linear_probes["theirs vs mine"]),
        ("output", "theirs vs mine", othello_linear_probes["theirs vs mine"]),
        ("input", "empty", othello_linear_probes["empty"]),
        ("output", "empty", othello_linear_probes["empty"]),
    ],
    tokens=othello_tokens,
    target_logits=othello_target_logits,
    cfg=SaeVisConfig(
        features=alive_feats[:64],
        seqpos_slice=(5, -5),
        feature_centric_layout=SaeVisLayoutConfig.default_othello_layout(),
    ),
    vocab_dict=load_othello_vocab(),
    verbose=True,
    clear_memory_between_batches=True,
)
sae_vis_data.save_feature_centric_vis(
    filename=str(section_dir / "feature_vis_othello.html"),
    verbose=True,
)

display_vis_inline(section_dir / "feature_vis_othello.html")

## SAE 평가하기

> 참고 - 이 섹션은 아직 완성되지 않았습니다 (연습 문제의 정확한 형태가 아직 잡히지 않았습니다). 저의 의도는 평가에 관한 몇 가지 핵심 논문의 결과들을 살펴보는 것입니다. 특히 SAE를 평가하는 4가지 방법(downstream loss, probe loss, interpretability, ablation sparsity)을 다루는 [Scaling and evaluating SAEs](https://cdn.openai.com/papers/sparse-autoencoders.pdf)과, indirect object identification circuit와 관련된 일련의 평가를 수행한 [Towards Principled Evaluations of SAEs](https://arxiv.org/abs/2405.08366)을 살펴볼 예정입니다. 여기서 평가는 **sparse control**(희소한 입력 latent 세트를 변경하여 출력을 예측 가능한 방식으로 변경할 수 있는지 여부)과 **interpretability**(SAE latent를 probe로 사용하여 예상되는 latent를 찾을 수 있는지 여부)라는 질문에 집중합니다.
>
> 또한 새로운 SAELens 튜토리얼 [here](https://github.com/jbloomAus/SAELens/blob/improving-evals/tutorials/evaluating_saes_with_sae_lens_evals.ipynb)이 있으며, 여기서는 학습에 대해 이야기할 때 다루었던 여러 가지 지표들을 살펴보면서 더 깊이 있게 분석합니다 (예: log latent density의 분포 확인 및 **consistent activation heuristic**). 평가에 관심이 있는 분들은 이 노트북을 살펴보시는 것을 추천합니다.
>
> 하지만 다른 콘텐츠와 겹치는 부분이 많을 수 있으며 (예: 연습 문제 세트 1.4.2의 latent-to-latent gradients, 이 연습 문제 세트의 2️⃣ 섹션에 있는 autointerp), 이 경우 이 섹션은 학습에만 집중하도록 축소되고 평가 연습 문제는 적절한 다른 곳으로 이동될 수 있습니다.